# GraviHAR revision experiments (draft 3), single notebook

This notebook contains everything needed for the revision: the original pipeline
(cell 20 of `Gravity_Model.ipynb`, unchanged), the revision experiments and the
analysis that produces every table and number. The plan and the comment-by-comment
mapping are in `revision/REVISION_PLAN.md`.

**How to use**
1. Runtime → Change runtime type → GPU.
2. Run the setup cell and Parts 1–3. They only define functions and take a few seconds.
3. Run the stage cells in order. Each stage appends one row per run to
   `MyDrive/revision/runs/*.csv` and skips runs that are already there, so after a
   disconnect re-run the setup cell and Parts 1–3, then the interrupted stage.
4. Run the last cell for tables, tests, figures and `headline_numbers.json`
   (written to `MyDrive/revision/tables/`).

**Expected data layout on Drive** (same as the original notebook)
- `MyDrive/HHAR/Activity recognition exp/Phones_accelerometer.csv`, `Phones_gyroscope.csv`
- `MyDrive/UCI HAR Dataset/{train,test}/...` (including `subject_*.txt`)
- `MyDrive/MotionSense/A_DeviceMotion_data/<activity>_<trial>/sub_<k>.csv`
- `MyDrive/Shoaib/*.csv` (Participant files)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install xgboost tabulate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- switch the runtime to GPU")

## Part 1: pipeline (notebook cell 20, verbatim)
Loaders, LIMU-BERT, GRU head, augmentation operators, XGBoost features.

In [ ]:
# =============================================================================
#  pipeline_base.py -- VERBATIM copy of Gravity_Model.ipynb cell 20 (the pipeline
#  cell: loaders, LIMU-BERT, GRU head, operators, XGBoost features).
#  Kept as a file so the revision runner can load it with
#      %run -i -n revision/pipeline_base.py
#  (-n stops the __main__ block from running the old entry point).
#  Do not edit here; edit the notebook and re-export if the pipeline changes.
# =============================================================================
# =============================================================================
# HAR : Tier-1 (XGBoost, engineered features)  +  Tier-2 (LIMU-BERT + AutoAugHAR)
# =============================================================================

import os, glob, warnings, math, random
import numpy as np
import pandas as pd
import scipy.fftpack
import joblib

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB  = True
    DATA_ROOT = '/content/drive/MyDrive'
except Exception:
    IN_COLAB  = False
    DATA_ROOT = os.environ.get('HAR_DATA_ROOT', os.path.expanduser('~/har_data'))

BASE_PATH         = os.path.join(DATA_ROOT, 'HHAR')
ACTIVITY_PATH     = os.path.join(BASE_PATH, 'Activity recognition exp')
ANDROID_DATA_PATH = os.path.join(DATA_ROOT, 'Android_Data')
UCI_PATH          = os.path.join(DATA_ROOT, 'UCI HAR Dataset')
SAVE_PATH         = DATA_ROOT if IN_COLAB else os.path.join(DATA_ROOT, 'results')
os.makedirs(SAVE_PATH, exist_ok=True)

MS_PATH = os.path.join(DATA_ROOT, 'MotionSense', 'A_DeviceMotion_data')
SHOAIB_PATH = os.path.join(DATA_ROOT, 'Shoaib')

# ------------------------------------------------------- window geometry -----
STEP_SEC   = 2.5
MIN_SAMPLES_HHAR = 30
TARGET_HZ  = 20.0                              # LIMU-BERT operating rate
N_CHANNELS = 6                                 # acc xyz + gyro xyz

WINDOW_SEC = 5.0                               # in-domain HHAR window
SEQ_LEN    = int(round(WINDOW_SEC * TARGET_HZ))   # L = 100 (5 s @ 20 Hz)

XDOMAIN_WINDOW_SEC = 2.56                       # UCI native window (128 @ 50 Hz)
XDOMAIN_SEQ_LEN    = int(round(XDOMAIN_WINDOW_SEC * TARGET_HZ))  # L = 51 @ 20 Hz

MAX_LEN = 128     # positional-embedding capacity; must be >= any L used above

USE_GRAVITY_ALIGN = False

QUICK  = False
RESUME = True
SEED   = 42
SEEDS  = (42,) if QUICK else (42, 43, 44)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
PRETRAIN_EPOCHS = 5  if QUICK else 80
FINETUNE_EPOCHS = 6  if QUICK else 40
POLICY_EPOCHS   = 4  if QUICK else 15
PRETRAIN_BS, PRETRAIN_LR = 256, 1e-3
FINETUNE_BS, FINETUNE_LR = 128, 1e-3
ALPHA_LR = 3e-3
FREEZE_ENCODER = False
PRETRAIN_CONSISTENCY = True
CONSISTENCY_LAMBDA   = 0.1
CONSISTENCY_MIN_N    = 15000

# Balanced multi-source pool: cap each dataset's contribution so one large set
MULTI_CAP = 25000

# Identity floor for fixed-policy augmentation: with this probability the batch
# passes through unaugmented
AUG_IDENTITY_FLOOR = 0.3

# LIMU-BERT hyper-params (paper defaults)
H_DIM, FF_DIM, N_LAYERS, N_HEADS = 72, 144, 4, 4
SHARE_LAYERS = True
MASK_RATIO, MASK_PROB, SPAN_P, SPAN_LMAX = 0.15, 0.80, 0.20, 10

LABEL_MAP = {
    'stand': 0, 'sit': 0, 'null': -1, 'bike': 6,
    'stairsup': 2, 'stairsdown': 3, 'walk': 4, 'elevator': 1,
    'running': 5, 'walking': 4,
}
CLASS_NAMES = {0:'static(sit/stand)',1:'elevator',2:'stairsUP',3:'stairsDOWN',
               4:'walk',5:'run',6:'bike'}
UCI2OURS = {1: 4, 2: 2, 3: 3, 4: 0, 5: 0}      # WALK,UP,DOWN,SIT,STAND
UCI_G, UCI_HZ = 9.80665, 50.0


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def map_gt_hhar(v):
    return LABEL_MAP.get(str(v).strip().lower(), -1)


# =============================================================================
# TIER-1 : gravity canonicalization + GenHAR/LLM4HAR feature extraction
def _rot_to_vertical(ghat):
    z = np.array([0.0, 0.0, 1.0]); v = np.cross(ghat, z)
    c = float(np.dot(ghat, z)); s = np.linalg.norm(v)
    if s < 1e-8:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / (s * s))

def _apply_R(win_df, R):
    out = win_df.copy()
    out[['x', 'y', 'z']] = win_df[['x', 'y', 'z']].values @ R.T
    return out

def align_window(a_win, g_win):
    m = a_win[['x', 'y', 'z']].values.mean(0); n = np.linalg.norm(m)
    if n < 5.0:
        return a_win, g_win, False
    R = _rot_to_vertical(m / n)
    return _apply_R(a_win, R), _apply_R(g_win, R), True

def align_window_with_ref(a_win, g_win, grav_win):
    if grav_win is None or len(grav_win) == 0:
        return a_win, g_win, False
    m = grav_win[['x', 'y', 'z']].values.mean(0); n = np.linalg.norm(m)
    if n < 1e-6:
        return a_win, g_win, False
    R = _rot_to_vertical(m / n)
    return _apply_R(a_win, R), _apply_R(g_win, R), True

def instance_norm(win):
    w = win.copy()
    for c in "xyz":
        v = w[c].values
        w[c] = (v - v.mean()) / (v.std() + 1e-8)
    return w

def amp_features(win, prefix, hz):
    f = {}
    for ax in "xyz":
        sig = win[ax].values - win[ax].values.mean()
        amp = np.abs(np.fft.rfft(sig)); amp = amp / (amp.sum() + 1e-9)
        frq = np.fft.rfftfreq(len(sig), d=1.0 / hz)
        for lo, hi, nm in [(0,1,"b01"),(1,2,"b12"),(2,3,"b23"),(3,5,"b35")]:
            m = (frq >= lo) & (frq < hi)
            f[f"{prefix}_{ax}_amp_{nm}"] = float(amp[m].sum())
        f[f"{prefix}_{ax}_spec_centroid"] = float((frq * amp).sum())
        di = 1 + int(np.argmax(amp[1:])) if len(amp) > 1 else 0
        h2 = min(2 * di, len(amp) - 1)
        f[f"{prefix}_{ax}_harm_ratio"] = float(amp[h2] / (amp[di] + 1e-9))
    return f

def corr_features(a_win, g_win):
    def cc(u, v):
        n = min(len(u), len(v))
        if n < 2:
            return 0.0
        c = np.corrcoef(u[:n], v[:n])[0, 1]
        return float(c) if np.isfinite(c) else 0.0
    ax, ay, az = a_win.x.values, a_win.y.values, a_win.z.values
    gx, gy, gz = g_win.x.values, g_win.y.values, g_win.z.values
    return {"acc_xy_corr": cc(ax, ay), "acc_xz_corr": cc(ax, az), "acc_yz_corr": cc(ay, az),
            "accgyro_x_corr": cc(ax, gx), "accgyro_y_corr": cc(ay, gy), "accgyro_z_corr": cc(az, gz)}

def features_from_windows(a_win, g_win, sample_hz=100.0, grav_win=None,
                          add_amp=True, add_corr=True, use_instnorm=False):
    if USE_GRAVITY_ALIGN:
        if grav_win is not None:
            a_al, g_al, aligned = align_window_with_ref(a_win, g_win, grav_win)
        else:
            a_al, g_al, aligned = align_window(a_win, g_win)
    else:
        a_al, g_al, aligned = a_win, g_win, False
    a_use, g_use = (instance_norm(a_al), instance_norm(g_al)) if use_instnorm else (a_al, g_al)
    f = {**extract_sensor_features(a_use, "acc", sample_hz),
         **extract_sensor_features(g_use, "gyro", sample_hz)}
    f.pop('acc_z_drift', None); f.pop('acc_z_drift_abs', None)
    vz = a_al['z'].values - a_al['z'].values.mean()
    vel = np.cumsum(vz) * (1.0 / sample_hz)
    f['acc_vert_vel_range'] = float(vel.max() - vel.min())
    if add_amp:
        f.update(amp_features(a_use, "acc", sample_hz))
        f.update(amp_features(g_use, "gyro", sample_hz))
    if add_corr:
        f.update(corr_features(a_use, g_use))
    return f, aligned

def extract_sensor_features(window_df, sensor_prefix, sample_hz=100.0):
    feats = {}
    x = window_df['x'].values.astype(float); y = window_df['y'].values.astype(float)
    z = window_df['z'].values.astype(float); mag = np.sqrt(x*x + y*y + z*z)
    if sensor_prefix == 'acc' and mag.mean() > 5.0:
        x = x - x.mean(); y = y - y.mean(); z = z - z.mean()
        mag = np.sqrt(x*x + y*y + z*z)
    mag_centered = mag - mag.mean()
    feats[f'{sensor_prefix}_x_mean'] = float(np.mean(x)); feats[f'{sensor_prefix}_y_mean'] = float(np.mean(y))
    feats[f'{sensor_prefix}_z_mean'] = float(np.mean(z)); feats[f'{sensor_prefix}_x_std'] = float(np.std(x))
    feats[f'{sensor_prefix}_y_std'] = float(np.std(y));  feats[f'{sensor_prefix}_z_std'] = float(np.std(z))
    feats[f'{sensor_prefix}_mag_mean'] = float(np.mean(mag)); feats[f'{sensor_prefix}_mag_std'] = float(np.std(mag))
    feats[f'{sensor_prefix}_mag_p2p'] = float(np.ptp(mag))
    feats[f'{sensor_prefix}_mag_energy'] = float(np.sum(mag**2) / len(mag))
    feats[f'{sensor_prefix}_sma'] = float(np.sum(np.abs(x)+np.abs(y)+np.abs(z)) / len(mag))
    feats[f'{sensor_prefix}_mag_75th'] = float(np.percentile(mag, 75))
    feats[f'{sensor_prefix}_mag_25th'] = float(np.percentile(mag, 25))
    feats[f'{sensor_prefix}_zcr'] = float(((mag_centered[:-1] * mag_centered[1:]) < 0).sum() / len(mag))
    dt = 1.0 / sample_hz
    feats[f'{sensor_prefix}_z_drift'] = float(np.sum(z) * dt)
    feats[f'{sensor_prefix}_z_drift_abs'] = float(abs(np.sum(z) * dt))
    def _skew(a):
        m, s = a.mean(), a.std(); return float(((a-m)**3).mean() / (s**3 + 1e-9))
    def _kurt(a):
        m, s = a.mean(), a.std(); return float(((a-m)**4).mean() / (s**4 + 1e-9) - 3.0)
    feats[f'{sensor_prefix}_z_skew'] = _skew(z); feats[f'{sensor_prefix}_z_kurt'] = _kurt(z)
    feats[f'{sensor_prefix}_mag_skew'] = _skew(mag); feats[f'{sensor_prefix}_mag_kurt'] = _kurt(mag)
    z_pos = z[z > 0]; z_neg = z[z < 0]
    feats[f'{sensor_prefix}_z_pos_mean'] = float(z_pos.mean()) if len(z_pos) else 0.0
    feats[f'{sensor_prefix}_z_neg_mean'] = float(z_neg.mean()) if len(z_neg) else 0.0
    feats[f'{sensor_prefix}_z_peak_asym'] = feats[f'{sensor_prefix}_z_pos_mean'] + feats[f'{sensor_prefix}_z_neg_mean']
    fft_vals = np.abs(scipy.fftpack.fft(mag)); freqs = scipy.fftpack.fftfreq(len(mag), d=1.0/sample_hz)
    peak_idx = 1 + np.argmax(fft_vals[1:]) if len(fft_vals) > 1 else 0
    fs = fft_vals.sum() + 1e-9
    feats[f'{sensor_prefix}_dominant_freq'] = float(abs(freqs[peak_idx]))
    feats[f'{sensor_prefix}_spectral_entropy'] = float(-np.sum((fft_vals / fs) * np.log(fft_vals / fs + 1e-9)))
    fft_z = np.abs(scipy.fftpack.fft(z - z.mean()))
    z_pidx = 1 + np.argmax(fft_z[1:]) if len(fft_z) > 1 else 0
    feats[f'{sensor_prefix}_z_dominant_freq'] = float(abs(freqs[z_pidx]))
    return feats

def safe_read_csv(path):
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:    return pd.read_csv(path, encoding=enc)
        except Exception: continue
    return None

def standardize_columns(df):
    return df.rename(columns={c: c.strip().lower() for c in df.columns})


# =============================================================================
# TIER-1 : HHAR loading + windowing (features)  +  raw windowing (Tier-2)
# =============================================================================
def infer_sensor(path):
    n = os.path.basename(path).lower()
    return 'gyro' if 'gyro' in n else 'acc' if 'acc' in n else 'unknown'

def load_raw_phone_csv(file_path):
    df = safe_read_csv(file_path)
    if df is None: return None
    df = standardize_columns(df)
    req = ['creation_time', 'x', 'y', 'z', 'user', 'model', 'device', 'gt']
    if not all(c in df.columns for c in req): return None
    df = df[req].dropna().copy()
    for col in ['creation_time', 'x', 'y', 'z']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna()
    df['class_label'] = df['gt'].apply(map_gt_hhar)
    df = df[df['class_label'] >= 0]
    df['time_sec'] = df['creation_time'] / 1e9
    return df.sort_values(['user', 'device', 'time_sec']).reset_index(drop=True)

def load_hhar_phone():
    all_csv = [f for f in glob.glob(os.path.join(ACTIVITY_PATH, '*.csv')) if 'readme' not in f.lower()]
    files_df = pd.DataFrame({'file_path': f, 'sensor_type': infer_sensor(f),
                             'is_phone': 'phone' in os.path.basename(f).lower()} for f in all_csv)
    files_df = files_df[files_df['is_phone']].copy()
    acc_p  = next((r.file_path for _, r in files_df.iterrows() if r.sensor_type == 'acc'),  None)
    gyro_p = next((r.file_path for _, r in files_df.iterrows() if r.sensor_type == 'gyro'), None)
    return load_raw_phone_csv(acc_p), load_raw_phone_csv(gyro_p)


# ---- feature windower (Tier-1) ----------------------------------------------
def window_hhar(acc_raw, gyro_raw, window_sec=WINDOW_SEC, step_sec=STEP_SEC,
                min_samples=MIN_SAMPLES_HHAR):
    results = []
    if acc_raw is None or gyro_raw is None: return results
    for (user, device), acc_grp in acc_raw.groupby(['user', 'device']):
        gyro_grp = gyro_raw[(gyro_raw['user'] == user) & (gyro_raw['device'] == device)]
        if gyro_grp.empty: continue
        t0 = max(acc_grp['time_sec'].min(), gyro_grp['time_sec'].min())
        t1 = min(acc_grp['time_sec'].max(), gyro_grp['time_sec'].max())
        if t1 - t0 < window_sec: continue
        at, gt = acc_grp['time_sec'].values, gyro_grp['time_sec'].values
        t, wid = t0, 0
        while t + window_sec <= t1:
            a_win = acc_grp[(at >= t) & (at < t + window_sec)]
            g_win = gyro_grp[(gt >= t) & (gt < t + window_sec)]
            if len(a_win) >= min_samples and len(g_win) >= min_samples:
                lbl = int(a_win['class_label'].mode().iloc[0])
                feats, _ = features_from_windows(a_win, g_win)
                results.append({'class_label': lbl, 'carrying_load': 0, 'user': user,
                                'device': device, 'source': 'hhar', 'window_id': wid, **feats})
            t += step_sec; wid += 1
    return results


# ---- raw windower (Tier-2), with array-based gravity alignment ---------------
def _resample(arr, L):
    n = arr.shape[0]
    if n == L: return arr.astype(np.float32)
    old = np.linspace(0.0, 1.0, n); new = np.linspace(0.0, 1.0, L)
    out = np.empty((L, arr.shape[1]), dtype=np.float32)
    for c in range(arr.shape[1]):
        out[:, c] = np.interp(new, old, arr[:, c])
    return out

def _align_arrays(acc, gyro):
    """Self-align from accelerometer gravity (array version of align_window)."""
    m = acc.mean(0); n = np.linalg.norm(m)
    if n < 5.0:
        return acc, gyro, False
    R = _rot_to_vertical(m / n)
    return acc @ R.T, gyro @ R.T, True

def raw_from_arrays(acc, gyro, align, has_gravity=True, seq_len=SEQ_LEN):
    """[n,3]+[n,3] -> normalized [L,6]. acc/9.80665 (LIMU-BERT Eq.1); gyro as-is."""
    if align and has_gravity:
        acc, gyro, _ = _align_arrays(acc, gyro)
    acc = _resample(acc, seq_len); gyro = _resample(gyro, seq_len)
    win = np.concatenate([acc, gyro], axis=1).astype(np.float32)
    win[:, 0:3] = win[:, 0:3] / 9.80665
    return win

def window_hhar_raw(acc_raw, gyro_raw, align, window_sec=WINDOW_SEC, step_sec=STEP_SEC,
                    min_samples=MIN_SAMPLES_HHAR, seq_len=SEQ_LEN):
    X, y, groups = [], [], []
    if acc_raw is None or gyro_raw is None:
        return np.empty((0, seq_len, N_CHANNELS), np.float32), np.array([]), np.array([])
    for (user, device), acc_grp in acc_raw.groupby(['user', 'device']):
        gyro_grp = gyro_raw[(gyro_raw['user'] == user) & (gyro_raw['device'] == device)]
        if gyro_grp.empty: continue
        t0 = max(acc_grp['time_sec'].min(), gyro_grp['time_sec'].min())
        t1 = min(acc_grp['time_sec'].max(), gyro_grp['time_sec'].max())
        if t1 - t0 < window_sec: continue
        at, gt = acc_grp['time_sec'].values, gyro_grp['time_sec'].values
        t = t0
        while t + window_sec <= t1:
            a_win = acc_grp[(at >= t) & (at < t + window_sec)]
            g_win = gyro_grp[(gt >= t) & (gt < t + window_sec)]
            if len(a_win) >= min_samples and len(g_win) >= min_samples:
                lbl = int(a_win['class_label'].mode().iloc[0])
                acc = a_win[['x', 'y', 'z']].values.astype(np.float32)
                gyr = g_win[['x', 'y', 'z']].values.astype(np.float32)
                X.append(raw_from_arrays(acc, gyr, align, has_gravity=True, seq_len=seq_len))
                y.append(lbl); groups.append(str(user))
            t += step_sec
    return (np.asarray(X, np.float32) if X else np.empty((0, seq_len, N_CHANNELS), np.float32),
            np.asarray(y), np.asarray(groups))


# =============================================================================
# TIER-1 : Phyphox (features only
def load_phyphox_csv(path):
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    df.columns = ['time_sec', 'x', 'y', 'z'] + list(df.columns[4:])
    return df[['time_sec', 'x', 'y', 'z']].astype(np.float32)

def load_phyphox_gravity_ref(folder_path):
    for fn in ['Gravity.csv', 'Accelerometer.csv', 'Acceleration with g.csv', 'Raw Acceleration.csv']:
        df = load_phyphox_csv(os.path.join(folder_path, fn))
        if df is not None:
            return df, fn
    return None, None

def map_folder_to_label(folder_name):
    name = folder_name.lower(); label = -1
    if 'elevator' in name: label = 1
    elif 'stairsup' in name: label = 2
    elif 'stairsdown' in name: label = 3
    elif 'walking' in name: label = 4
    elif 'running' in name: label = 5
    return label, ('with_box' in name)

def process_phyphox_folder(folder_path, label, carrying_load,
                           window_sec=WINDOW_SEC, step_sec=2.0, min_samples=50):
    acc_df  = load_phyphox_csv(os.path.join(folder_path, 'Linear Acceleration.csv'))
    gyro_df = load_phyphox_csv(os.path.join(folder_path, 'Gyroscope.csv'))
    if acc_df is None or gyro_df is None: return []
    grav_df = None
    if USE_GRAVITY_ALIGN:
        grav_df, _ = load_phyphox_gravity_ref(folder_path)
    max_t = min(acc_df['time_sec'].max(), gyro_df['time_sec'].max())
    at, gt = acc_df['time_sec'].values, gyro_df['time_sec'].values
    grav_t = grav_df['time_sec'].values if grav_df is not None else None
    results, t = [], 0.0
    while t + window_sec <= max_t:
        a_win = acc_df.iloc[np.searchsorted(at, t):np.searchsorted(at, t + window_sec)]
        g_win = gyro_df.iloc[np.searchsorted(gt, t):np.searchsorted(gt, t + window_sec)]
        grav_win = None
        if grav_df is not None:
            grav_win = grav_df.iloc[np.searchsorted(grav_t, t):np.searchsorted(grav_t, t + window_sec)]
        if len(a_win) >= min_samples and len(g_win) >= min_samples:
            feats, _ = features_from_windows(a_win, g_win, grav_win=grav_win)
            results.append({'class_label': label, 'carrying_load': int(carrying_load),
                            'user': 'phyphox_team', 'device': 'phyphox', 'source': 'phyphox',
                            'window_id': int(t / step_sec), **feats})
        t += step_sec
    return results

def load_phyphox_features():
    rows = []
    if os.path.exists(ANDROID_DATA_PATH):
        for folder_name in os.listdir(ANDROID_DATA_PATH):
            fp = os.path.join(ANDROID_DATA_PATH, folder_name)
            if not os.path.isdir(fp): continue
            label, load = map_folder_to_label(folder_name)
            if label < 0: continue
            rows.extend(process_phyphox_folder(fp, label, load))
    return pd.DataFrame(rows).fillna(0) if rows else pd.DataFrame()


# =============================================================================
# TIER-1 : UCI cross-dataset builders (features for Tier-1, raw for Tier-2)
# =============================================================================
def _uci_load(split, sig):
    p = os.path.join(UCI_PATH, split, 'Inertial Signals', f'{sig}_{split}.txt')
    return np.loadtxt(p)

def build_uci_features(feature_cols):
    frames, ys = [], []
    for split in ['train', 'test']:
        ax, ay, az = (_uci_load(split, f'total_acc_{a}') * UCI_G for a in 'xyz')
        gx, gy, gz = (_uci_load(split, f'body_gyro_{a}')       for a in 'xyz')
        yv = np.loadtxt(os.path.join(UCI_PATH, split, f'y_{split}.txt')).astype(int)
        keep = np.isin(yv, list(UCI2OURS))
        rows = []
        for i in range(ax.shape[0]):
            a_df = pd.DataFrame({'x': ax[i], 'y': ay[i], 'z': az[i]})
            g_df = pd.DataFrame({'x': gx[i], 'y': gy[i], 'z': gz[i]})
            feats, _ = features_from_windows(a_df, g_df, sample_hz=UCI_HZ)
            feats['carrying_load'] = 0
            rows.append(feats)
        X = pd.DataFrame(rows).reindex(columns=feature_cols, fill_value=0.0).iloc[np.where(keep)[0]]
        frames.append(X); ys.append(np.array([UCI2OURS[v] for v in yv[keep]]))
    return pd.concat(frames, ignore_index=True), np.concatenate(ys)

def build_uci_raw(align, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for split in ['train', 'test']:
        ax, ay, az = (_uci_load(split, f'total_acc_{a}') * UCI_G for a in 'xyz')
        gx, gy, gz = (_uci_load(split, f'body_gyro_{a}')       for a in 'xyz')
        yv = np.loadtxt(os.path.join(UCI_PATH, split, f'y_{split}.txt')).astype(int)
        for i in range(ax.shape[0]):
            if yv[i] not in UCI2OURS: continue
            acc = np.stack([ax[i], ay[i], az[i]], axis=1)
            gyr = np.stack([gx[i], gy[i], gz[i]], axis=1)
            Xs.append(raw_from_arrays(acc, gyr, align, has_gravity=True, seq_len=seq_len))
            ys.append(UCI2OURS[yv[i]])
    return np.asarray(Xs, np.float32), np.asarray(ys)


# =============================================================================
# TIER-1 : XGBoost evaluation (LOSO + cross-UCI), returns metrics
# =============================================================================
def make_xgb(n_classes):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                         objective='multi:softprob', num_class=n_classes, eval_metric='mlogloss',
                         random_state=42, n_jobs=-1)

def tier1_loso(hhar_df, feature_cols):
    X = hhar_df[feature_cols].copy(); y = hhar_df['class_label'].astype(int).copy()
    groups = hhar_df['user'].astype(str)
    logo = LeaveOneGroupOut(); accs = []
    for tr_idx, te_idx in logo.split(X, y, groups=groups):
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        if (set(y_te.unique()) - set(y_tr.unique())) or len(y_te) < 10:
            continue
        classes = sorted(y_tr.unique())
        lmap = {o: n for n, o in enumerate(classes)}; inv = {n: o for o, n in lmap.items()}
        model = make_xgb(len(classes))
        model.fit(X.iloc[tr_idx], y_tr.map(lmap))
        pred = pd.Series(model.predict(X.iloc[te_idx])).map(inv).values
        accs.append(accuracy_score(y_te, pred))
    return float(np.mean(accs)) if accs else float('nan'), float(np.std(accs)) if accs else float('nan')

def tier1_cross_uci(hhar_df, feature_cols, X_uci, y_uci, seeds=(SEED,)):
    classes = sorted(hhar_df['class_label'].unique())
    lmap = {o: n for n, o in enumerate(classes)}; inv = {n: o for o, n in lmap.items()}
    f4s, fsts = [], []
    for sd in seeds:
        model = make_xgb(len(classes)); model.set_params(random_state=sd)
        model.fit(hhar_df[feature_cols], hhar_df['class_label'].map(lmap))
        pred = pd.Series(model.predict(X_uci[feature_cols])).map(inv).values
        f4s.append(f1_score(y_uci, pred, labels=[0, 2, 3, 4], average='macro'))
        fsts.append(f1_score(y_uci, pred, labels=[2, 3], average='macro'))
    return (float(np.mean(f4s)), float(np.std(f4s)),
            float(np.mean(fsts)), float(np.std(fsts)))


# =============================================================================
# TIER-2 : LIMU-BERT
# =============================================================================
class Embeddings(nn.Module):
    def __init__(self, s_dim, h_dim, max_len):
        super().__init__()
        self.proj = nn.Linear(s_dim, h_dim)
        self.norm1 = nn.LayerNorm(h_dim)
        self.pos = nn.Parameter(torch.zeros(1, max_len, h_dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.norm2 = nn.LayerNorm(h_dim)
    def forward(self, x):
        h = self.norm1(self.proj(x)); h = h + self.pos[:, :x.size(1)]
        return self.norm2(h)

class MultiHeadAttention(nn.Module):
    """Plain matmul + softmax attention. Unlike the fused SDPA kernel, this
    supports double-backward (needed by the AutoAugHAR bilevel search) on GPU."""
    def __init__(self, h_dim, n_heads):
        super().__init__()
        assert h_dim % n_heads == 0
        self.h, self.d = n_heads, h_dim // n_heads
        self.qkv = nn.Linear(h_dim, 3 * h_dim)
        self.out = nn.Linear(h_dim, h_dim)
    def forward(self, x):                                   # x: [B, L, H]
        B, L, H = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.h, self.d).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                    # each [B, heads, L, d]
        att = (q @ k.transpose(-2, -1)) / (self.d ** 0.5)  # [B, heads, L, L]
        att = att.softmax(dim=-1)
        o = (att @ v).transpose(1, 2).reshape(B, L, H)      # [B, L, H]
        return self.out(o)

class EncoderBlock(nn.Module):
    def __init__(self, h_dim, ff_dim, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(h_dim, n_heads)
        self.norm_a = nn.LayerNorm(h_dim)
        self.proj = nn.Linear(h_dim, h_dim); self.norm_p = nn.LayerNorm(h_dim)
        self.ff = nn.Sequential(nn.Linear(h_dim, ff_dim), nn.GELU(), nn.Linear(ff_dim, h_dim))
        self.norm_h = nn.LayerNorm(h_dim)
    def forward(self, h):
        a = self.norm_a(self.attn(h) + h)
        p = self.norm_p(self.proj(a) + a)
        return self.norm_h(self.ff(p) + p)

class LimuEncoder(nn.Module):
    def __init__(self, s_dim=N_CHANNELS, h_dim=H_DIM, ff_dim=FF_DIM, n_layers=N_LAYERS,
                 n_heads=N_HEADS, max_len=MAX_LEN, share=SHARE_LAYERS):
        super().__init__()
        self.embed = Embeddings(s_dim, h_dim, max_len)
        self.share = share; self.n_layers = n_layers
        if share:
            self.block = EncoderBlock(h_dim, ff_dim, n_heads)
        else:
            self.blocks = nn.ModuleList([EncoderBlock(h_dim, ff_dim, n_heads) for _ in range(n_layers)])
    def forward(self, x):
        h = self.embed(x)
        if self.share:
            for _ in range(self.n_layers): h = self.block(h)
        else:
            for blk in self.blocks: h = blk(h)
        return h

class LimuDecoder(nn.Module):
    def __init__(self, h_dim=H_DIM, s_dim=N_CHANNELS):
        super().__init__()
        self.proj = nn.Linear(h_dim, h_dim); self.norm = nn.LayerNorm(h_dim)
        self.pred = nn.Linear(h_dim, s_dim)
    def forward(self, e):
        return self.pred(self.norm(F.gelu(self.proj(e))))

class LimuBERT(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.encoder = LimuEncoder(**kw)
        self.decoder = LimuDecoder(kw.get('h_dim', H_DIM), kw.get('s_dim', N_CHANNELS))
    def forward(self, x_masked):
        return self.decoder(self.encoder(x_masked))

def _span_mask_np(B, L, mask_ratio, p, l_max):
    """Build the [B, L] boolean mask on CPU (numpy) — cheap, no GPU sync."""
    mask = np.zeros((B, L), dtype=bool)
    n_mask = max(1, int(L * mask_ratio))
    for b in range(B):
        chosen = mask[b]; m, guard = 0, 0
        while m < n_mask and guard < 10 * L:
            guard += 1
            s = np.random.randint(0, L)
            if chosen[s]: continue
            l = min(int(np.random.geometric(p)), l_max, n_mask - m)
            e = min(s + l, L)
            for j in range(s, e):
                if not chosen[j]:
                    chosen[j] = True; m += 1
    return mask

def span_mask(x, mask_ratio=MASK_RATIO, p=SPAN_P, l_max=SPAN_LMAX, p_m=MASK_PROB):
    """Algorithm 1, vectorized. Returns (x_masked, mask[B,L]).
    Positions are always recorded for the loss; actual zeroing happens with
    per-sequence probability p_m. Mask is built on CPU then applied in ONE GPU
    op, so no per-element kernel launches stall the device."""
    B, L, C = x.shape
    mask = torch.from_numpy(_span_mask_np(B, L, mask_ratio, p, l_max)).to(x.device)
    do_zero = (torch.rand(B, device=x.device) < p_m).view(B, 1, 1)   # per-sequence
    x_masked = x.masked_fill(mask.unsqueeze(-1) & do_zero, 0.0)
    return x_masked, mask

class GRUClassifier(nn.Module):
    def __init__(self, h_dim=H_DIM, n_classes=7):
        super().__init__()
        self.g1 = nn.GRU(h_dim, 20, batch_first=True)
        self.g2 = nn.GRU(20, 20, batch_first=True)
        self.g3 = nn.GRU(20, 10, batch_first=True)
        self.drop = nn.Dropout(0.5); self.fc1 = nn.Linear(10, 10); self.fc2 = nn.Linear(10, n_classes)
    def forward(self, e):
        h, _ = self.g1(e); h, _ = self.g2(h); h, _ = self.g3(h)
        h = self.drop(h[:, -1]); return self.fc2(F.relu(self.fc1(h)))

class HARModel(nn.Module):
    def __init__(self, n_classes, freeze_encoder=FREEZE_ENCODER, **kw):
        super().__init__()
        self.encoder = LimuEncoder(**kw)
        self.head = GRUClassifier(kw.get('h_dim', H_DIM), n_classes)
        self.freeze = freeze_encoder
    def load_encoder(self, sd): self.encoder.load_state_dict(sd)
    def forward(self, x):
        if self.freeze:
            with torch.no_grad(): e = self.encoder(x)
        else:
            e = self.encoder(x)
        return self.head(e)


# =============================================================================
# TIER-2 : AutoAugHAR (label-preserving operator set + differentiable policy)
# =============================================================================
def op_identity(x): return x
def op_jitter(x, s):  return x + torch.randn_like(x) * s
def op_scaling(x, s):
    f = torch.randn(x.size(0), 1, x.size(2), device=x.device) * s + 1.0; return x * f
def op_moving_avg(x, ws):
    pad = ws // 2; xt = x.transpose(1, 2)
    w = torch.ones(x.size(2), 1, ws, device=x.device) / ws
    y = F.conv1d(F.pad(xt, (pad, pad), mode='replicate'), w, groups=x.size(2))
    return y[..., :x.size(1)].transpose(1, 2)
def op_slope(x, sr=0.1):
    B, L, C = x.shape
    s = (torch.rand(B, 1, C, device=x.device) * 2 - 1) * sr
    t = torch.linspace(0, 1, L, device=x.device).view(1, L, 1)
    return x + s * t
def _smooth_curve(B, L, C, sigma, n_knots, device):
    knots = torch.randn(B, n_knots, C, device=device) * sigma + 1.0
    return F.interpolate(knots.transpose(1, 2), size=L, mode='linear', align_corners=True).transpose(1, 2)
def op_mag_warp(x, s, n_knots=4):
    return x * _smooth_curve(x.size(0), x.size(1), x.size(2), s, n_knots, x.device)
def op_time_warp(x, s, n_knots=4):
    B, L, C = x.shape
    speed = _smooth_curve(B, L, 1, s, n_knots, x.device).clamp(min=0.1)
    cum = torch.cumsum(speed, dim=1); cum = (cum - cum[:, :1]) / (cum[:, -1:] - cum[:, :1] + 1e-6)
    xs = (cum.squeeze(-1) * 2 - 1).unsqueeze(1).expand(B, C, L)
    ys = torch.linspace(-1, 1, C, device=x.device).view(1, C, 1).expand(B, C, L)
    grid = torch.stack([xs, ys], dim=-1)
    img = x.permute(0, 2, 1).unsqueeze(1)
    out = F.grid_sample(img, grid, align_corners=True, padding_mode='border')
    return out.squeeze(1).transpose(1, 2)
def op_window_slice(x, lam_range=(0.7, 0.9)):
    B, L, C = x.shape; lam = random.uniform(*lam_range)
    w = max(2, int(L * lam)); s = random.randint(0, L - w)
    crop = x[:, s:s + w].transpose(1, 2)
    return F.interpolate(crop, size=L, mode='linear', align_corners=True).transpose(1, 2)

def op_rot_z(x, max_deg=180.0):
    """Random rotation about the VERTICAL axis, applied jointly to acc and gyro
    xy-components. After gravity canonicalization, z is vertical, so pitch/roll
    are already fixed and this op spans exactly the remaining free DOF (heading).
    Alignment (2 DOF, analytic) + yaw augmentation (1 DOF, learned invariance)
    = full orientation invariance. CrossHAR-style rotation aug, made principled
    by the canonicalized frame."""
    B = x.size(0)
    th = (torch.rand(B, 1, device=x.device) * 2 - 1) * math.radians(max_deg)
    c, s = torch.cos(th), torch.sin(th)                     # [B,1]
    out = x.clone()
    for i0 in (0, 3):                                       # acc xy, gyro xy
        xs_, ys_ = x[:, :, i0], x[:, :, i0 + 1]
        out[:, :, i0]     = c * xs_ - s * ys_
        out[:, :, i0 + 1] = s * xs_ + c * ys_
    return out

LABEL_PRESERVING_OPS = [
    ("identity", op_identity),
    ("jitter_0.05", lambda x: op_jitter(x, 0.05)), ("jitter_0.10", lambda x: op_jitter(x, 0.10)),
    ("jitter_0.15", lambda x: op_jitter(x, 0.15)),
    ("scale_0.1", lambda x: op_scaling(x, 0.1)), ("scale_0.2", lambda x: op_scaling(x, 0.2)),
    ("magwarp_0.2", lambda x: op_mag_warp(x, 0.2)), ("magwarp_0.4", lambda x: op_mag_warp(x, 0.4)),
    ("timewarp_0.1", lambda x: op_time_warp(x, 0.1)), ("timewarp_0.2", lambda x: op_time_warp(x, 0.2)),
    ("movavg_3", lambda x: op_moving_avg(x, 3)), ("movavg_5", lambda x: op_moving_avg(x, 5)),
    ("movavg_7", lambda x: op_moving_avg(x, 7)),
    ("winslice", lambda x: op_window_slice(x)), ("slope", lambda x: op_slope(x, 0.1)),
    ("rotz_30",  lambda x: op_rot_z(x, 30.0)), ("rotz_180", lambda x: op_rot_z(x, 180.0)),
]

def _default_policy():
    """Physics-informed fallback policy for when the bilevel search cannot be
    trusted (small source, or search converged to ~uniform). Emphasizes the ops
    that (a) the search consistently selected on large sources (jitter, magwarp)
    and (b) span the residual orientation DOF (yaw rotation)."""
    boost = {'rotz_180': 4.0, 'rotz_30': 2.0, 'jitter_0.10': 3.0, 'jitter_0.15': 3.0,
             'magwarp_0.4': 3.0, 'magwarp_0.2': 2.0, 'scale_0.1': 2.0}
    p = np.array([boost.get(n, 1.0) for n, _ in LABEL_PRESERVING_OPS], dtype=float)
    return p / p.sum()
DEFAULT_POLICY = _default_policy()

# Guards for the AutoAugHAR search
MIN_POLICY_N     = 15000
POLICY_MIN_PEAK  = 1.5

class AugPolicy(nn.Module):
    def __init__(self, n_ops=len(LABEL_PRESERVING_OPS), tau=1.0):
        super().__init__()
        self.alpha = nn.Parameter(torch.zeros(n_ops)); self.tau = tau
        self.ops = [op for _, op in LABEL_PRESERVING_OPS]
    def probs(self): return F.softmax(self.alpha.detach(), dim=0)
    def forward(self, x, hard=False):
        g = F.gumbel_softmax(self.alpha, tau=self.tau, hard=hard)
        outs = torch.stack([op(x) for op in self.ops], dim=0)
        return (g.view(-1, 1, 1, 1) * outs).sum(0)

def mixup_cutmix(x, y, n_classes, a_mix=0.3, a_cut=0.8, p=0.5):
    B = x.size(0); y1 = F.one_hot(y, n_classes).float()
    if random.random() > p: return x, y1
    perm = torch.randperm(B, device=x.device)
    if random.random() < 0.5:
        lam = np.random.beta(a_mix, a_mix); x = lam * x + (1 - lam) * x[perm]
    else:
        lam = np.random.beta(a_cut, a_cut); L = x.size(1)
        w = int(L * (1 - lam)); s = random.randint(0, max(0, L - w))
        x = x.clone(); x[:, s:s + w] = x[perm, s:s + w]; lam = 1 - w / L
    return x, lam * y1 + (1 - lam) * y1[perm]


# =============================================================================
# TIER-2 : training / eval
# =============================================================================
def pretrain_limubert(X_unlabeled, epochs=None, bs=PRETRAIN_BS, lr=PRETRAIN_LR,
                      min_epochs=30, patience=8, verbose=True):
    """Span-masking MLM pretraining with plateau early stop, plus an optional
    UniMTS-style ROTATION-CONSISTENCY term (Zhang et al., NeurIPS'24): two random
    yaw-rotated views of the same window must produce similar pooled embeddings.
    MLM makes the representation informative; the consistency term makes it
    orientation-invariant BY CONSTRUCTION rather than hoping fine-tuning learns
    it. The reconstruction loss prevents the trivial collapse a pure consistency
    objective would allow."""
    if epochs is None:
        epochs = PRETRAIN_EPOCHS
    use_cons = PRETRAIN_CONSISTENCY and len(X_unlabeled) >= CONSISTENCY_MIN_N
    if PRETRAIN_CONSISTENCY and not use_cons and verbose:
        print(f"    [pretrain] pool={len(X_unlabeled)} < {CONSISTENCY_MIN_N} "
              f"-> consistency OFF (plain MLM) to avoid small-pool collapse")
    set_seed(); model = LimuBERT().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X = torch.tensor(X_unlabeled, dtype=torch.float32); n = len(X)
    best, stall = float('inf'), 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n); tot = 0.0
        for i in range(0, n, bs):
            xb = X[perm[i:i + bs]].to(DEVICE)
            xm, mask = span_mask(xb)
            loss = F.mse_loss(model(xm)[mask], xb[mask])
            if use_cons:
                v1 = op_rot_z(op_jitter(xb, 0.03), 180.0)
                v2 = op_rot_z(op_jitter(xb, 0.03), 180.0)
                e1 = model.encoder(v1).mean(1)          # [B, H] pooled embeddings
                e2 = model.encoder(v2).mean(1)
                loss = loss + CONSISTENCY_LAMBDA * (1.0 - F.cosine_similarity(e1, e2, dim=1).mean())
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * xb.size(0)
        mse = tot / n
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            print(f"    [pretrain] ep {ep:3d}  loss {mse:.4f}")
        if mse < best * 0.995:
            best, stall = mse, 0
        else:
            stall += 1
            if ep >= min_epochs and stall >= patience:
                if verbose:
                    print(f"    [pretrain] early stop at ep {ep} (plateau, best {best:.4f})")
                break
    return {k: v.cpu() for k, v in model.encoder.state_dict().items()}

def _soft_ce(logits, soft): return -(soft * F.log_softmax(logits, dim=1)).sum(1).mean()

def train_classifier(encoder_state, X_tr, y_tr, n_classes, aug_mode="off",
                     epochs=FINETUNE_EPOCHS, bs=FINETUNE_BS, lr=FINETUNE_LR,
                     fixed_policy=None, return_policy=False, freeze=None,
                     seed=SEED, verbose=False):
    """aug_mode:
        'off'    - no augmentation
        'random' - MixUp/CutMix + one random label-preserving op
        'search' - AutoAugHAR bilevel policy search (expensive: double-backward)
        'fixed'  - apply a PRE-LEARNED policy (fixed_policy over the ops), cheap.
                   Use 'search' once via learn_policy(), then 'fixed' everywhere.
       freeze:   override FREEZE_ENCODER. The bilevel search REQUIRES freeze=False
                 (gradients must reach the augmentation params through the encoder).
       seed:     controls init + batch order. MUST be threaded from the caller for
                 multi-seed error bars; a hardcoded set_seed() here would silently
                 make every 'seed' identical (std = 0.000)."""
    set_seed(seed)
    fz = FREEZE_ENCODER if freeze is None else freeze
    model = HARModel(n_classes, freeze_encoder=fz).to(DEVICE)
    # class-weighted CE: static classes dominate several targets
    cnt = np.bincount(np.asarray(y_tr), minlength=n_classes).astype(float)
    cnt[cnt == 0] = 1.0
    cw = torch.tensor(len(y_tr) / (n_classes * cnt), dtype=torch.float32).to(DEVICE)
    if encoder_state is not None:
        model.load_encoder({k: v.to(DEVICE) for k, v in encoder_state.items()})
    w_params = ([p for p in model.parameters() if p.requires_grad]
                if not model.freeze else list(model.head.parameters()))
    w_opt = torch.optim.Adam(w_params, lr=lr)
    Xtr = torch.tensor(X_tr, dtype=torch.float32); ytr = torch.tensor(y_tr, dtype=torch.long)
    fp = np.asarray(fixed_policy, dtype=float) if fixed_policy is not None else None
    if fp is not None:
        fp = fp / fp.sum()

    aug = None; va_idx = []
    if aug_mode == "search":
        aug = AugPolicy().to(DEVICE); a_opt = torch.optim.Adam(aug.parameters(), lr=ALPHA_LR)
        idx = torch.randperm(len(Xtr)); cut = int(0.85 * len(Xtr))
        tr_idx, va_idx = idx[:cut], idx[cut:]
        Xva, yva = Xtr[va_idx], ytr[va_idx]; Xtr, ytr = Xtr[tr_idx], ytr[tr_idx]

    n = len(Xtr)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            xb = Xtr[perm[i:i + bs]].to(DEVICE); yb = ytr[perm[i:i + bs]].to(DEVICE)
            if aug_mode == "search" and len(va_idx) >= bs:
                vsel = torch.randint(0, len(Xva), (bs,))
                xv = Xva[vsel].to(DEVICE); yv = yva[vsel].to(DEVICE)
                with torch.backends.cudnn.flags(enabled=False):
                    params = {k: v for k, v in model.named_parameters() if v.requires_grad}
                    x_aug = aug(xb)
                    loss_tr = F.cross_entropy(
                        functional_call(model, {**dict(model.named_parameters()), **params}, (x_aug,)), yb,
                        weight=cw)
                    grads = torch.autograd.grad(loss_tr, params.values(), create_graph=True, allow_unused=True)
                    vparams = {k: (v - lr * g if g is not None else v)
                               for (k, v), g in zip(params.items(), grads)}
                    full = {**dict(model.named_parameters()), **vparams}
                    loss_val = F.cross_entropy(functional_call(model, full, (xv,)), yv, weight=cw)
                    a_opt.zero_grad(); loss_val.backward(); a_opt.step()
            if aug_mode == "search":
                xin = aug(xb).detach(); loss = F.cross_entropy(model(xin), yb, weight=cw)
            elif aug_mode == "fixed":                       # apply learned policy (cheap)
                if random.random() < AUG_IDENTITY_FLOOR:
                    xin = xb                                # clean-data anchor
                else:
                    op = LABEL_PRESERVING_OPS[int(np.random.choice(len(fp), p=fp))][1]
                    xin = op(xb).detach()
                loss = F.cross_entropy(model(xin), yb, weight=cw)
            elif aug_mode == "random":
                xin, soft = mixup_cutmix(xb, yb, n_classes)
                xin = random.choice(LABEL_PRESERVING_OPS)[1](xin).detach()
                loss = _soft_ce(model(xin), soft)
            else:
                loss = F.cross_entropy(model(xb), yb, weight=cw)
            w_opt.zero_grad(); loss.backward(); w_opt.step()

    policy_out = aug.probs().cpu().numpy() if aug is not None else None
    if aug is not None and verbose:
        print("    learned policy:", {LABEL_PRESERVING_OPS[j][0]: round(float(policy_out[j]), 3)
                                       for j in np.argsort(-policy_out)[:5]})
    return (model, policy_out) if return_policy else model


def learn_policy(encoder_state, X, y, n_classes, epochs=None, seed=SEED):
    """Run the AutoAugHAR bilevel search ONCE and return the learned op
    distribution (numpy). Apply it afterwards with aug_mode='fixed' -- this is
    the search-once/apply-many pattern, ~1 search instead of one per LOSO fold.
    The second-order step may lack kernels on Apple MPS; if it fails there,
    retry the search on CPU (slow but correct), then restore the device."""
    global DEVICE
    ep = epochs if epochs is not None else POLICY_EPOCHS
    try:
        _, policy = train_classifier(encoder_state, X, y, n_classes, aug_mode="search",
                                     epochs=ep, return_policy=True, freeze=False,
                                     seed=seed, verbose=True)
        return policy
    except RuntimeError as e:
        if DEVICE.type != "cpu":
            print(f"    [policy] search failed on {DEVICE} ({str(e)[:90]}...)")
            print("    [policy] retrying the policy search on CPU (slower, one-time) ...")
            old, DEVICE = DEVICE, torch.device("cpu")
            try:
                _, policy = train_classifier(encoder_state, X, y, n_classes, aug_mode="search",
                                             epochs=ep, return_policy=True, freeze=False,
                                             seed=seed, verbose=True)
            finally:
                DEVICE = old
            return policy
        raise


# ---- disk caches: pretraining
CACHE_VERSION = "v3"
def _cache_dir():
    d = os.path.join(SAVE_PATH, 'enc_cache'); os.makedirs(d, exist_ok=True); return d

def cached_pretrain(tag, X_pool, **kw):
    cons = PRETRAIN_CONSISTENCY and len(X_pool) >= CONSISTENCY_MIN_N
    tag = tag + ("" if cons else "_nc")
    fp = os.path.join(_cache_dir(), f"enc_{CACHE_VERSION}_{tag}.pt")
    if RESUME and os.path.exists(fp):
        print(f"    [cache] encoder '{tag}' loaded from disk (delete enc_cache to retrain)")
        return torch.load(fp, map_location='cpu')
    enc = pretrain_limubert(X_pool, **kw)
    try:
        torch.save(enc, fp)
    except Exception as e:
        print(f"    [warn] could not save encoder cache: {e}")
    return enc

def cached_policy(tag, enc, X, y, n_classes):
    fp = os.path.join(_cache_dir(), f"pol_{CACHE_VERSION}_{tag}.npy")
    if RESUME and os.path.exists(fp):
        pol = np.load(fp)
        if len(pol) == len(LABEL_PRESERVING_OPS):
            print(f"    [cache] policy '{tag}' loaded from disk")
            return pol
        print(f"    [cache] policy '{tag}' has stale op count ({len(pol)}) -> re-searching")
    pol = learn_policy(enc, X, y, n_classes)
    try:
        np.save(fp, pol)
    except Exception as e:
        print(f"    [warn] could not save policy cache: {e}")
    return pol

@torch.no_grad()
def predict(model, X, bs=512):
    model.eval(); X = torch.tensor(X, dtype=torch.float32); out = []
    for i in range(0, len(X), bs):
        out.append(model(X[i:i + bs].to(DEVICE)).argmax(1).cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def predict_proba_t2(model, X, bs=512):
    model.eval(); X = torch.tensor(X, dtype=torch.float32); out = []
    for i in range(0, len(X), bs):
        out.append(F.softmax(model(X[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy()

def tier2_loso(encoder_state, X, y, groups, aug_mode="off", fixed_policy=None):
    accs = []
    for subj in np.unique(groups):
        te = groups == subj; tr = ~te
        classes = sorted(np.unique(y[tr]))
        if (set(np.unique(y[te])) - set(classes)) or te.sum() < 10:
            continue
        lmap = {c: i for i, c in enumerate(classes)}; inv = {i: c for c, i in lmap.items()}
        y_tr = np.array([lmap[v] for v in y[tr]])
        model = train_classifier(encoder_state, X[tr], y_tr, len(classes),
                                 aug_mode=aug_mode, fixed_policy=fixed_policy)
        pred = np.array([inv[p] for p in predict(model, X[te])])
        accs.append(accuracy_score(y[te], pred))
    return float(np.mean(accs)) if accs else float('nan'), float(np.std(accs)) if accs else float('nan')

def tier2_cross_uci(encoder_state, X_hhar, y_hhar, X_uci, y_uci, aug_mode="off",
                    fixed_policy=None, seeds=(SEED,), verbose=False):
    """Repeat the fine-tune over `seeds` and return mean/std of each metric.
    The cross-domain metric is a SINGLE train/test split, so without repetition it
    carries no error bar at all -- the biggest weakness a reviewer will attack."""
    classes = sorted(np.unique(y_hhar))
    lmap = {c: i for i, c in enumerate(classes)}; inv = {i: c for c, i in lmap.items()}
    y_tr = np.array([lmap[v] for v in y_hhar])
    f4s, fsts = [], []
    for sd in seeds:
        model = train_classifier(encoder_state, X_hhar, y_tr, len(classes), aug_mode=aug_mode,
                                 fixed_policy=fixed_policy, seed=sd,
                                 verbose=(verbose and sd == seeds[0]))
        pred = np.array([inv[p] for p in predict(model, X_uci)])
        f4s.append(f1_score(y_uci, pred, labels=[0, 2, 3, 4], average='macro'))
        fsts.append(f1_score(y_uci, pred, labels=[2, 3], average='macro'))
    return (float(np.mean(f4s)), float(np.std(f4s)),
            float(np.mean(fsts)), float(np.std(fsts)))

G0 = 9.80665

# --- MotionSense --------------------------------------------------------------
# Layout: MS_PATH/<act>_<trial>/sub_<k>.csv with columns gravity.{x,y,z},
# userAcceleration.{x,y,z}, rotationRate.{x,y,z} (units of g / rad/s).
# Total acceleration = (gravity + userAcceleration) * 9.80665  -> gravity PRESENT
MS_LABEL = {'dws': 3, 'ups': 2, 'wlk': 4, 'jog': 5, 'sit': 0, 'std': 0}

def load_motionsense_raw(align, seq_len=XDOMAIN_SEQ_LEN, path=MS_PATH,
                         win=128, step=64):
    if not os.path.exists(path):
        print(f"  [MotionSense] path not found: {path}"); return None, None
    Xs, ys, n_files = [], [], 0
    trial_dirs = []
    for root, dirs, _ in os.walk(path):
        dirs[:] = [d for d in dirs if d != '__MACOSX' and not d.startswith('.')]
        if '__MACOSX' in root:
            continue
        for d in dirs:
            act = d.split('_')[0].lower()
            if act in MS_LABEL:
                trial_dirs.append((os.path.join(root, d), MS_LABEL[act]))
    for tdir, lbl in sorted(trial_dirs):
        for fn in sorted(os.listdir(tdir)):
            if not fn.endswith('.csv') or fn.startswith('.'):
                continue
            try:
                df = pd.read_csv(os.path.join(tdir, fn))
            except Exception as e:
                print(f"  [MotionSense] unreadable {fn}: {e} -- skipped")
                continue
            need = ['gravity.x', 'gravity.y', 'gravity.z',
                    'userAcceleration.x', 'userAcceleration.y', 'userAcceleration.z',
                    'rotationRate.x', 'rotationRate.y', 'rotationRate.z']
            if not all(c in df.columns for c in need):
                continue
            n_files += 1
            acc = (df[['gravity.x', 'gravity.y', 'gravity.z']].values
                   + df[['userAcceleration.x', 'userAcceleration.y',
                         'userAcceleration.z']].values) * G0
            gyr = df[['rotationRate.x', 'rotationRate.y', 'rotationRate.z']].values
            for s in range(0, len(acc) - win + 1, step):
                Xs.append(raw_from_arrays(acc[s:s+win].astype(np.float32),
                                          gyr[s:s+win].astype(np.float32),
                                          align, has_gravity=True, seq_len=seq_len))
                ys.append(lbl)
    if not Xs:
        print("  [MotionSense] no windows parsed -- check folder layout/columns.")
        return None, None
    X, y = np.asarray(Xs, np.float32), np.asarray(ys)
    print(f"  [MotionSense] files={n_files}  windows={len(X)}  "
          f"classes={dict(pd.Series(y).value_counts().sort_index())}")
    return X, y

# --- Shoaib 2014 ---------------------------------------------------------------
# Layout: Participant_1.csv .. Participant_10.csv. Each file: a header row
# containing repeated blocks (one per body position: left pocket, right pocket,
# wrist, upper arm, belt) of columns time_stamp, Ax,Ay,Az, Lx,Ly,Lz, Gx,Gy,Gz,
# Mx,My,Mz, and a trailing activity-label column. A=acc WITH gravity, G=gyro.
SHOAIB_LABEL = {'walking': 4, 'jogging': 5, 'running': 5, 'sitting': 0,
                'standing': 0, 'biking': 6, 'upstairs': 2, 'downstairs': 3}

_SHOAIB_META = {}   # align -> {"pos": [N], "subj": [N]}, filled by load_shoaib_raw

def _shoaib_label(s):
    s = str(s).strip().lower()
    for k, v in SHOAIB_LABEL.items():
        if k in s:
            return v
    return -1

def load_shoaib_raw(align, seq_len=XDOMAIN_SEQ_LEN, path=SHOAIB_PATH,
                    win=128, step=64):
    if not os.path.exists(path):
        print(f"  [Shoaib] path not found: {path}"); return None, None
    files = sorted(glob.glob(os.path.join(path, '**', '*.csv'), recursive=True))
    if not files:
        print("  [Shoaib] no CSV files found under path."); return None, None
    Xs, ys, ps, subj = [], [], [], []
    for f_i, fp in enumerate(files):
        raw = pd.read_csv(fp, header=None, low_memory=False, dtype=str)
        hdr_row = None
        for r in range(min(5, len(raw))):
            if (raw.iloc[r].astype(str).str.strip() == 'Ax').any():
                hdr_row = r; break
        if hdr_row is None:
            print(f"  [Shoaib] {os.path.basename(fp)}: no 'Ax' header found, skipped.")
            continue
        names = raw.iloc[hdr_row].astype(str).str.strip().tolist()
        data = raw.iloc[hdr_row + 1:].reset_index(drop=True)
        ax_cols = [i for i, n in enumerate(names) if n == 'Ax']
        gx_cols = [i for i, n in enumerate(names) if n == 'Gx']
        # activity label = last column
        lab = data.iloc[:, -1].map(_shoaib_label).values
        n_pos = min(len(ax_cols), len(gx_cols))
        if n_pos == 0:
            print(f"  [Shoaib] {os.path.basename(fp)}: no sensor blocks, skipped.")
            continue
        for p in range(n_pos):
            try:
                acc = data.iloc[:, ax_cols[p]:ax_cols[p]+3].astype(float).values
                gyr = data.iloc[:, gx_cols[p]:gx_cols[p]+3].astype(float).values
            except Exception:
                continue
            # run-length segments of constant, valid label
            change = np.where(np.diff(lab) != 0)[0] + 1
            bounds = np.concatenate([[0], change, [len(lab)]])
            for b0, b1 in zip(bounds[:-1], bounds[1:]):
                if lab[b0] < 0 or (b1 - b0) < win:
                    continue
                a_seg, g_seg = acc[b0:b1], gyr[b0:b1]
                ok = np.isfinite(a_seg).all(1) & np.isfinite(g_seg).all(1)
                a_seg, g_seg = a_seg[ok], g_seg[ok]
                for s in range(0, len(a_seg) - win + 1, step):
                    Xs.append(raw_from_arrays(a_seg[s:s+win].astype(np.float32),
                                              g_seg[s:s+win].astype(np.float32),
                                              align, has_gravity=True, seq_len=seq_len))
                    ys.append(int(lab[b0])); ps.append(p); subj.append(f_i)
    if not Xs:
        print("  [Shoaib] parsed 0 windows -- check the file format notes above.")
        return None, None
    X, y = np.asarray(Xs, np.float32), np.asarray(ys)
    # side-channel: body position + subject per window
    _SHOAIB_META[align] = {"pos": np.asarray(ps), "subj": np.asarray(subj)}
    print(f"  [Shoaib] files={len(files)}  windows={len(X)}  "
          f"positions={len(np.unique(ps))}  "
          f"classes={dict(pd.Series(y).value_counts().sort_index())}")
    return X, y

# --- HHAR / UCI raw at cross-domain geometry
_HHAR_RAW_CACHE = {}
def _hhar_xd_raw(align):
    if align not in _HHAR_RAW_CACHE:
        acc_raw, gyro_raw = load_hhar_phone()
        Xh, yh, _ = window_hhar_raw(acc_raw, gyro_raw, align,
                                    window_sec=XDOMAIN_WINDOW_SEC,
                                    seq_len=XDOMAIN_SEQ_LEN)
        _HHAR_RAW_CACHE[align] = (Xh, yh)
    return _HHAR_RAW_CACHE[align]

DATASETS = {
    'hhar':   lambda align: _hhar_xd_raw(align),
    'uci':    lambda align: build_uci_raw(align, seq_len=XDOMAIN_SEQ_LEN),
    'motion': lambda align: load_motionsense_raw(align),
    'shoaib': lambda align: load_shoaib_raw(align),
}

_DS_CACHE = {}
def get_dataset_raw(name, align):
    key = (name, align)
    if key not in _DS_CACHE:
        print(f"[data] loading {name} (align={align}) ...")
        _DS_CACHE[key] = DATASETS[name](align)
    return _DS_CACHE[key]


TRANSFER_PAIRS = [
    ("hhar", "uci"), ("uci", "hhar"),
    ("hhar", "motion"), ("motion", "hhar"),
    ("hhar", "shoaib"), ("shoaib", "hhar"),
    ("uci", "motion"), ("motion", "uci"),
    ("uci", "shoaib"), ("shoaib", "uci"),
    ("motion", "shoaib"), ("shoaib", "motion"),
]
ALL_DATASET_NAMES = ["hhar", "uci", "motion", "shoaib"]
TRANSFER_CONFIGS = [
    {"name": "T1 XGBoost",              "kind": "t1"},
    {"name": "T2 gravON [source]",      "kind": "t2", "align": True, "scope": "source", "aug": "off"},
    {"name": "T2 gravON [multi-src]",   "kind": "t2", "align": True, "scope": "multi",  "aug": "off"},
    {"name": "T2 gravON [pooled/UDA]",  "kind": "t2", "align": True, "scope": "pooled", "aug": "off"},
    {"name": "T2+Aug gravON [source]",  "kind": "t2", "align": True, "scope": "source", "aug": "search"},
    # Soft-voting
    {"name": "T1+T2 ensemble [source]", "kind": "ens", "align": True, "scope": "source", "aug": "search"},
]

def _features_df_from_raw(X_raw):
    """Tier-1 features from rate-consistent raw windows (acc re-scaled to m/s^2
    so the gravity-centering branch in extract_sensor_features behaves as in the
    original pipeline)."""
    rows = []
    for w in X_raw:
        a = pd.DataFrame(w[:, 0:3] * G0, columns=['x', 'y', 'z'])
        g = pd.DataFrame(w[:, 3:6],      columns=['x', 'y', 'z'])
        feats, _ = features_from_windows(a, g, sample_hz=TARGET_HZ)
        feats['carrying_load'] = 0
        rows.append(feats)
    return pd.DataFrame(rows).fillna(0)

_T1_FEAT_CACHE = {}
def _t1_features(name, X_raw):
    """Tier-1 features with a DISK cache: extraction is the slow CPU stage
    (~minutes for 48k Shoaib windows) and was previously recomputed every
    session. Keyed by dataset name + cache version; window definitions are
    deterministic per dataset, so the cache is stable across sessions."""
    if name in _T1_FEAT_CACHE:
        return _T1_FEAT_CACHE[name]
    fp = os.path.join(_cache_dir(), f"t1feat_{CACHE_VERSION}_{name}.joblib")
    if RESUME and os.path.exists(fp):
        try:
            df = joblib.load(fp)
            if len(df) == len(X_raw):
                print(f"    [cache] T1 features '{name}' loaded from disk ({len(df)} rows)")
                _T1_FEAT_CACHE[name] = df
                return df
            print(f"    [cache] T1 features '{name}' stale (rows {len(df)} != {len(X_raw)}) -> recompute")
        except Exception as e:
            print(f"    [cache] T1 features '{name}' unreadable ({e}) -> recompute")
    print(f"    [T1] extracting features for {name} ({len(X_raw)} windows) ...")
    df = _features_df_from_raw(X_raw)
    try:
        joblib.dump(df, fp)
    except Exception as e:
        print(f"    [warn] could not save T1 feature cache: {e}")
    _T1_FEAT_CACHE[name] = df
    return df

def _macro_f1(y_true, y_pred, labels):
    out = {"f1": f1_score(y_true, y_pred, labels=labels, average='macro')}
    if 2 in labels and 3 in labels:
        out["stairs"] = f1_score(y_true, y_pred, labels=[2, 3], average='macro')
    else:
        out["stairs"] = float('nan')
    return out

def transfer_eval_t2(enc, Xs, ys, Xt, yt, labels, aug_mode="off",
                     fixed_policy=None, seeds=SEEDS, report=False):
    lmap = {c: i for i, c in enumerate(labels)}; inv = {i: c for c, i in lmap.items()}
    y_tr = np.array([lmap[v] for v in ys])
    f1s, sts = [], []
    for si, sd in enumerate(seeds):
        m = train_classifier(enc, Xs, y_tr, len(labels), aug_mode=aug_mode,
                             fixed_policy=fixed_policy, seed=sd)
        pred = np.array([inv[p] for p in predict(m, Xt)])
        r = _macro_f1(yt, pred, labels)
        f1s.append(r["f1"]); sts.append(r["stairs"])
        if report and si == 0:
            per = f1_score(yt, pred, labels=labels, average=None)
            print("      per-class F1 (seed %d): %s" % (
                sd, {CLASS_NAMES.get(l, l): round(float(v), 3)
                     for l, v in zip(labels, per)}))
    return (float(np.mean(f1s)), float(np.std(f1s)),
            float(np.nanmean(sts)), float(np.nanstd(sts)))

def run_transfer_matrix(pairs=None, configs=None):
    pairs = pairs or TRANSFER_PAIRS
    configs = configs or TRANSFER_CONFIGS
    results, enc_cache, pol_cache, pol_src = [], {}, {}, {}
    tm_fp = os.path.join(SAVE_PATH, f"transfer_matrix_partial_{CACHE_VERSION}.csv")
    done = set()
    if RESUME and os.path.exists(tm_fp):
        prev = pd.read_csv(tm_fp)
        results = prev.to_dict('records')
        done = set(zip(prev['pair'], prev['config']))
        print(f"[resume] transfer matrix: {len(done)} cells already done, skipping them.")
    for (src, tgt) in pairs:
        pair_name = f"{src}->{tgt}"
        if all((pair_name, cfg["name"]) in done for cfg in configs):
            print(f"[resume] pair {pair_name}: all configs done, skipping data load.")
            continue
        Xsrc = get_dataset_raw(src, True);  Xtgt = get_dataset_raw(tgt, True)
        if Xsrc[0] is None or Xtgt[0] is None:
            print(f"[transfer] SKIP {src}->{tgt}: dataset missing."); continue
        Xs_a, ys_a = Xsrc; Xt_a, yt_a = Xtgt
        labels = sorted(int(c) for c in (set(np.unique(ys_a)) & set(np.unique(yt_a))))
        if len(labels) < 2:
            print(f"[transfer] SKIP {src}->{tgt}: <2 shared classes."); continue
        ms = np.isin(ys_a, labels); mt = np.isin(yt_a, labels)
        Xs, ys = Xs_a[ms], ys_a[ms]; Xt, yt = Xt_a[mt], yt_a[mt]
        print(f"\n[transfer] {src} -> {tgt} | shared={labels} | "
              f"train n={len(Xs)}  test n={len(Xt)}")
        # unaligned copies for T1
        Xs_u = get_dataset_raw(src, False)[0]; Xt_u = get_dataset_raw(tgt, False)[0]
        ys_u = get_dataset_raw(src, False)[1]; yt_u = get_dataset_raw(tgt, False)[1]
        msu = np.isin(ys_u, labels); mtu = np.isin(yt_u, labels)

        for cfg in configs:
            policy_used = "n/a"
            if (pair_name, cfg["name"]) in done:
                print(f"    [resume] skip {pair_name} / {cfg['name']}"); continue
            if cfg["kind"] == "t1":
                Ftr = _t1_features(f"{src}_u", Xs_u).iloc[np.where(msu)[0]]
                Fte = _t1_features(f"{tgt}_u", Xt_u).iloc[np.where(mtu)[0]]
                lmap = {c: i for i, c in enumerate(labels)}
                inv = {i: c for c, i in lmap.items()}
                y_map = pd.Series(ys_u[msu]).map(lmap)
                # class-balanced sample weights
                freq = y_map.value_counts()
                sw = y_map.map(lambda c: len(y_map) / (len(labels) * freq[c])).values
                f1s, sts = [], []
                for sd in SEEDS:
                    mdl = make_xgb(len(labels)); mdl.set_params(random_state=sd)
                    mdl.fit(Ftr, y_map, sample_weight=sw)
                    pred = pd.Series(mdl.predict(Fte)).map(inv).values
                    r = _macro_f1(yt_u[mtu], pred, labels)
                    f1s.append(r["f1"]); sts.append(r["stairs"])
                f1m, f1sd = float(np.mean(f1s)), float(np.std(f1s))
                stm, stsd = float(np.nanmean(sts)), float(np.nanstd(sts))
            elif cfg["kind"] == "ens":
                assert len(Xt) == int(mtu.sum()), "aligned/unaligned window mismatch"
                align, scope, aug = cfg["align"], cfg["scope"], cfg["aug"]
                ekey = ('src', src, align)
                enc = enc_cache.get(ekey)
                if enc is None:
                    print("    [ens] source encoder not in cache -> pretraining ...")
                    enc = cached_pretrain(f"tm_src_{src}_{align}", Xs, verbose=False)
                    enc_cache[ekey] = enc
                pol = pol_cache.get(ekey)
                if pol is None:
                    if len(Xs) < MIN_POLICY_N:
                        pol = DEFAULT_POLICY
                        pol_src[ekey] = "fixed-default(small-source)"
                    else:
                        lm0 = {c: i for i, c in enumerate(labels)}
                        pol = cached_policy(f"tm_src_{src}_{align}", enc, Xs,
                                            np.array([lm0[v] for v in ys]), len(labels))
                        if pol.max() < POLICY_MIN_PEAK / len(pol):
                            pol = DEFAULT_POLICY
                            pol_src[ekey] = "fixed-default(search~uniform)"
                        else:
                            pol_src[ekey] = "searched(AutoAugHAR)"
                    pol_cache[ekey] = pol
                policy_used = pol_src.get(ekey, "unknown")
                Ftr = _t1_features(f"{src}_u", Xs_u).iloc[np.where(msu)[0]]
                Fte = _t1_features(f"{tgt}_u", Xt_u).iloc[np.where(mtu)[0]]
                lmap = {c: i for i, c in enumerate(labels)}
                inv = {i: c for c, i in lmap.items()}
                y_map = pd.Series(ys_u[msu]).map(lmap)
                freq = y_map.value_counts()
                sw = y_map.map(lambda c: len(y_map) / (len(labels) * freq[c])).values
                y_tr2 = np.array([lmap[v] for v in ys])
                f1s, sts = [], []
                for sd in SEEDS:
                    mdl = make_xgb(len(labels)); mdl.set_params(random_state=sd)
                    mdl.fit(Ftr, y_map, sample_weight=sw)
                    p1 = mdl.predict_proba(Fte)              # [N, n_labels]
                    m2 = train_classifier(enc, Xs, y_tr2, len(labels), aug_mode="fixed",
                                          fixed_policy=pol, seed=sd)
                    p2 = predict_proba_t2(m2, Xt)            # [N, n_labels]
                    pred = np.array([inv[p] for p in (0.5 * p1 + 0.5 * p2).argmax(1)])
                    r = _macro_f1(yt, pred, labels)
                    f1s.append(r["f1"]); sts.append(r["stairs"])
                f1m, f1sd = float(np.mean(f1s)), float(np.std(f1s))
                stm, stsd = float(np.nanmean(sts)), float(np.nanstd(sts))
            else:
                align, scope, aug = cfg["align"], cfg["scope"], cfg["aug"]
                if scope == "source":
                    ekey = ('src', src, align);  etag = f"tm_src_{src}_{align}"
                elif scope == "pooled":
                    ekey = ('pool', frozenset({src, tgt}), align)
                    etag = f"tm_pool_{'-'.join(sorted({src, tgt}))}_{align}"
                else:                                    # 'multi': all datasets but target
                    ekey = ('multi', tgt, align);        etag = f"tm_multi_no-{tgt}_{align}"
                if ekey not in enc_cache:
                    if scope == "source":
                        pool = Xs
                    elif scope == "pooled":
                        pool = np.concatenate([Xs, Xt], 0)
                    else:
                        parts, rng = [], np.random.RandomState(SEED)
                        for nm in ALL_DATASET_NAMES:
                            if nm == tgt:
                                continue
                            d = get_dataset_raw(nm, align)
                            if d[0] is None:
                                continue
                            Xd = d[0]
                            if len(Xd) > MULTI_CAP:         # balance: cap each dataset
                                Xd = Xd[rng.choice(len(Xd), MULTI_CAP, replace=False)]
                            parts.append(Xd)
                        pool = np.concatenate(parts, 0)
                    print(f"    [T2] encoder {etag} (pool={len(pool)}) ...")
                    enc_cache[ekey] = cached_pretrain(etag, pool, verbose=False)
                enc = enc_cache[ekey]
                pol = None
                if aug == "search":
                    pkey = ekey
                    if pkey not in pol_cache:
                        if len(Xs) < MIN_POLICY_N:
                            print(f"    [T2] source too small for policy search "
                                  f"(n={len(Xs)} < {MIN_POLICY_N}) -> DEFAULT_POLICY")
                            pol_cache[pkey] = DEFAULT_POLICY
                            pol_src[pkey] = "fixed-default(small-source)"
                        else:
                            lm = {c: i for i, c in enumerate(labels)}
                            print(f"    [T2] aug policy for {etag} ...")
                            learned = cached_policy(
                                etag, enc, Xs, np.array([lm[v] for v in ys]), len(labels))
                            if learned.max() < POLICY_MIN_PEAK / len(learned):
                                print("    [T2] search stayed ~uniform -> DEFAULT_POLICY")
                                learned = DEFAULT_POLICY
                                pol_src[pkey] = "fixed-default(search~uniform)"
                            else:
                                pol_src[pkey] = "searched(AutoAugHAR)"
                            pol_cache[pkey] = learned
                    pol = pol_cache[pkey]
                    policy_used = pol_src.get(pkey, "unknown")
                eff = "fixed" if aug == "search" else aug
                f1m, f1sd, stm, stsd = transfer_eval_t2(
                    enc, Xs, ys, Xt, yt, labels, aug_mode=eff, fixed_policy=pol,
                    report=True)
            results.append({"pair": f"{src}->{tgt}", "config": cfg["name"],
                            "shared_classes": str(labels), "policy_used": policy_used,
                            "macroF1": f1m, "macroF1_std": f1sd,
                            "stairsF1": stm, "stairsF1_std": stsd})
            print(f"    {cfg['name']:28s} F1={f1m:.3f}±{f1sd:.3f}  "
                  f"stairs={stm:.3f}±{stsd:.3f}")
            try:
                pd.DataFrame(results).to_csv(tm_fp, index=False)
            except Exception as e:
                print(f"    [warn] save failed: {e}")

    df = pd.DataFrame(results)
    if not df.empty:
        disp = df.copy()
        disp["macro-F1"] = disp.apply(lambda r: f"{r.macroF1:.3f} ± {r.macroF1_std:.3f}", axis=1)
        disp["stairs-F1"] = disp.apply(
            lambda r: ("--" if np.isnan(r.stairsF1) else f"{r.stairsF1:.3f} ± {r.stairsF1_std:.3f}"), axis=1)
        print("\n" + "=" * 84)
        print("TRANSFER MATRIX  (rate-consistent 2.56 s / 20 Hz windows, shared-class macro-F1)")
        print("=" * 84)
        if "policy_used" not in disp.columns:
            disp["policy_used"] = "n/a"
        disp["policy_used"] = disp["policy_used"].fillna("n/a")
        print(disp[["pair", "config", "policy_used", "macro-F1", "stairs-F1"]]
              .to_string(index=False))
        print("=" * 84)
    return df


# =============================================================================
# PROBE : MotionSense walk collapse
# =============================================================================
def _walk_fingerprint(X, y, name, hz=TARGET_HZ):
    w = X[y == 4]
    if len(w) == 0:
        print(f"  [{name}] no walk windows"); return
    mag = np.linalg.norm(w[:, :, 0:3], axis=2)              # [N, L] acc magnitude (g)
    mag = mag - mag.mean(axis=1, keepdims=True)
    fft = np.abs(np.fft.rfft(mag, axis=1))
    frq = np.fft.rfftfreq(mag.shape[1], d=1.0 / hz)
    dom = frq[1 + fft[:, 1:].argmax(axis=1)]
    eng = mag.std(axis=1)
    gyr = np.linalg.norm(w[:, :, 3:6], axis=2).std(axis=1)
    print(f"  [{name:8s}] walk n={len(w):5d} | dom-freq {np.median(dom):.2f} Hz "
          f"(IQR {np.percentile(dom,25):.2f}-{np.percentile(dom,75):.2f}) | "
          f"acc-energy {np.median(eng):.3f} g (IQR {np.percentile(eng,25):.3f}-"
          f"{np.percentile(eng,75):.3f}) | gyro-energy {np.median(gyr):.3f}")

def apply_instance_norm_raw(X):
    """Per-window, per-channel z-score. Window-local statistics only ->
    zero target access. Destroys absolute magnitude; keeps shape/frequency."""
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6
    return ((X - mu) / sd).astype(np.float32)

def probe_policy_confound(n_match=24208, align=True):
    """SUPERSEDED by probe_policy_datasize(retrain_encoder=True), which runs
    this same cut plus a size-matched single-position arm and multiple seeds.
    Kept so earlier results stay reproducible.

    Disentangle WHY the bilevel search converged on Shoaib but not elsewhere:
    data volume (34-41k windows) or placement diversity (5 body positions)?
    Design: subsample Shoaib (shared classes with UCI, i.e. [0,2,3,4]) down to
    n_match -- the size at which the HHAR search stayed ~uniform -- while keeping
    all five positions represented (positions are balanced by construction, so a
    uniform random subsample preserves their proportions in expectation). Then
    pretrain on the subsample and run the search end-to-end.
      * search CONVERGES (peak >= 1.5/17)  -> placement diversity drives it
      * search stays ~uniform              -> data volume is the requirement
    Caveat for the paper: subsampled Shoaib still differs from HHAR in device
    and subject population, so this is a strong probe, not a perfect isolation."""
    print("\n" + "=" * 78)
    print(f"PROBE: policy-search confound (size vs diversity), n_match={n_match}")
    print("=" * 78)
    X, y = get_dataset_raw('shoaib', align)
    if X is None:
        print("Shoaib not available."); return
    labels = [0, 2, 3, 4]                       # match the hhar->uci search setting
    m = np.isin(y, labels)
    Xs, ys = X[m], y[m]
    print(f"  Shoaib shared-class pool: {len(Xs)} windows "
          f"(full-size search peaked at 0.121 and CONVERGED)")
    if len(Xs) <= n_match:
        print("  pool already <= n_match; nothing to subsample."); return
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(Xs), n_match, replace=False)
    Xsub, ysub = Xs[idx], ys[idx]
    print(f"  subsampled to {len(Xsub)} windows "
          f"(size at which the HHAR search stayed ~uniform)")
    enc = cached_pretrain(f"probe_shoaib_sub{n_match}_{align}", Xsub, verbose=False)
    lm = {c: i for i, c in enumerate(labels)}
    pol = learn_policy(enc, Xsub, np.array([lm[v] for v in ysub]), len(labels))
    thr = POLICY_MIN_PEAK / len(pol)
    top = {LABEL_PRESERVING_OPS[j][0]: round(float(pol[j]), 3)
           for j in np.argsort(-pol)[:5]}
    print(f"  search peak = {pol.max():.3f}  (uniform = {1/len(pol):.3f}, "
          f"convergence threshold = {thr:.3f})")
    print(f"  top-5 ops: {top}")
    if pol.max() >= thr:
        print("  VERDICT: search CONVERGED at HHAR-matched size ->")
        print("           placement DIVERSITY drives search convergence.")
    else:
        print("  VERDICT: search stayed ~uniform at HHAR-matched size ->")
        print("           data VOLUME is the binding requirement.")
    print("=" * 78)


def probe_motion(align=True):
    """Run after the matrix. Prints P1 fingerprints for all datasets' walk
    windows, then P2: hhar->motion with and without instance norm, per-class."""
    print("\n" + "=" * 78)
    print("PROBE: MotionSense walk collapse")
    print("=" * 78)
    data = {nm: get_dataset_raw(nm, align) for nm in ALL_DATASET_NAMES}
    print("P1 -- walk fingerprints (dominant cadence vs signal energy):")
    for nm, (X, y) in data.items():
        if X is not None:
            _walk_fingerprint(X, y, nm)
    Xs, ys = data['hhar']; Xt, yt = data['motion']
    if Xs is None or Xt is None:
        print("P2 skipped: dataset missing."); return
    labels = sorted(int(c) for c in (set(np.unique(ys)) & set(np.unique(yt))))
    ms, mt = np.isin(ys, labels), np.isin(yt, labels)
    Xs, ys, Xt, yt = Xs[ms], ys[ms], Xt[mt], yt[mt]
    enc = cached_pretrain(f"tm_src_hhar_{align}", Xs, verbose=False)
    print("\nP2 -- instance-norm ablation (hhar->motion, T2+FixedAug source):")
    for tag, fn in [("raw", lambda a: a), ("instnorm", apply_instance_norm_raw)]:
        Xs_v, Xt_v = fn(Xs), fn(Xt)
        e = enc if tag == "raw" else cached_pretrain(f"probe_hhar_instnorm_{align}",
                                                     Xs_v, verbose=False)
        f1m, f1sd, stm, stsd = transfer_eval_t2(
            e, Xs_v, ys, Xt_v, yt, labels, aug_mode="fixed",
            fixed_policy=DEFAULT_POLICY, report=True)
        print(f"  {tag:9s} macro-F1={f1m:.3f}±{f1sd:.3f}  stairs={stm:.3f}±{stsd:.3f}")
    print("Reading: if instnorm recovers walk but dents static, the Motion gap is")
    print("device gain (dynamics), confirming the orientation-vs-dynamics boundary.")
    print("=" * 78)


# Subsampling is stratified over (position x class) cells
def _stratified_subsample(y, pos, n_target, rng, keep_positions=None):
    """Indices of an n_target-sized subset preserving the (position, class)
    composition of the input; optionally restricted to `keep_positions` first."""
    idx = np.arange(len(y))
    if keep_positions is not None:
        idx = idx[np.isin(pos[idx], keep_positions)]
    if n_target >= len(idx):
        return idx
    cells = {}
    for i in idx:
        cells.setdefault((int(pos[i]), int(y[i])), []).append(i)
    total, keep = len(idx), []
    for _, members in sorted(cells.items()):
        take = min(len(members), int(round(n_target * len(members) / total)))
        if take > 0:
            keep.extend(rng.choice(members, take, replace=False))
    keep = np.asarray(keep, dtype=int)
    if len(keep) > n_target:
        keep = rng.choice(keep, n_target, replace=False)
    return np.sort(keep)

def probe_policy_datasize(align=True, seeds=(42,), n_hhar=28914,
                          include_full=False, retrain_encoder=False,
                          downstream=False, downstream_target="motion"):
    """2x2 probe: does the augmentation search need volume or diversity?

    `retrain_encoder=False` (default) holds the encoder FIXED at the cached
    full-Shoaib source encoder, so only the labelled set the search runs on
    changes. This isolates the search itself: a failure cannot be blamed on a
    weaker encoder. `retrain_encoder=True` instead pretrains a fresh encoder per
    arm, which answers the different (also fair) question of what would happen
    if the dataset really were that small end-to-end -- at roughly double the
    compute. Reporting both is stronger than either alone; if time allows, run
    the fixed-encoder version first.

    `seeds` repeats each arm (the search is stochastic; a single run is thin
    evidence). `downstream=True` additionally fine-tunes with each arm's policy
    on Shoaib-><target> to check whether a converged policy still delivers its
    gain -- costs one fine-tune per arm per seed, so it is off by default."""
    print("\n" + "=" * 78)
    print("PROBE: policy-search convergence -- data volume vs placement diversity")
    print("=" * 78)
    X, y = get_dataset_raw('shoaib', align)
    meta = _SHOAIB_META.get(align)
    if X is None or meta is None:
        print("  Shoaib not loaded (or position metadata missing); probe skipped.")
        return None
    pos = meta["pos"]
    if len(pos) != len(y):
        print("  position metadata length mismatch; probe skipped."); return None

    labels = [0, 2, 3, 4, 6]
    m = np.isin(y, labels)
    X, y, pos = X[m], y[m], pos[m]
    lm = {c: i for i, c in enumerate(labels)}
    y_idx = np.array([lm[v] for v in y])

    enc = cached_pretrain(f"tm_src_shoaib_{align}", X, verbose=False)

    positions = np.unique(pos)
    per_pos = {int(p): int((pos == p).sum()) for p in positions}
    by_size = sorted(per_pos, key=per_pos.get, reverse=True)
    p_big = by_size[0]
    n_single = per_pos[p_big]
    p_mid = by_size[:3]
    n_mid = sum(per_pos[p] for p in p_mid)
    rng = np.random.RandomState(SEED)
    print(f"  source n={len(X)}  positions={len(positions)}  per-position={per_pos}")

    arms = [
        ("size-cut",   _stratified_subsample(y, pos, n_hhar,   rng)),          # 5 pos, HHAR volume
        ("mid-5pos",   _stratified_subsample(y, pos, n_mid,    rng)),          # 5 pos, mid volume
        ("mid-3pos",   _stratified_subsample(y, pos, n_mid,    rng, p_mid)),   # 3 pos, same volume
        ("div-cut",    _stratified_subsample(y, pos, n_single, rng, [p_big])), # 1 pos, low volume
    ]
    if include_full:
        arms.insert(0, ("full", np.arange(len(X))))
    print("  arms: " + ", ".join(f"{n}(n={len(i)})" for n, i in arms))

    uniform = 1.0 / len(LABEL_PRESERVING_OPS)
    thr = POLICY_MIN_PEAK / len(LABEL_PRESERVING_OPS)
    rows = []
    for name, idx in arms:
        got = {int(p): int((pos[idx] == p).sum()) for p in np.unique(pos[idx])}
        print(f"\n  --- arm '{name}': n={len(idx)}, positions={len(got)} {got}")
        if retrain_encoder and name != "full":
            enc_arm = cached_pretrain(f"probe_arm_{name}_{len(idx)}_{align}",
                                      X[idx], verbose=False)
        else:
            enc_arm = enc
        for sd in seeds:
            pol = learn_policy(enc_arm, X[idx], y_idx[idx], len(labels), seed=sd)
            peak = float(pol.max())
            top = [LABEL_PRESERVING_OPS[j][0] for j in np.argsort(-pol)[:3]]
            conv = peak >= thr
            print(f"      seed {sd}: peak={peak:.4f} ({peak/uniform:.2f}x uniform)  "
                  f"{'CONVERGED' if conv else 'near-uniform'}  top3={top}")
            row = {"arm": name, "n": len(idx), "positions": len(got), "seed": sd,
                   "encoder": "per-arm" if (retrain_encoder and name != "full") else "fixed",
                   "peak": peak, "peak_over_uniform": peak / uniform,
                   "converged": bool(conv), "top3": ",".join(top)}
            if downstream:
                Xt, yt = get_dataset_raw(downstream_target, align)
                if Xt is not None:
                    sh = sorted(int(c) for c in (set(labels) & set(np.unique(yt))))
                    a = np.isin(y[idx], sh); b = np.isin(yt, sh)
                    f1m, f1sd, _, _ = transfer_eval_t2(
                        enc_arm, X[idx][a], y[idx][a], Xt[b], yt[b], sh,
                        aug_mode="fixed", fixed_policy=pol, seeds=(sd,))
                    row["downstream_f1"] = f1m
                    print(f"                downstream shoaib->{downstream_target}: "
                          f"macro-F1={f1m:.3f}")
            rows.append(row)

    df = pd.DataFrame(rows)
    print("\n" + "=" * 78)
    print("SUMMARY (guard threshold = %.4f; uniform = %.4f)" % (thr, uniform))
    agg = df.groupby("arm", sort=False).agg(
        n=("n", "first"), positions=("positions", "first"),
        peak_mean=("peak", "mean"), peak_min=("peak", "min"),
        converged=("converged", "all"))
    print(agg.to_string())
    print("\nReading:")
    print("  size-cut CONVERGED  -> HHAR-scale volume suffices when 5 positions")
    print("                         are present; placement DIVERSITY is the driver.")
    print("  size-cut ~uniform   -> data VOLUME is the binding requirement.")
    print("  mid-5pos vs mid-3pos-> volume matched, diversity cut: the direct test.")
    print("  reference: full Shoaib (41,005 / 5 pos) converged at peak 0.121.")
    print("=" * 78)
    try:
        df.to_csv(os.path.join(SAVE_PATH, f"policy_probe_{CACHE_VERSION}.csv"), index=False)
    except Exception as e:
        print(f"  [warn] could not save probe CSV: {e}")
    return df


TIER2_RUNS = [
    # --- gravity ablation (pooled) : grav-ON row is also the pooled member below
    {"name": "T2 LIMU-BERT (grav OFF)  [pooled/UDA]",  "align": False, "aug": "off",    "scope": "pooled"},
    {"name": "T2 LIMU-BERT (grav ON)   [pooled/UDA]",  "align": True,  "aug": "off",    "scope": "pooled"},
    # --- source counterpart of grav-ON LIMU-BERT (completes the scope pair)
    {"name": "T2 LIMU-BERT (grav ON)   [source-only]", "align": True,  "aug": "off",    "scope": "source"},
    # --- +AutoAugHAR : source-vs-pooled pair at best alignment
    {"name": "T2 +AutoAugHAR (grav ON) [pooled/UDA]",  "align": True,  "aug": "search", "scope": "pooled"},
    {"name": "T2 +AutoAugHAR (grav ON) [source-only]", "align": True,  "aug": "search", "scope": "source"},
]

def run_head_to_head(acc_raw, gyro_raw, hhar_df, feature_cols):
    set_seed()
    results, done = [], set()
    h2h_fp = os.path.join(SAVE_PATH, "head_to_head_partial.csv")
    if RESUME and os.path.exists(h2h_fp):
        prev = pd.read_csv(h2h_fp)
        results = prev.to_dict('records'); done = set(prev['model'])
        print(f"[resume] head-to-head: {len(done)} rows already done, skipping them.")

    if "T1 XGBoost (GenHAR+LLM4HAR)" not in done:
        print("\n[Tier-1] XGBoost LOSO + cross-UCI ...")
        t1_loso_m, t1_loso_s = tier1_loso(hhar_df, feature_cols)
        X_uci_feat, y_uci_feat = build_uci_features(feature_cols)
        t1_f4, t1_f4s, t1_fst, t1_fsts = tier1_cross_uci(hhar_df, feature_cols,
                                                         X_uci_feat, y_uci_feat, seeds=SEEDS)
        results.append({"model": "T1 XGBoost (GenHAR+LLM4HAR)", "align": "n/a", "aug": "n/a",
                        "scope": "n/a", "loso_acc": t1_loso_m, "loso_std": t1_loso_s,
                        "uci_4f1": t1_f4, "uci_4f1_std": t1_f4s,
                        "uci_stairs_f1": t1_fst, "uci_stairs_std": t1_fsts})
        pd.DataFrame(results).to_csv(h2h_fp, index=False)

    in_raw, in_enc = {}, {}       # in_enc keyed by align (HHAR-only pretrain)
    xd_raw, xd_enc = {}, {}       # xd_enc keyed by (align, scope)
    policy_cache = {}             # align -> learned AutoAugHAR policy (searched once)
    loso_cache = {}               # (align, eff_aug) -> (loso_m, loso_s)  [scope-independent]

    for spec in TIER2_RUNS:
        align, aug, scope = spec["align"], spec["aug"], spec["scope"]
        if spec["name"] in done:
            print(f"[resume] skip completed row: {spec['name']}"); continue

        # -- in-domain windows + HHAR-only encoder
        if align not in in_raw:
            print(f"\n[Tier-2] in-domain HHAR windows (align={align}, L={SEQ_LEN}) ...")
            in_raw[align] = window_hhar_raw(acc_raw, gyro_raw, align,
                                            window_sec=WINDOW_SEC, seq_len=SEQ_LEN)
            print(f"    HHAR raw: {len(in_raw[align][0])}")
        Xh, yh, gh = in_raw[align]
        if align not in in_enc:
            print(f"[Tier-2] pretraining in-domain encoder (align={align}, HHAR-only, "
                  f"pool={len(Xh)}) ...")
            in_enc[align] = cached_pretrain(f'in_{align}', Xh)

        # -- cross-domain windows
        if align not in xd_raw:
            print(f"[Tier-2] cross-domain windows (align={align}, L={XDOMAIN_SEQ_LEN}) ...")
            Xh2, yh2, _ = window_hhar_raw(acc_raw, gyro_raw, align,
                                          window_sec=XDOMAIN_WINDOW_SEC, seq_len=XDOMAIN_SEQ_LEN)
            Xu, yu = build_uci_raw(align, seq_len=XDOMAIN_SEQ_LEN)
            xd_raw[align] = (Xh2, yh2, Xu, yu)
            print(f"    HHAR raw: {len(Xh2)}  |  UCI raw: {len(Xu)}")
        Xh2, yh2, Xu, yu = xd_raw[align]
        key = (align, scope)
        if key not in xd_enc:
            pool = Xh2 if scope == "source" else np.concatenate([Xh2, Xu], axis=0)
            print(f"[Tier-2] pretraining cross-domain encoder (align={align}, scope={scope}, "
                  f"pool={len(pool)}) ...")
            xd_enc[key] = cached_pretrain(f'xd_{align}_{scope}', pool)

        # -- learn AutoAugHAR policy per (align, scope)
        if aug == "search" and key not in policy_cache:
            print(f"[Tier-2] learning AutoAugHAR policy (align={align}, scope={scope}) "
                  f"-> applied fixed across folds ...")
            cls_x = sorted(np.unique(yh2)); lm = {c: i for i, c in enumerate(cls_x)}
            policy_cache[key] = cached_policy(f'xd_{align}_{scope}', xd_enc[key], Xh2,
                                              np.array([lm[v] for v in yh2]), len(cls_x))
        eff_aug = "fixed" if aug == "search" else aug
        eff_pol = policy_cache.get(key) if aug == "search" else None

        # -- evaluate this row
        print(f"[Tier-2] evaluating: {spec['name']} ...")
        lkey = (align, eff_aug)
        if lkey not in loso_cache:
            loso_cache[lkey] = tier2_loso(in_enc[align], Xh, yh, gh,
                                          aug_mode=eff_aug, fixed_policy=eff_pol)
        loso_m, loso_s = loso_cache[lkey]
        f4, f4s, fst, fsts = tier2_cross_uci(xd_enc[key], Xh2, yh2, Xu, yu,
                                             aug_mode=eff_aug, fixed_policy=eff_pol,
                                             seeds=SEEDS, verbose=True)
        results.append({"model": spec["name"], "align": str(align), "aug": aug,
                        "scope": scope, "loso_acc": loso_m, "loso_std": loso_s,
                        "uci_4f1": f4, "uci_4f1_std": f4s,
                        "uci_stairs_f1": fst, "uci_stairs_std": fsts})
        try:
            pd.DataFrame(results).to_csv(os.path.join(SAVE_PATH, "head_to_head_partial.csv"),
                                         index=False)
            print(f"    [saved] {len(results)} rows -> head_to_head_partial.csv")
        except Exception as e:
            print(f"    [warn] could not save partial results: {e}")

    # ---- summary table -------------------------------------------------------
    df = pd.DataFrame(results)
    disp = df.copy()
    disp["in-domain LOSO acc"] = disp.apply(lambda r: f"{r.loso_acc:.3f} ± {r.loso_std:.3f}", axis=1)
    disp["cross-UCI 4-class F1"] = disp.apply(lambda r: f"{r.uci_4f1:.3f} ± {r.uci_4f1_std:.3f}", axis=1)
    disp["cross-UCI stairs F1"] = disp.apply(lambda r: f"{r.uci_stairs_f1:.3f} ± {r.uci_stairs_std:.3f}", axis=1)
    disp = disp[["model", "align", "aug", "scope", "in-domain LOSO acc",
                 "cross-UCI 4-class F1", "cross-UCI stairs F1"]]
    print("\n" + "=" * 92)
    print("HEAD-TO-HEAD SUMMARY  (rate-consistent: in-domain @5s/20Hz, cross-domain @2.56s/20Hz)")
    print("=" * 92)
    print(disp.to_string(index=False))
    print("=" * 92)
    print("Reading:")
    print("  * grav OFF vs ON     -> value of gravity canonicalization for the deep model")
    print("  * +AutoAugHAR vs off -> augmentation lift on top of the better alignment")
    print("  * pooled vs source   -> cross-UCI gap = value of unlabeled TARGET signal")
    print("  NOTE: LOSO uses an HHAR-only encoder, so it is identical within a source/")
    print("        pooled pair by construction -- scope moves the CROSS-UCI column only.")
    return df


# =============================================================================
# ENTRY POINT
# =============================================================================
def _device_banner():
    if DEVICE.type == "cuda":
        print(f"[device] CUDA active: {torch.cuda.get_device_name(0)}  (Tier-2 runs on GPU)")
    elif DEVICE.type == "mps":
        print("[device] Apple MPS active (Tier-2 runs on the Mac GPU).")
        print("         If the one-time AutoAugHAR policy search hits a missing MPS kernel,")
        print("         it automatically retries on CPU and then continues on MPS.")
    else:
        print("[device] WARNING: no GPU found -> Tier-2 runs on CPU (expect a long run).")
        print("         Colab: Runtime > Change runtime type > T4 GPU, then restart.")
    print(f"[env]    {'Colab' if IN_COLAB else 'local'} | DATA_ROOT={DATA_ROOT}")
    print(f"[resume] RESUME={RESUME} -> partial CSVs + enc_cache under {SAVE_PATH}")
    print("[note] Tier-1 (XGBoost + feature extraction) is CPU-bound by design;")
    print("       an idle GPU during the Tier-1 phase is expected, not a problem.")

RUN_ABLATION = False    # stage 1: the 6-row gravity/scope/aug ablation (HHAR<->UCI)
RUN_TRANSFER = False    # stage 2: quartet transfer matrix (full 12 directed pairs)
RUN_PROBE    = False    # stage 3: MotionSense walk-collapse diagnostic
RUN_POLICY_PROBE = False  # stage 4: search convergence -- volume vs diversity

def main():
    _device_banner()
    acc_raw, gyro_raw = load_hhar_phone()
    if acc_raw is None or gyro_raw is None:
        print("HHAR phone CSVs not found — check ACTIVITY_PATH."); return
    print(f"HHAR acc rows {len(acc_raw):,} | gyro rows {len(gyro_raw):,}")

    if RUN_ABLATION:
        print("Windowing HHAR (features) ...")
        hhar_df = pd.DataFrame(window_hhar(acc_raw, gyro_raw)).fillna(0)
        print(f"HHAR feature windows: {len(hhar_df):,}")
        phyphox_df = load_phyphox_features()
        combined_df = pd.concat([hhar_df, phyphox_df], ignore_index=True) if not phyphox_df.empty else hhar_df
        meta_cols = ['class_label', 'carrying_load', 'user', 'device', 'source', 'window_id']
        feature_cols = [c for c in combined_df.columns if c not in meta_cols] + ['carrying_load']
        run_head_to_head(acc_raw, gyro_raw, hhar_df, feature_cols)

    if RUN_TRANSFER:
        print("\n" + "#" * 84)
        print("# STAGE 2: QUARTET TRANSFER MATRIX")
        print("#" * 84)
        run_transfer_matrix()

    if RUN_PROBE:
        probe_motion()

    if RUN_POLICY_PROBE:
        probe_policy_datasize()


def _smoke_test():
    print("SMOKE TEST (synthetic)")
    set_seed()
    N, ncl = 300, 4
    Xh = np.random.randn(N, SEQ_LEN, N_CHANNELS).astype(np.float32)
    yh = np.random.randint(0, ncl, N)
    for c in range(ncl):
        Xh[yh == c, :, 0] += 0.4 * c
    yh = np.array([[0, 2, 3, 4][v] for v in yh])
    gh = np.array([f"s{ i%3 }" for i in range(N)])
    Xu = np.random.randn(90, SEQ_LEN, N_CHANNELS).astype(np.float32)
    yu = np.array([[0, 2, 3, 4][i % 4] for i in range(90)])
    enc = pretrain_limubert(Xh, epochs=2, bs=64)
    for aug in ["off", "random", "search"]:
        la, ls = tier2_loso(enc, Xh, yh, gh, aug_mode=aug)
        f4, f4s, fs, fss = tier2_cross_uci(enc, Xh, yh, Xu, yu, aug_mode=aug)
        print(f"  aug={aug:6s} loso={la:.3f}±{ls:.3f}  uci4F1={f4:.3f}±{f4s:.3f}  stairsF1={fs:.3f}")
    print("SMOKE TEST PASSED")

## Part 2: revision experiments
Corrected data layer (MotionSense iOS sign, subject IDs), one training path for every arm, all experiments and training-free audits.

In [ ]:
# =============================================================================
#  REVISION EXPERIMENTS  (draft 2 -> draft 3, answers to Bernardo's 71 comments)
#
#  Load order in Colab (GPU runtime, Drive mounted):
#      %run -i -n revision/pipeline_base.py          # loaders, LIMU-BERT, ops
#      %run -i -n revision/revision_experiments.py   # this file
#  then call the stages listed in REVISION_PLAN.md (or run_stage("...")).
#
#  Every training run, whatever the experiment, goes through ONE pretraining
#  function and ONE fine-tuning function and is appended as one row to a
#  long-format CSV under SAVE_PATH/revision/runs/. All paper numbers are then
#  produced by revision_analysis.py from those CSVs, so a number can no longer
#  differ between the abstract, a table and a figure (c25 c30 c41 c57 c63 c65 c73).
#
#  Comment IDs (cNN) refer to GraviHAR_comments_2.docx.
# =============================================================================
import os, re, json, math, time, glob, hashlib, platform, random, subprocess, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

_REQUIRED = ["LimuBERT", "LimuEncoder", "HARModel", "GRUClassifier", "span_mask", "op_jitter", "op_rot_z",
             "_rot_to_vertical", "set_seed", "DEVICE", "load_hhar_phone", "_uci_load",
             "_resample", "UCI_PATH", "MS_PATH", "SHOAIB_PATH", "SAVE_PATH", "make_xgb",
             "_features_df_from_raw", "LABEL_PRESERVING_OPS", "train_classifier",
             "PRETRAIN_EPOCHS", "PRETRAIN_BS", "PRETRAIN_LR", "FINETUNE_EPOCHS",
             "FINETUNE_BS", "FINETUNE_LR", "XDOMAIN_WINDOW_SEC", "XDOMAIN_SEQ_LEN",
             "TARGET_HZ", "STEP_SEC", "MIN_SAMPLES_HHAR", "H_DIM"]
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise RuntimeError("Load revision/pipeline_base.py first (%run -i -n). Missing: "
                       + ", ".join(_missing))

G_REV = 9.80665
REV_VERSION = "rev1"
# d2 = MotionSense converted from the iOS sign convention (c33) + subject IDs kept
DATA_VERSION = "d2"

REV_DIR = os.path.join(SAVE_PATH, "revision")
for _d in ("runs", "cache", "enc", "tables", "unimts"):
    os.makedirs(os.path.join(REV_DIR, _d), exist_ok=True)

# -----------------------------------------------------------------------------
#  Configuration. Everything a reviewer may ask about is here (c40, c48, c67).
#  None of these values is chosen on target data; exp_hparam() re-selects lam and
#  floor with a source-only criterion and reports the full grid on the targets.
# -----------------------------------------------------------------------------
REV = dict(
    lam=0.1,            # consistency weight (Eq. 6)
    warm=0,             # linear ramp length in epochs; 0 = no ramp. The draft text says
                        # a 10-epoch ramp but cell 29 set warmup=0 before E2 -- check the
                        # cached encoder names (_w0_ vs _w10_) and report what was used.
    floor=0.3,          # identity floor: probability that a fine-tuning batch is clean
    view_jitter=0.03,   # jitter added to both consistency views
    pre_epochs=PRETRAIN_EPOCHS, pre_bs=PRETRAIN_BS, pre_lr=PRETRAIN_LR,
    pre_min_epochs=30, pre_patience=8,   # early stop on the SOURCE MLM training loss (c48)
    ft_epochs=FINETUNE_EPOCHS, ft_bs=FINETUNE_BS, ft_lr=FINETUNE_LR,  # fixed, no early stop
    # one run budget for every source (c67): 2 pretraining seeds x 4 fine-tuning seeds
    # for the ablation arms the claims rest on; 2 x 2 for the added baselines
    budget={"pre": (42, 43), "ft": (42, 43, 44, 45)},
    budget_baselines={"pre": (42, 43), "ft": (42, 43)},
    ios_sign_fix=True,  # c33: Core Motion reports -specific force (flat, face up: z=-1 g)
    gate_g=5.0 / G_REV, # gravity gate in g units (acc stored in g)
)

PAIRS = [("hhar", "uci"), ("uci", "hhar"), ("hhar", "motion"), ("motion", "hhar"),
         ("hhar", "shoaib"), ("shoaib", "hhar"), ("uci", "motion"), ("motion", "uci"),
         ("uci", "shoaib"), ("shoaib", "uci"), ("motion", "shoaib"), ("shoaib", "motion")]
DATASETS_REV = ["hhar", "uci", "motion", "shoaib"]

CLS = {0: "static", 2: "up", 3: "down", 4: "walk", 5: "run", 6: "bike",
       8: "sit", 9: "stand", 10: "lie"}
STAIRS = (2, 3)


def _p(*a):
    print(*a, flush=True)


# =============================================================================
#  1. DATA LAYER v2  (c33 c44 c45)
# -----------------------------------------------------------------------------
#  Every loader returns UNALIGNED 20 Hz windows of 2.56 s (L=51), acc in g and
#  gyro in rad/s, in the ANDROID sign convention (flat, face up: acc_z = +1 g),
#  plus subject IDs, two label vectors and per-window metadata:
#      y_merged  : static / up / down / walk / run / bike   (the paper's label set)
#      y_posture : sit / stand / lie kept apart             (c44)
#  -1 marks a window that has no label in that mode (e.g. UCI LAYING in merged
#  mode -- this is why UCI has 8,355 rather than 10,299 windows, c45).
#  Canonicalization and every other input transform is applied afterwards by
#  prep(), so all arms see exactly the same windows.
# =============================================================================
HHAR_POSTURE = {"sit": 8, "stand": 9}
UCI_MERGED = {1: 4, 2: 2, 3: 3, 4: 0, 5: 0}
UCI_POSTURE = {1: 4, 2: 2, 3: 3, 4: 8, 5: 9, 6: 10}
MS_MERGED = {"dws": 3, "ups": 2, "wlk": 4, "jog": 5, "sit": 0, "std": 0}
MS_POSTURE = {"dws": 3, "ups": 2, "wlk": 4, "jog": 5, "sit": 8, "std": 9}
SHOAIB_MERGED = {"walking": 4, "jogging": 5, "running": 5, "sitting": 0, "standing": 0,
                 "biking": 6, "upstairs": 2, "downstairs": 3}
SHOAIB_POSTURE = dict(SHOAIB_MERGED, sitting=8, standing=9)
SHOAIB_POSITIONS = {0: "left_pocket", 1: "right_pocket", 2: "wrist", 3: "upper_arm", 4: "belt"}


def _win20(acc, gyr):
    """[n,3]+[n,3] native-rate arrays -> [51,6] at 20 Hz, acc in g. Same resampler as
    the pipeline (linear interpolation over the window's own duration)."""
    a = _resample(np.asarray(acc, np.float32), XDOMAIN_SEQ_LEN)
    g = _resample(np.asarray(gyr, np.float32), XDOMAIN_SEQ_LEN)
    w = np.concatenate([a, g], axis=1).astype(np.float32)
    w[:, 0:3] /= G_REV
    return w


def _mode_label(v):
    vals, cnt = np.unique(np.asarray(v), return_counts=True)
    return vals[int(np.argmax(cnt))]          # ties -> smallest, as pandas .mode().iloc[0]


def _load_hhar_v2():
    acc_raw, gyro_raw = load_hhar_phone()
    if acc_raw is None or gyro_raw is None:
        raise FileNotFoundError("HHAR phone CSVs not found under ACTIVITY_PATH")
    W, step, mn = XDOMAIN_WINDOW_SEC, STEP_SEC, MIN_SAMPLES_HHAR
    acc_raw = acc_raw.copy()
    acc_raw["posture"] = [HHAR_POSTURE.get(str(s).strip().lower(), -2) for s in acc_raw["gt"]]
    gyro_groups = {k: g for k, g in gyro_raw.groupby(["user", "device"])}
    X, ym, yp, grp, dev, mdl, rate = [], [], [], [], [], [], []
    for (user, device), ag in acc_raw.groupby(["user", "device"]):
        gg = gyro_groups.get((user, device))
        if gg is None or gg.empty:
            continue
        t0 = max(ag["time_sec"].min(), gg["time_sec"].min())
        t1 = min(ag["time_sec"].max(), gg["time_sec"].max())
        if t1 - t0 < W:
            continue
        at, gt = ag["time_sec"].values, gg["time_sec"].values
        axyz, gxyz = ag[["x", "y", "z"]].values, gg[["x", "y", "z"]].values
        lab, pos = ag["class_label"].values, ag["posture"].values
        model = str(ag["model"].iloc[0])
        t = t0
        while t + W <= t1:
            i0, i1 = np.searchsorted(at, t, "left"), np.searchsorted(at, t + W, "left")
            j0, j1 = np.searchsorted(gt, t, "left"), np.searchsorted(gt, t + W, "left")
            if (i1 - i0) >= mn and (j1 - j0) >= mn:
                lm = int(_mode_label(lab[i0:i1]))
                pm = int(_mode_label(pos[i0:i1]))
                X.append(_win20(axyz[i0:i1], gxyz[j0:j1]))
                ym.append(lm)
                yp.append(lm if lm != 0 else (pm if pm >= 0 else -1))
                grp.append(f"hhar_{user}"); dev.append(str(device)); mdl.append(model)
                rate.append((i1 - i0) / W)
            t += step
    X = np.asarray(X, np.float32)
    return dict(X=X, y_merged=np.asarray(ym), y_posture=np.asarray(yp), groups=np.asarray(grp),
                meta=dict(device=np.asarray(dev), model=np.asarray(mdl),
                          native_hz=np.asarray(rate, np.float32)),
                info=dict(native="~50-200 Hz (per phone model)", units="m/s^2, rad/s",
                          convention="Android", placement="smartphones (waist pouch)",
                          overlap=f"{1 - STEP_SEC / W:.1%} (step {STEP_SEC} s)"))


def _load_uci_v2():
    X, ym, yp, grp, split_ = [], [], [], [], []
    for split in ("train", "test"):
        ax, ay, az = (_uci_load(split, f"total_acc_{a}") * G_REV for a in "xyz")
        gx, gy, gz = (_uci_load(split, f"body_gyro_{a}") for a in "xyz")
        yv = np.loadtxt(os.path.join(UCI_PATH, split, f"y_{split}.txt")).astype(int)
        sv = np.loadtxt(os.path.join(UCI_PATH, split, f"subject_{split}.txt")).astype(int)
        for i in range(ax.shape[0]):
            acc = np.stack([ax[i], ay[i], az[i]], 1); gyr = np.stack([gx[i], gy[i], gz[i]], 1)
            X.append(_win20(acc, gyr))
            ym.append(UCI_MERGED.get(int(yv[i]), -1)); yp.append(UCI_POSTURE.get(int(yv[i]), -1))
            grp.append(f"uci_{sv[i]}"); split_.append(split)
    return dict(X=np.asarray(X, np.float32), y_merged=np.asarray(ym), y_posture=np.asarray(yp),
                groups=np.asarray(grp), meta=dict(split=np.asarray(split_)),
                info=dict(native="50 Hz (pre-segmented 128-sample windows)", units="g, rad/s",
                          convention="Android", placement="waist (belt)",
                          overlap="50% (dataset's own segmentation)",
                          note="LAYING excluded in merged mode: 10,299 - 1,944 = 8,355 windows"))


def _load_motion_v2(win=128, step=64):
    path = MS_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    sign = -1.0 if REV["ios_sign_fix"] else 1.0
    need = ["gravity.x", "gravity.y", "gravity.z", "userAcceleration.x", "userAcceleration.y",
            "userAcceleration.z", "rotationRate.x", "rotationRate.y", "rotationRate.z"]
    trial_dirs = []
    for root, dirs, _ in os.walk(path):
        dirs[:] = [d for d in dirs if d != "__MACOSX" and not d.startswith(".")]
        if "__MACOSX" in root:
            continue
        for d in dirs:
            act = d.split("_")[0].lower()
            if act in MS_MERGED:
                trial_dirs.append((os.path.join(root, d), act))
    X, ym, yp, grp, trial = [], [], [], [], []
    for tdir, act in sorted(trial_dirs):
        for fn in sorted(os.listdir(tdir)):
            if not fn.endswith(".csv") or fn.startswith("."):
                continue
            df = pd.read_csv(os.path.join(tdir, fn))
            if not all(c in df.columns for c in need):
                continue
            m = re.search(r"sub_(\d+)", fn)
            subj = f"ms_{m.group(1) if m else fn}"
            # Core Motion: total = gravity + userAcceleration, in g, with the opposite
            # sign to Android (device flat, face up -> gravity.z = -1). Negating all
            # three axes gives the Android specific-force convention. rotationRate is
            # right-handed rad/s on both platforms and is left unchanged.
            acc = sign * (df[need[0:3]].values + df[need[3:6]].values) * G_REV
            gyr = df[need[6:9]].values
            for s in range(0, len(acc) - win + 1, step):
                X.append(_win20(acc[s:s + win], gyr[s:s + win]))
                ym.append(MS_MERGED[act]); yp.append(MS_POSTURE[act])
                grp.append(subj); trial.append(os.path.basename(tdir))
    return dict(X=np.asarray(X, np.float32), y_merged=np.asarray(ym), y_posture=np.asarray(yp),
                groups=np.asarray(grp), meta=dict(trial=np.asarray(trial)),
                info=dict(native="50 Hz", units="g (Core Motion), rad/s",
                          convention="iOS -> converted to Android" if REV["ios_sign_fix"]
                          else "iOS (NOT converted)",
                          placement="front trouser pocket (iPhone 6s)",
                          overlap=f"{1 - step / win:.0%} (win {win}, step {step} @ 50 Hz)"))


def _shoaib_lab(s, table):
    s = str(s).strip().lower()
    for k, v in table.items():
        if k in s:
            return v
    return -1


def _load_shoaib_v2(win=128, step=64):
    path = SHOAIB_PATH
    files = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(path)
    X, ym, yp, grp, pos = [], [], [], [], []
    for f_i, fp in enumerate(files):
        raw = pd.read_csv(fp, header=None, low_memory=False, dtype=str)
        hdr = next((r for r in range(min(5, len(raw)))
                    if (raw.iloc[r].astype(str).str.strip() == "Ax").any()), None)
        if hdr is None:
            continue
        names = raw.iloc[hdr].astype(str).str.strip().tolist()
        data = raw.iloc[hdr + 1:].reset_index(drop=True)
        ax_cols = [i for i, n in enumerate(names) if n == "Ax"]
        gx_cols = [i for i, n in enumerate(names) if n == "Gx"]
        lab_str = data.iloc[:, -1].values
        lm = np.array([_shoaib_lab(s, SHOAIB_MERGED) for s in lab_str])
        lp = np.array([_shoaib_lab(s, SHOAIB_POSTURE) for s in lab_str])
        m = re.search(r"(\d+)", os.path.basename(fp))
        subj = f"shoaib_{m.group(1) if m else f_i}"
        for p in range(min(len(ax_cols), len(gx_cols))):
            try:
                acc = data.iloc[:, ax_cols[p]:ax_cols[p] + 3].astype(float).values
                gyr = data.iloc[:, gx_cols[p]:gx_cols[p] + 3].astype(float).values
            except Exception:
                continue
            change = np.where(np.diff(lm) != 0)[0] + 1
            bounds = np.concatenate([[0], change, [len(lm)]])
            for b0, b1 in zip(bounds[:-1], bounds[1:]):
                if lm[b0] < 0 or (b1 - b0) < win:
                    continue
                a_seg, g_seg = acc[b0:b1], gyr[b0:b1]
                ok = np.isfinite(a_seg).all(1) & np.isfinite(g_seg).all(1)
                a_seg, g_seg = a_seg[ok], g_seg[ok]
                for s in range(0, len(a_seg) - win + 1, step):
                    X.append(_win20(a_seg[s:s + win], g_seg[s:s + win]))
                    ym.append(int(lm[b0])); yp.append(int(lp[b0])); grp.append(subj); pos.append(p)
    return dict(X=np.asarray(X, np.float32), y_merged=np.asarray(ym), y_posture=np.asarray(yp),
                groups=np.asarray(grp), meta=dict(position=np.asarray(pos)),
                info=dict(native="50 Hz", units="m/s^2, rad/s", convention="Android",
                          placement="5 simultaneous: L/R pocket, wrist, upper arm, belt",
                          overlap=f"{1 - step / win:.0%} (win {win}, step {step} @ 50 Hz)"))


# The smoke test replaces these with synthetic generators.
REV_LOADERS = {"hhar": _load_hhar_v2, "uci": _load_uci_v2,
               "motion": _load_motion_v2, "shoaib": _load_shoaib_v2}

_RAW_CACHE = {}


def _fingerprint(X):
    X = np.ascontiguousarray(X)
    step = max(1, len(X) // 64)
    h = hashlib.md5(X[::step].tobytes()).hexdigest()[:8]
    return f"n{len(X)}_{h}"


def load_raw_v2(name):
    """All windows of a dataset (both label modes), cached in memory and on disk."""
    key = (name, DATA_VERSION, REV["ios_sign_fix"])
    if key in _RAW_CACHE:
        return _RAW_CACHE[key]
    tag = f"{name}_{DATA_VERSION}_ios{int(REV['ios_sign_fix'])}"
    fp = os.path.join(REV_DIR, "cache", f"data_{tag}.npz")
    if os.path.exists(fp) and REV_LOADERS.get(name) in (
            _load_hhar_v2, _load_uci_v2, _load_motion_v2, _load_shoaib_v2):
        z = np.load(fp, allow_pickle=True)
        d = dict(X=z["X"], y_merged=z["y_merged"], y_posture=z["y_posture"],
                 groups=z["groups"], meta=z["meta"].item(), info=z["info"].item())
    else:
        _p(f"[data] parsing {name} ...")
        d = REV_LOADERS[name]()
        if REV_LOADERS.get(name) in (_load_hhar_v2, _load_uci_v2, _load_motion_v2, _load_shoaib_v2):
            np.savez(fp, X=d["X"], y_merged=d["y_merged"], y_posture=d["y_posture"],
                     groups=d["groups"], meta=np.array(d["meta"], dtype=object),
                     info=np.array(d["info"], dtype=object))
    d["fp"] = _fingerprint(d["X"])
    _RAW_CACHE[key] = d
    _p(f"[data] {name}: {len(d['X'])} windows, {len(np.unique(d['groups']))} subjects")
    return d


def get_ds(name, label_mode="merged"):
    """-> X_raw [N,51,6] (unaligned, Android convention), y, groups, meta(dict), fp."""
    d = load_raw_v2(name)
    y = d["y_merged"] if label_mode == "merged" else d["y_posture"]
    k = y >= 0
    meta = {kk: np.asarray(v)[k] for kk, v in d["meta"].items()}
    X = d["X"][k]
    return X, y[k], d["groups"][k], meta, f"{d['fp']}_{label_mode}"


# =============================================================================
#  2. INPUT TRANSFORMS  (one implementation, used by every arm and analysis)
# =============================================================================
def gravity_dirs(X):
    """Unit gravity direction g_hat (device frame) and |a_bar| (g units) per window."""
    m = np.asarray(X)[:, :, 0:3].astype(np.float64).mean(1)
    n = np.linalg.norm(m, axis=1)
    return m / np.maximum(n, 1e-12)[:, None], n


def _skew(v):
    z = np.zeros(len(v))
    return np.stack([np.stack([z, -v[:, 2], v[:, 1]], 1),
                     np.stack([v[:, 2], z, -v[:, 0]], 1),
                     np.stack([-v[:, 1], v[:, 0], z], 1)], 1)


def rot_to_vertical_batch(g):
    """Vectorized Eq. 3: the minimal (Rodrigues) rotation with R g = z. Identical to
    the pipeline's _rot_to_vertical, including its antipodal convention: when g = -z
    exactly (s = 0, c < 0) it returns diag(1,-1,-1), a 180 deg turn about x (c38)."""
    g = np.asarray(g, np.float64)
    v = np.cross(g, np.array([0.0, 0.0, 1.0]))
    c = g[:, 2]; s = np.linalg.norm(v, axis=1)
    K = _skew(v)
    coef = np.where(s > 1e-8, (1 - c) / np.maximum(s * s, 1e-300), 0.0)
    R = np.eye(3)[None] + K + (K @ K) * coef[:, None, None]
    sing = s <= 1e-8
    if sing.any():
        R[sing] = np.where((c[sing] > 0)[:, None, None], np.eye(3)[None],
                           np.diag([1.0, -1.0, -1.0])[None])
    return R


def rotate_windows(X, R):
    """Apply a per-window rotation [N,3,3] (or one [3,3]) to acc and gyro."""
    X = np.asarray(X, np.float32)
    R = np.asarray(R, np.float64)
    if R.ndim == 2:
        R = np.broadcast_to(R, (len(X), 3, 3))
    out = X.copy()
    out[:, :, 0:3] = np.einsum("nij,ntj->nti", R, X[:, :, 0:3]).astype(np.float32)
    out[:, :, 3:6] = np.einsum("nij,ntj->nti", R, X[:, :, 3:6]).astype(np.float32)
    return out


def canonicalize(X, return_R=False):
    """Gravity canonicalization (Eqs. 2-4) on the 20 Hz window: g_hat from the window
    mean, Rodrigues rotation to +z, same R on acc and gyro. Windows failing the
    |a_bar| > 5 m/s^2 gate are passed through (fraction reported by audit)."""
    g, n = gravity_dirs(X)
    R = rot_to_vertical_batch(g)
    R[n < REV["gate_g"]] = np.eye(3)
    out = rotate_windows(X, R)
    return (out, R) if return_R else out


def pca_heading(Xc, return_ratio=False):
    """Classical non-learned heading alignment (closest prior work: Gil-Martin et
    al., Sensors 2023; Henpraserttae/Thiemjarus-style PCA): after canonicalization,
    rotate about z so the principal horizontal acceleration axis lands on +x. The
    axis sign is fixed by making the third moment of the projection positive
    (eigenvectors are only defined up to sign). ratio = lambda_min/lambda_max of
    the horizontal covariance; -> 1 means no dominant direction (static windows)."""
    Xc = np.asarray(Xc, np.float32)
    H = Xc[:, :, 0:2].astype(np.float64)
    H = H - H.mean(1, keepdims=True)
    C = np.einsum("nti,ntj->nij", H, H)
    w, V = np.linalg.eigh(C)
    p = V[:, :, 1]
    proj = np.einsum("nti,ni->nt", H, p)
    p = p * np.where((proj ** 3).mean(1) < 0, -1.0, 1.0)[:, None]
    th = np.arctan2(p[:, 1], p[:, 0])
    c, s = np.cos(-th), np.sin(-th)
    Rz = np.zeros((len(Xc), 3, 3)); Rz[:, 0, 0] = c; Rz[:, 0, 1] = -s
    Rz[:, 1, 0] = s; Rz[:, 1, 1] = c; Rz[:, 2, 2] = 1.0
    out = rotate_windows(Xc, Rz)
    ratio = w[:, 0] / np.maximum(w[:, 1], 1e-12)
    return (out, ratio) if return_ratio else out


def mizell_transform(X):
    """Mizell (ISWC 2003) vertical/horizontal decomposition, extended to the gyro:
    [a.g, |a - (a.g)g|, w.g, |w - (w.g)g|]. Invariant to ANY device rotation, but it
    discards the horizontal direction -- the information our method keeps."""
    X = np.asarray(X, np.float32)
    g, _ = gravity_dirs(X)
    out = []
    for sl in (slice(0, 3), slice(3, 6)):
        v = X[:, :, sl].astype(np.float64)
        par = np.einsum("nti,ni->nt", v, g)
        perp = np.linalg.norm(v - par[..., None] * g[:, None, :], axis=-1)
        out += [par, perp]
    return np.stack(out, -1).astype(np.float32)


def _angle(a, b):
    na = np.linalg.norm(a, axis=-1); nb = np.linalg.norm(b, axis=-1)
    cos = (a * b).sum(-1) / np.maximum(na * nb, 1e-12)
    return np.arccos(np.clip(cos, -1.0, 1.0))


def _pad_to(x, L):
    if x.shape[1] == L:
        return x
    return np.concatenate([x, np.repeat(x[:, -1:], L - x.shape[1], axis=1)], 1)


def oit_transform(X):
    """Heuristic orientation-invariant transformation of Yurtman & Barshan (Sensors
    2017), per sensor: norms of v, dv, d2v; angles between successive v, dv, d2v;
    angles between successive cross products of v, dv, d2v -> 9 sequences per sensor,
    18 channels. VERIFY against Sec. 3 of the paper before submission; the
    sequences shorter than L are edge-padded to L."""
    X = np.asarray(X, np.float64)
    L = X.shape[1]
    chans = []
    for sl in (slice(0, 3), slice(3, 6)):
        v = X[:, :, sl]
        d1 = v[:, 1:] - v[:, :-1]
        d2 = d1[:, 1:] - d1[:, :-1]
        seqs = [np.linalg.norm(v, axis=-1), np.linalg.norm(d1, axis=-1), np.linalg.norm(d2, axis=-1),
                _angle(v[:, :-1], v[:, 1:]), _angle(d1[:, :-1], d1[:, 1:]), _angle(d2[:, :-1], d2[:, 1:])]
        for u in (v, d1, d2):
            cr = np.cross(u[:, :-1], u[:, 1:])
            seqs.append(_angle(cr[:, :-1], cr[:, 1:]))
        chans += [_pad_to(s, L) for s in seqs]
    return np.stack(chans, -1).astype(np.float32)


KIND_DIM = {"raw": 6, "canon": 6, "pca": 6, "mizell": 4, "oit": 18}


def prep(X_raw, kind):
    if kind == "raw":
        return np.asarray(X_raw, np.float32)
    if kind == "canon":
        return canonicalize(X_raw)
    if kind == "pca":
        return pca_heading(canonicalize(X_raw))
    if kind == "mizell":
        return mizell_transform(X_raw)
    if kind == "oit":
        return oit_transform(X_raw)
    raise ValueError(kind)


_PREP_CACHE = {}


def prep_cached(name, label_mode, kind):
    X, y, g, meta, fp = get_ds(name, label_mode)
    key = (fp, kind)
    if key not in _PREP_CACHE:
        if len(_PREP_CACHE) > 12:
            _PREP_CACHE.pop(next(iter(_PREP_CACHE)))
        _PREP_CACHE[key] = prep(X, kind)
    return _PREP_CACHE[key], y, g, meta, fp


# =============================================================================
#  3. TRAINING  (one pretraining and one fine-tuning path for every arm)
# =============================================================================
def _haar_rot_torch(B, device):
    q = torch.randn(B, 4, device=device)
    q = q / q.norm(dim=1, keepdim=True)
    w, x, y, z = q.unbind(1)
    return torch.stack([
        torch.stack([1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], 1),
        torch.stack([2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], 1),
        torch.stack([2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)], 1)], 1)


def _apply_R_torch(x, R):
    out = x.clone()
    out[:, :, 0:3] = torch.einsum("bij,blj->bli", R, x[:, :, 0:3])
    out[:, :, 3:6] = torch.einsum("bij,blj->bli", R, x[:, :, 3:6])
    return out


def _axis_angle_torch(axis, th):
    a = axis / axis.norm(dim=1, keepdim=True).clamp(min=1e-9)
    z = torch.zeros_like(a[:, 0])
    K = torch.stack([torch.stack([z, -a[:, 2], a[:, 1]], 1),
                     torch.stack([a[:, 2], z, -a[:, 0]], 1),
                     torch.stack([-a[:, 1], a[:, 0], z], 1)], 1)
    I = torch.eye(3, device=a.device).expand(len(a), 3, 3)
    s = torch.sin(th).view(-1, 1, 1); c = torch.cos(th).view(-1, 1, 1)
    return I + s * K + (1 - c) * torch.bmm(K, K)


def op_yaw(x):
    """Rotation about the z axis, theta ~ U(-180,180): the canonical vertical after
    canonicalization (Eq. 7); the device z axis in the raw frame (arm C)."""
    return op_rot_z(x, 180.0)


def op_so3(x):
    """Haar-uniform random rotation per window (acc and gyro jointly)."""
    return _apply_R_torch(x, _haar_rot_torch(x.size(0), x.device))


def make_axis_op(axis=(1.0, 0.0, 0.0), max_deg=180.0):
    def op(x):
        B = x.size(0)
        th = (torch.rand(B, device=x.device) * 2 - 1) * math.radians(max_deg)
        ax = torch.tensor(axis, dtype=torch.float32, device=x.device).expand(B, 3)
        return _apply_R_torch(x, _axis_angle_torch(ax, th))
    return op


def make_so3_angle_op(max_deg=180.0):
    """Random axis, angle ~ U(-max,max) (the draft's 'SO(3)' control in Fig. 11)."""
    def op(x):
        B = x.size(0)
        th = (torch.rand(B, device=x.device) * 2 - 1) * math.radians(max_deg)
        return _apply_R_torch(x, _axis_angle_torch(torch.randn(B, 3, device=x.device), th))
    return op


AUG_OPS = {None: None, "yaw": op_yaw, "so3": op_so3}


def pretrain_rev(X, objective="mlm", seed=42, lam=None, warm=None, verbose=False):
    """Span-masked reconstruction (LIMU-BERT defaults, not tuned) + optional
    consistency between two independently rotated views: objective 'yaw' rotates
    about z, 'so3' draws Haar-uniform rotations. Early stopping monitors the
    plateau of the reconstruction loss on the SOURCE pool itself (c48)."""
    lam = REV["lam"] if lam is None else lam
    warm = REV["warm"] if warm is None else warm
    s_dim = X.shape[2]
    set_seed(seed)
    model = LimuBERT(s_dim=s_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=REV["pre_lr"])
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE); n = len(Xt)
    op = {"yaw": op_yaw, "so3": op_so3}.get(objective)
    if op is not None and s_dim != 6:
        raise ValueError("rotation consistency needs the 6-channel acc+gyro input")
    best, stall, hist = float("inf"), 0, []
    for ep in range(REV["pre_epochs"]):
        lam_ep = 0.0 if op is None else (lam if warm <= 0 else lam * min(1.0, (ep + 1) / warm))
        model.train(); perm = torch.randperm(n); tm = tc = 0.0
        for i in range(0, n, REV["pre_bs"]):
            xb = Xt[perm[i:i + REV["pre_bs"]].to(DEVICE)]
            xm, mask = span_mask(xb)
            l_mlm = F.mse_loss(model(xm)[mask], xb[mask])
            loss = l_mlm
            if lam_ep > 0:
                e1 = model.encoder(op(op_jitter(xb, REV["view_jitter"]))).mean(1)
                e2 = model.encoder(op(op_jitter(xb, REV["view_jitter"]))).mean(1)
                l_c = 1.0 - F.cosine_similarity(e1, e2, dim=1).mean()
                loss = loss + lam_ep * l_c
                tc += float(l_c) * xb.size(0)
            opt.zero_grad(); loss.backward(); opt.step()
            tm += l_mlm.item() * xb.size(0)
        mlm = tm / n
        hist.append({"ep": ep, "lam": lam_ep, "mlm": mlm, "cons": tc / n})
        if verbose and (ep % 10 == 0):
            _p(f"    [pre] ep {ep:3d} lam {lam_ep:.3f} mlm {mlm:.4f} cons {tc / n:.4f}")
        if mlm < best * 0.995:
            best, stall = mlm, 0
        else:
            stall += 1
            if ep >= REV["pre_min_epochs"] and stall >= REV["pre_patience"]:
                break
    return {k: v.cpu() for k, v in model.encoder.state_dict().items()}, hist


def _enc_path(name, kind, objective, seed, fp, lam, warm, tag_extra=""):
    lam_t = lam if objective != "mlm" else 0.0
    warm_t = warm if objective != "mlm" else 0
    tag = (f"{REV_VERSION}_{name}_{fp}_{kind}_{objective}_l{lam_t}_w{warm_t}_s{seed}"
           f"_e{REV['pre_epochs']}{tag_extra}")
    return tag, os.path.join(REV_DIR, "enc", tag + ".pt")


def get_encoder_rev(name, kind, objective, seed, label_mode="merged", lam=None, warm=None,
                    X_override=None, tag_extra=""):
    """Source-scope encoder: pretrained on the full unlabeled pool of `name` (no
    target data). Cached on Drive; the tag carries the data fingerprint, so the
    MotionSense sign fix cannot silently reuse an old encoder."""
    lam = REV["lam"] if lam is None else lam
    warm = REV["warm"] if warm is None else warm
    if X_override is None:
        X, _, _, _, fp = prep_cached(name, label_mode, kind)
    else:
        X, fp = X_override, _fingerprint(X_override)
    tag, path = _enc_path(name, kind, objective, seed, fp, lam, warm, tag_extra)
    if os.path.exists(path):
        return torch.load(path, map_location="cpu")
    t0 = time.time()
    enc, hist = pretrain_rev(X, objective, seed, lam=lam, warm=warm)
    torch.save(enc, path)
    pd.DataFrame(hist).to_csv(path.replace(".pt", "_log.csv"), index=False)
    _p(f"  [enc] {tag}  n={len(X)}  {len(hist)} ep  {(time.time() - t0) / 60:.1f} min")
    return enc


class GRUHeadAux(nn.Module):
    """The paper's GRU head (20-20-10) with an optional auxiliary vector joined
    before the last layers (used only by the posture side-channel arm, c44)."""
    def __init__(self, h_dim, n_classes, aux_dim=0):
        super().__init__()
        self.g1 = nn.GRU(h_dim, 20, batch_first=True)
        self.g2 = nn.GRU(20, 20, batch_first=True)
        self.g3 = nn.GRU(20, 10, batch_first=True)
        self.drop = nn.Dropout(0.5)
        self.fc1 = nn.Linear(10 + aux_dim, 10); self.fc2 = nn.Linear(10, n_classes)

    def forward(self, e, aux=None):
        h, _ = self.g1(e); h, _ = self.g2(h); h, _ = self.g3(h)
        h = self.drop(h[:, -1])
        if aux is not None:
            h = torch.cat([h, aux], 1)
        return self.fc2(F.relu(self.fc1(h)))


class HARModelRev(nn.Module):
    def __init__(self, n_classes, s_dim=6, aux_dim=0):
        super().__init__()
        self.encoder = LimuEncoder(s_dim=s_dim)
        self.head = GRUHeadAux(H_DIM, n_classes, aux_dim)

    def forward(self, x, aux=None):
        return self.head(self.encoder(x), aux)


def finetune_rev(enc_state, X, y, n_classes, aug=None, seed=42, floor=None, aux=None,
                 epochs=None):
    """Fine-tune encoder + GRU head end to end, class-weighted CE (Eq. 8). If `aug` is
    set, each batch passes clean with probability `floor`, otherwise through the op.
    Fixed epoch count, no early stopping, no validation data of any kind (c48)."""
    floor = REV["floor"] if floor is None else floor
    epochs = REV["ft_epochs"] if epochs is None else epochs
    set_seed(seed)
    s_dim = X.shape[2]
    aux_dim = 0 if aux is None else aux.shape[1]
    model = HARModelRev(n_classes, s_dim, aux_dim).to(DEVICE)
    if enc_state is not None:
        model.encoder.load_state_dict({k: v.to(DEVICE) for k, v in enc_state.items()})
    cnt = np.bincount(np.asarray(y), minlength=n_classes).astype(float); cnt[cnt == 0] = 1
    cw = torch.tensor(len(y) / (n_classes * cnt), dtype=torch.float32, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=REV["ft_lr"])
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(np.asarray(y), dtype=torch.long, device=DEVICE)
    At = None if aux is None else torch.tensor(aux, dtype=torch.float32, device=DEVICE)
    op = aug if callable(aug) else AUG_OPS[aug]
    n, bs = len(Xt), REV["ft_bs"]
    for _ in range(epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs].to(DEVICE)
            xb = Xt[idx]; yb = yt[idx]
            ab = None if At is None else At[idx]
            if op is not None and random.random() >= floor:
                with torch.no_grad():
                    xb = op(xb)
            loss = F.cross_entropy(model(xb, ab), yb, weight=cw)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


@torch.no_grad()
def predict_rev(model, X, aux=None, bs=1024):
    model.eval(); out = []
    Xt = torch.tensor(X, dtype=torch.float32)
    At = None if aux is None else torch.tensor(aux, dtype=torch.float32)
    for i in range(0, len(Xt), bs):
        ab = None if At is None else At[i:i + bs].to(DEVICE)
        out.append(model(Xt[i:i + bs].to(DEVICE), ab).argmax(1).cpu())
    return torch.cat(out).numpy()


def metrics_rev(y_true, y_pred, labels):
    """Macro-F1 over the pair's shared classes; stairs-F1 = macro-F1 over {up, down}
    computed from the SAME predictions, defined for every pair that has both
    stairs classes (all 12 here). Absolute F1, never relative % (c25)."""
    per = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    st = (f1_score(y_true, y_pred, labels=list(STAIRS), average="macro", zero_division=0)
          if all(s in labels for s in STAIRS) else float("nan"))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    wad = float("nan")
    if 4 in labels and 3 in labels:
        r = cm[labels.index(4)]
        wad = float(r[labels.index(3)] / max(r.sum(), 1))
    return {"macro_f1": float(per.mean()), "stairs_f1": float(st),
            "acc": float((np.asarray(y_true) == np.asarray(y_pred)).mean()),
            "per_class": json.dumps({int(l): round(float(v), 4) for l, v in zip(labels, per)}),
            "confusion": json.dumps(cm.tolist()), "walk_as_down": wad}


# =============================================================================
#  4. RUN LOG  (one row per training run, resumable)
# =============================================================================
LOG_COLS = ["exp", "source", "target", "arm", "pre_seed", "ft_seed", "fold", "perturb",
            "angle", "labels", "n_train", "n_test", "macro_f1", "stairs_f1", "acc",
            "per_class", "confusion", "walk_as_down", "extra", "cfg", "data_fp", "time"]


class RunLog:
    def __init__(self, name):
        self.path = os.path.join(REV_DIR, "runs", f"{name}.csv")
        self.keys = set()
        if os.path.exists(self.path):
            prev = pd.read_csv(self.path)
            for _, r in prev.iterrows():
                self.keys.add(self._key(r))

    @staticmethod
    def _key(r):
        ang = r.get("angle", 0.0)
        ang = 0.0 if ang is None or (isinstance(ang, float) and np.isnan(ang)) else float(ang)
        return (str(r["exp"]), str(r["source"]), str(r["target"]), str(r["arm"]),
                int(r["pre_seed"]), int(r["ft_seed"]), int(r["fold"]), str(r["perturb"]),
                round(ang, 3))

    def has(self, **k):
        r = dict(fold=-1, perturb="none", angle=0.0, pre_seed=-1, ft_seed=-1)
        r.update(k)
        return self._key(r) in self.keys

    def add(self, row):
        r = {c: row.get(c, "") for c in LOG_COLS}
        for k, v in dict(fold=-1, perturb="none", angle=0.0, pre_seed=-1, ft_seed=-1).items():
            if r[k] == "":
                r[k] = v
        r["cfg"] = row.get("cfg", json.dumps({k: REV[k] for k in ("lam", "warm", "floor")}))
        r["time"] = time.strftime("%Y-%m-%d %H:%M:%S")
        pd.DataFrame([r], columns=LOG_COLS).to_csv(
            self.path, mode="a", header=not os.path.exists(self.path), index=False)
        self.keys.add(self._key(r))

    def df(self):
        return pd.read_csv(self.path) if os.path.exists(self.path) else pd.DataFrame(columns=LOG_COLS)


# =============================================================================
#  5. ARMS
# -----------------------------------------------------------------------------
#  kind      input transform (raw device frame / canonicalized / ...)
#  obj       pretraining objective (mlm / mlm+yaw consistency / mlm+SO(3) consistency)
#  aug       fine-tuning operator
# =============================================================================
ARMS_REV = {
    # 2x2 component ablation (advisor's email: Baseline SSL -> +Gravity -> +Yaw -> both)
    "A_ssl":     dict(kind="raw",   obj="mlm", aug=None,  label="Baseline SSL (LIMU-BERT)"),
    "B_grav":    dict(kind="canon", obj="mlm", aug=None,  label="+ Gravity canonicalization"),
    "C_yaw":     dict(kind="raw",   obj="yaw", aug="yaw", label="+ Yaw invariance only"),
    "D_gravyaw": dict(kind="canon", obj="yaw", aug="yaw", label="Gravity + yaw invariance (ours)"),
    # where the yaw invariance has to live (c39 c41)
    "D_pre":     dict(kind="canon", obj="yaw", aug=None,  label="Gravity + yaw consistency only"),
    "D_ft":      dict(kind="canon", obj="mlm", aug="yaw", label="Gravity + yaw augmentation only"),
    # baselines requested in c46 and by the e-mail ('lack of the baseline',
    # 'the gravity-based approach has existed before')
    "E_so3aug":  dict(kind="raw",   obj="mlm", aug="so3", label="LIMU-BERT + SO(3) augmentation"),
    "F_so3inv":  dict(kind="raw",   obj="so3", aug="so3", label="Learned SO(3) invariance (no canon.)"),
    "G_pca":     dict(kind="pca",   obj="mlm", aug=None,  label="Gravity + PCA heading alignment"),
    "H_mizell":  dict(kind="mizell", obj="mlm", aug=None, label="Vertical/horizontal decomposition [Mizell]"),
    "I_oit":     dict(kind="oit",   obj="mlm", aug=None,  label="Orientation-invariant transform [Yurtman]"),
}
ABLATION_ARMS = ["A_ssl", "B_grav", "C_yaw", "D_gravyaw"]
SITE_ARMS = ["D_pre", "D_ft"]
BASELINE_ARMS = ["E_so3aug", "F_so3inv", "G_pca", "H_mizell", "I_oit"]


def pair_data(src, tgt, kind, label_mode="merged"):
    Xs, ys, gs, ms, fps = prep_cached(src, label_mode, kind)
    Xt, yt, gt, mt, fpt = prep_cached(tgt, label_mode, kind)
    labels = sorted(int(c) for c in set(np.unique(ys)) & set(np.unique(yt)))
    a, b = np.isin(ys, labels), np.isin(yt, labels)
    lmap = {c: i for i, c in enumerate(labels)}
    return (Xs[a], np.array([lmap[v] for v in ys[a]]), Xt[b], yt[b], labels,
            f"{fps}|{fpt}", a, b)


def _run_cell(log, exp, src, tgt, arm, Xs, ys_idx, Xt, yt, labels, enc_src, label_mode,
              pre_seeds, ft_seeds, fold=-1, aux_s=None, aux_t=None, data_fp="", extra=None,
              enc_kw=None, ft_kw=None):
    a = ARMS_REV[arm] if isinstance(arm, str) else arm
    name = arm if isinstance(arm, str) else a["name"]
    for ps in pre_seeds:
        enc = None
        for fs in ft_seeds:
            if log.has(exp=exp, source=src, target=tgt, arm=name, pre_seed=ps, ft_seed=fs, fold=fold):
                continue
            if enc is None:
                enc = (None if a["obj"] == "none" else
                       get_encoder_rev(enc_src, a["kind"], a["obj"], ps, label_mode, **(enc_kw or {})))
            m = finetune_rev(enc, Xs, ys_idx, len(labels), aug=a["aug"], seed=fs, aux=aux_s,
                             **(ft_kw or {}))
            pred = np.array([labels[p] for p in predict_rev(m, Xt, aux_t)])
            r = metrics_rev(yt, pred, labels)
            log.add(dict(exp=exp, source=src, target=tgt, arm=name, pre_seed=ps, ft_seed=fs,
                         fold=fold, labels=json.dumps(labels), n_train=len(Xs), n_test=len(Xt),
                         data_fp=data_fp, extra=json.dumps(extra or {}), **r))
            _p(f"  [{exp}] {src}->{tgt} {name:10s} fold {fold:2d} pre {ps} ft {fs}  "
               f"F1 {r['macro_f1']:.3f}  stairs {r['stairs_f1']:.3f}")


# =============================================================================
#  6. EXPERIMENTS
# =============================================================================
#  E-MAIN  12-pair matrix: ablation + yaw site + baselines            [RQ1 RQ2 RQ4]
#  c26 c41 c46 c47 c49 c52 c53 c57 c62 c63 c67 + e-mail (baselines)
# -----------------------------------------------------------------------------
def exp_main(arms=None, pairs=None, budget=None, label_mode="merged", log_name="main"):
    arms = arms or (ABLATION_ARMS + SITE_ARMS + BASELINE_ARMS)
    pairs = pairs or PAIRS
    budget = budget or REV["budget"]
    log = RunLog(log_name)
    for src, tgt in pairs:
        for arm in arms:
            kind = ARMS_REV[arm]["kind"]
            Xs, ys, Xt, yt, labels, fp, _, _ = pair_data(src, tgt, kind, label_mode)
            _run_cell(log, log_name, src, tgt, arm, Xs, ys, Xt, yt, labels, src, label_mode,
                      budget["pre"], budget["ft"], data_fp=fp)
    return log.df()


# -----------------------------------------------------------------------------
#  E-XGB  engineered features, raw and canonicalized frame, all 12 pairs, both
#  metrics from the same predictions (c25 c26 c30 c52 c54 c62 c63 c73)
# -----------------------------------------------------------------------------
_XGB_FEAT = {}


def xgb_features(name, kind, label_mode="merged"):
    X, y, g, meta, fp = get_ds(name, label_mode)
    key = (fp, kind)
    if key in _XGB_FEAT:
        return _XGB_FEAT[key]
    path = os.path.join(REV_DIR, "cache", f"xgbfeat_{name}_{fp}_{kind}.pkl")
    if os.path.exists(path):
        df = pd.read_pickle(path)
    else:
        _p(f"  [xgb] extracting features {name}/{kind} ({len(X)} windows, CPU) ...")
        df = _features_df_from_raw(prep(X, kind)).reset_index(drop=True)
        df.to_pickle(path)
    _XGB_FEAT[key] = df
    return df


def _xgb_fit_predict(Fs, ys_idx, Ft, n_classes, seed):
    cnt = np.bincount(ys_idx, minlength=n_classes).astype(float); cnt[cnt == 0] = 1
    w = np.array([len(ys_idx) / (n_classes * cnt[c]) for c in ys_idx])
    mdl = make_xgb(n_classes); mdl.set_params(random_state=seed)
    mdl.fit(Fs, ys_idx, sample_weight=w)
    return mdl.predict(Ft)


def exp_xgb(pairs=None, kinds=("raw", "canon"), seeds=(42, 43, 44), label_mode="merged"):
    pairs = pairs or PAIRS
    log = RunLog("xgb")
    for src, tgt in pairs:
        for kind in kinds:
            arm = f"XGB_{kind}"
            if all(log.has(exp="xgb", source=src, target=tgt, arm=arm, pre_seed=0, ft_seed=s)
                   for s in seeds):
                continue
            _, ys, _, _, fps = get_ds(src, label_mode)
            _, yt, _, _, fpt = get_ds(tgt, label_mode)
            Fs, Ft = xgb_features(src, kind, label_mode), xgb_features(tgt, kind, label_mode)
            labels = sorted(int(c) for c in set(np.unique(ys)) & set(np.unique(yt)))
            a, b = np.isin(ys, labels), np.isin(yt, labels)
            cols = [c for c in Fs.columns if c in Ft.columns]
            lmap = {c: i for i, c in enumerate(labels)}
            ys_idx = np.array([lmap[v] for v in ys[a]])
            for s in seeds:
                if log.has(exp="xgb", source=src, target=tgt, arm=arm, pre_seed=0, ft_seed=s):
                    continue
                p = _xgb_fit_predict(Fs.loc[a, cols], ys_idx, Ft.loc[b, cols], len(labels), s)
                r = metrics_rev(yt[b], np.array([labels[i] for i in p]), labels)
                log.add(dict(exp="xgb", source=src, target=tgt, arm=arm, pre_seed=0, ft_seed=s,
                             labels=json.dumps(labels), n_train=int(a.sum()), n_test=int(b.sum()),
                             data_fp=f"{fps}|{fpt}", **r))
                _p(f"  [xgb] {src}->{tgt} {arm:9s} seed {s}  F1 {r['macro_f1']:.3f} "
                   f"stairs {r['stairs_f1']:.3f}")
    return log.df()


# -----------------------------------------------------------------------------
#  E-CEIL  in-domain ceiling on the SAME label set as each pair (c29 c64 c74)
#  Subject-disjoint 3-fold CV inside the target. Encoders are the target's own
#  source-scope encoders (pretrained on its whole unlabeled pool, i.e. the
#  ceiling is if anything optimistic -> the 'gap closed' fractions are
#  conservative). Gives: gap = ceiling - cross, orientation share of the gap.
# -----------------------------------------------------------------------------
def exp_ceiling(arms=("A_ssl", "B_grav", "D_gravyaw"), xgb_kinds=("raw", "canon"),
                pairs=None, n_folds=3, pre_seeds=(42,), ft_seeds=(42, 43), label_mode="merged"):
    pairs = pairs or PAIRS
    log = RunLog("ceiling")
    combos = {}
    for src, tgt in pairs:
        _, ys, _, _, _ = get_ds(src, label_mode)
        _, yt, _, _, _ = get_ds(tgt, label_mode)
        labels = tuple(sorted(int(c) for c in set(np.unique(ys)) & set(np.unique(yt))))
        combos.setdefault((tgt, labels), []).append(src)
    for (tgt, labels), srcs in combos.items():
        labels = list(labels)
        lab_tag = "-".join(map(str, labels))
        Xr, y, g, _, fp = get_ds(tgt, label_mode)
        keep = np.isin(y, labels)
        idx_all = np.where(keep)[0]
        folds = list(GroupKFold(n_splits=n_folds).split(idx_all, y[idx_all], g[idx_all]))
        lmap = {c: i for i, c in enumerate(labels)}
        for k, (tr, te) in enumerate(folds):
            itr, ite = idx_all[tr], idx_all[te]
            for arm in arms:
                Xk, _, _, _, _ = prep_cached(tgt, label_mode, ARMS_REV[arm]["kind"])
                _run_cell(log, "ceiling", f"in:{tgt}", f"{tgt}[{lab_tag}]", arm, Xk[itr],
                          np.array([lmap[v] for v in y[itr]]), Xk[ite], y[ite], labels, tgt,
                          label_mode, pre_seeds, ft_seeds, fold=k, data_fp=fp,
                          extra={"for_sources": srcs})
            for kind in xgb_kinds:
                arm = f"XGB_{kind}"
                if log.has(exp="ceiling", source=f"in:{tgt}", target=f"{tgt}[{lab_tag}]", arm=arm,
                           pre_seed=0, ft_seed=42, fold=k):
                    continue
                Fd = xgb_features(tgt, kind, label_mode)
                p = _xgb_fit_predict(Fd.iloc[itr], np.array([lmap[v] for v in y[itr]]),
                                     Fd.iloc[ite], len(labels), 42)
                r = metrics_rev(y[ite], np.array([labels[i] for i in p]), labels)
                log.add(dict(exp="ceiling", source=f"in:{tgt}", target=f"{tgt}[{lab_tag}]", arm=arm,
                             pre_seed=0, ft_seed=42, fold=k, labels=json.dumps(labels),
                             n_train=len(itr), n_test=len(ite), data_fp=fp,
                             extra=json.dumps({"for_sources": srcs}), **r))
    return log.df()


# -----------------------------------------------------------------------------
#  E-ROT  controlled rotations on ALL FOUR datasets, subject-disjoint   [RQ3]
#  advisor e-mail (controlled pitch/roll/yaw), c37 c50 c51, limitation 'single
#  dataset'. Two families of perturbation:
#    device_x/y/z  : rotation about a fixed DEVICE axis (what the draft called
#                    pitch/roll/yaw). Its physical meaning depends on the pose:
#                    on UCI-HAR gravity lies along device x, so 'pitch' about x
#                    is a physical heading change -- this is the answer to c37.
#    phys_tilt     : rotation about a HORIZONTAL axis (perpendicular to each
#                    window's own gravity, random azimuth) = physical pitch/roll.
#    phys_heading  : rotation about each window's own gravity = physical yaw.
#  For every test window we also log the residual heading psi left after
#  canonicalization (Prop. 1); accuracy binned by |psi| shows that a
#  canonicalized model depends on the rotation ONLY through psi.
# -----------------------------------------------------------------------------
ROT_PERTURB = {"device_x": (0, 30, 60, 90, 120, 150, 180),
               "device_y": (0, 30, 60, 90, 120, 150, 180),
               "device_z": (0, 30, 60, 90, 120, 150, 180),
               "phys_tilt": (0, 15, 30, 45, 60, 75, 90),
               "phys_heading": (0, 30, 60, 90, 120, 150, 180)}
PSI_BINS = (0, 10, 30, 60, 90, 120, 150, 180.01)


def axis_angle_np(axis, deg):
    axis = np.asarray(axis, np.float64)
    if axis.ndim == 1:
        axis = axis[None]
    a = axis / np.maximum(np.linalg.norm(axis, axis=1, keepdims=True), 1e-12)
    K = _skew(a); t = math.radians(deg)
    return np.eye(3)[None] + math.sin(t) * K + (1 - math.cos(t)) * (K @ K)


def make_perturbation(X_raw, perturb, deg, seed=0):
    """Per-window rotation matrices Q [N,3,3] for one perturbation family/angle."""
    N = len(X_raw)
    if deg == 0:
        return np.broadcast_to(np.eye(3), (N, 3, 3)).copy()
    if perturb.startswith("device_"):
        ax = {"device_x": (1, 0, 0), "device_y": (0, 1, 0), "device_z": (0, 0, 1)}[perturb]
        return np.broadcast_to(axis_angle_np(ax, deg)[0], (N, 3, 3)).copy()
    g, _ = gravity_dirs(X_raw)
    if perturb == "phys_heading":
        return axis_angle_np(g, deg)
    if perturb == "phys_tilt":
        rng = np.random.default_rng(seed)
        u = rng.normal(size=(N, 3))
        h = np.cross(g, u)
        return axis_angle_np(h, deg)
    raise ValueError(perturb)


def residual_heading(g, Q):
    """psi (deg) of S(Q) = R(Q g) Q R(g)^T, the rotation left on the canonicalized
    window (Prop. 1: always a rotation about z)."""
    R0 = rot_to_vertical_batch(g)
    g1 = np.einsum("nij,nj->ni", Q, g)
    R1 = rot_to_vertical_batch(g1 / np.linalg.norm(g1, axis=1, keepdims=True))
    S = R1 @ Q @ np.transpose(R0, (0, 2, 1))
    return np.degrees(np.arctan2(S[:, 1, 0], S[:, 0, 0])), S


def exp_rotation(datasets=None, arms=None, perturbs=None, n_folds=3, pre_seeds=(42,),
                 ft_seeds=(42, 43), strict_pretrain=False):
    """strict_pretrain=False reuses the dataset's source-scope encoder (pretrained on
    unrotated windows of all subjects; labels and rotations never enter it). Set True
    to pretrain per fold on training subjects only (3x the encoders)."""
    datasets = datasets or DATASETS_REV
    arms = arms or (ABLATION_ARMS + ["E_so3aug", "G_pca", "H_mizell"])
    perturbs = perturbs or list(ROT_PERTURB)
    log = RunLog("rotation")
    for dname in datasets:
        Xr, y, g, _, fp = get_ds(dname)
        labels = sorted(int(c) for c in np.unique(y))
        lmap = {c: i for i, c in enumerate(labels)}
        folds = list(GroupKFold(n_splits=n_folds).split(Xr, y, g))
        gdir, _ = gravity_dirs(Xr)
        for k, (tr, te) in enumerate(folds):
            for arm in arms:
                a = ARMS_REV[arm]
                todo = [(pt, d) for pt in perturbs for d in ROT_PERTURB[pt]
                        if not all(log.has(exp="rotation", source=dname, target=dname, arm=arm,
                                           pre_seed=ps, ft_seed=fs, fold=k, perturb=pt, angle=d)
                                   for ps in pre_seeds for fs in ft_seeds)]
                if not todo:
                    continue
                Xtr = prep(Xr[tr], a["kind"])
                ytr = np.array([lmap[v] for v in y[tr]])
                for ps in pre_seeds:
                    if strict_pretrain:
                        enc = get_encoder_rev(dname, a["kind"], a["obj"], ps,
                                              X_override=prep(Xr[tr], a["kind"]),
                                              tag_extra=f"_fold{k}of{n_folds}")
                    else:
                        enc = get_encoder_rev(dname, a["kind"], a["obj"], ps)
                    for fs in ft_seeds:
                        m = None
                        for pt, d in todo:
                            if log.has(exp="rotation", source=dname, target=dname, arm=arm,
                                       pre_seed=ps, ft_seed=fs, fold=k, perturb=pt, angle=d):
                                continue
                            if m is None:
                                m = finetune_rev(enc, Xtr, ytr, len(labels), aug=a["aug"], seed=fs)
                            Q = make_perturbation(Xr[te], pt, d, seed=1000 * k + int(d))
                            Xte = prep(rotate_windows(Xr[te], Q), a["kind"])
                            pred = np.array([labels[p] for p in predict_rev(m, Xte)])
                            r = metrics_rev(y[te], pred, labels)
                            psi, _ = residual_heading(gdir[te], Q)
                            ok = pred == y[te]
                            b = np.digitize(np.abs(psi), PSI_BINS) - 1
                            bins = {str(PSI_BINS[i]): [int(ok[b == i].sum()), int((b == i).sum())]
                                    for i in range(len(PSI_BINS) - 1) if (b == i).any()}
                            log.add(dict(exp="rotation", source=dname, target=dname, arm=arm,
                                         pre_seed=ps, ft_seed=fs, fold=k, perturb=pt, angle=d,
                                         labels=json.dumps(labels), n_train=len(tr), n_test=len(te),
                                         data_fp=fp, extra=json.dumps(
                                             {"psi_bins": bins,
                                              "psi_median": float(np.median(np.abs(psi)))}), **r))
                        _p(f"  [rot] {dname} {arm:10s} fold {k} pre {ps} ft {fs} done")
    return log.df()


# -----------------------------------------------------------------------------
#  E-POSTURE  cost of canonicalization on posture classes (c44)
#  sit / stand / lie are separated by device pose, which canonicalization
#  removes. Within-dataset (subject-disjoint) and cross-dataset on the full label
#  set. D_pose re-injects the removed pose (g_hat in device coordinates) into the
#  classifier head as a 3-d side channel.
# -----------------------------------------------------------------------------
def _pose_aux(X_raw):
    g, _ = gravity_dirs(X_raw)
    return g.astype(np.float32)


def exp_posture(datasets=("uci", "motion", "shoaib", "hhar"),
                pairs=(("uci", "motion"), ("motion", "uci"), ("uci", "shoaib"),
                       ("shoaib", "uci"), ("motion", "shoaib"), ("shoaib", "motion")),
                arms=("A_ssl", "B_grav", "D_gravyaw", "D_pose"), n_folds=3,
                pre_seeds=(42,), ft_seeds=(42, 43)):
    log = RunLog("posture")
    lm = "posture"

    def _arm(a):
        return ARMS_REV["D_gravyaw"] if a == "D_pose" else ARMS_REV[a]

    for dname in datasets:
        Xr, y, g, _, fp = get_ds(dname, lm)
        labels = sorted(int(c) for c in np.unique(y))
        lmap = {c: i for i, c in enumerate(labels)}
        for k, (tr, te) in enumerate(GroupKFold(n_splits=n_folds).split(Xr, y, g)):
            for arm in arms:
                a = _arm(arm)
                Xk, _, _, _, _ = prep_cached(dname, lm, a["kind"])
                aux = _pose_aux(Xr) if arm == "D_pose" else None
                _run_cell(log, "posture", dname, dname, dict(a, name=arm), Xk[tr],
                          np.array([lmap[v] for v in y[tr]]), Xk[te], y[te], labels, dname, lm,
                          pre_seeds, ft_seeds, fold=k, data_fp=fp,
                          aux_s=None if aux is None else aux[tr],
                          aux_t=None if aux is None else aux[te])
    for src, tgt in pairs:
        for arm in arms:
            a = _arm(arm)
            Xs, ys, Xt, yt, labels, fp, ms, mt = pair_data(src, tgt, a["kind"], lm)
            aux_s = aux_t = None
            if arm == "D_pose":
                aux_s = _pose_aux(get_ds(src, lm)[0][ms]); aux_t = _pose_aux(get_ds(tgt, lm)[0][mt])
            _run_cell(log, "posture", src, tgt, dict(a, name=arm), Xs, ys, Xt, yt, labels, src, lm,
                      pre_seeds, ft_seeds, data_fp=fp, aux_s=aux_s, aux_t=aux_t)
    return log.df()


# -----------------------------------------------------------------------------
#  E-HPARAM  source-only hyperparameter selection + target sensitivity (c40)
#  Selection criterion uses ONLY the source: macro-F1 on held-out SOURCE subjects
#  after a random Haar rotation of their raw windows (a source-only proxy for an
#  orientation shift). The whole grid is then reported on the targets, so the
#  reader sees both the selected point and how much the choice matters.
#  (The earlier E4 run chose lambda on the UCI->HHAR TARGET; it is superseded.)
# -----------------------------------------------------------------------------
def exp_hparam(lams=(0.03, 0.1, 0.3), floors=(0.0, 0.3, 0.5), sources=None, pairs=None,
               pre_seed=42, ft_seeds=(42, 43), val_frac=0.2):
    sources = sources or DATASETS_REV
    pairs = pairs or PAIRS
    log = RunLog("hparam")
    for src in sources:
        Xr, y, g, _, fp = get_ds(src)
        labels = sorted(int(c) for c in np.unique(y))
        lmap = {c: i for i, c in enumerate(labels)}
        tr, va = next(GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=0)
                      .split(Xr, y, g))
        rng = np.random.default_rng(0)
        Qv = np.stack([_haar_np(rng) for _ in range(len(va))])
        Xva_rot = canonicalize(rotate_windows(Xr[va], Qv))
        Xva = canonicalize(Xr[va]); Xtr = canonicalize(Xr[tr])
        for lam in lams:
            enc = get_encoder_rev(src, "canon", "yaw", pre_seed, lam=lam)
            for fl in floors:
                arm = f"D_l{lam}_f{fl}"
                for fs in ft_seeds:
                    if log.has(exp="hparam_val", source=src, target=src, arm=arm,
                               pre_seed=pre_seed, ft_seed=fs):
                        continue
                    m = finetune_rev(enc, Xtr, np.array([lmap[v] for v in y[tr]]), len(labels),
                                     aug="yaw", seed=fs, floor=fl)
                    r_clean = metrics_rev(y[va], np.array([labels[p] for p in predict_rev(m, Xva)]), labels)
                    r = metrics_rev(y[va], np.array([labels[p] for p in predict_rev(m, Xva_rot)]), labels)
                    log.add(dict(exp="hparam_val", source=src, target=src, arm=arm, pre_seed=pre_seed,
                                 ft_seed=fs, labels=json.dumps(labels), n_train=len(tr), n_test=len(va),
                                 data_fp=fp, cfg=json.dumps({"lam": lam, "floor": fl, "warm": REV["warm"]}),
                                 extra=json.dumps({"clean_f1": r_clean["macro_f1"]}), **r))
                    _p(f"  [hp-val] {src} lam {lam} floor {fl} ft {fs}: rotated-val F1 "
                       f"{r['macro_f1']:.3f} (clean {r_clean['macro_f1']:.3f})")
    for src, tgt in pairs:
        Xs, ys, Xt, yt, labels, fp, _, _ = pair_data(src, tgt, "canon")
        for lam in lams:
            for fl in floors:
                arm = f"D_l{lam}_f{fl}"
                _run_cell(log, "hparam", src, tgt, dict(ARMS_REV["D_gravyaw"], name=arm), Xs, ys, Xt,
                          yt, labels, src, "merged", (pre_seed,), ft_seeds, data_fp=fp,
                          enc_kw={"lam": lam}, ft_kw={"floor": fl})
    return log.df()


def _haar_np(rng):
    q = rng.normal(size=4); q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


# -----------------------------------------------------------------------------
#  E-AUGSPEC  is it the AXIS or any augmentation? (draft Fig. 11, re-run on the
#  corrected data with the new logger; all 12 pairs instead of 3/6)
# -----------------------------------------------------------------------------
def _perturb_mag(X, op, n=2048, seed=0):
    idx = np.random.default_rng(seed).choice(len(X), min(n, len(X)), replace=False)
    xb = torch.tensor(X[idx], dtype=torch.float32, device=DEVICE)
    torch.manual_seed(seed)
    with torch.no_grad():
        d = (op(xb) - xb).flatten(1).norm(dim=1) / xb.flatten(1).norm(dim=1).clamp(min=1e-9)
    return float(d.mean())


def _bisect(fn, target, lo, hi, iters=22):
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if fn(mid) < target else (lo, mid)
    return 0.5 * (lo + hi)


def exp_aug_specificity(pairs=None, pre_seeds=(42, 43), ft_seeds=(42, 43)):
    pairs = pairs or PAIRS
    log = RunLog("augspec")
    Xref, _, _, _, _ = prep_cached("hhar", "merged", "canon")
    target = _perturb_mag(Xref, op_yaw)
    d_ax = _bisect(lambda d: _perturb_mag(Xref, make_axis_op((1, 0, 0), d)), target, 0.5, 180)
    d_so3 = _bisect(lambda d: _perturb_mag(Xref, make_so3_angle_op(d)), target, 0.5, 180)
    sig = _bisect(lambda s: _perturb_mag(Xref, lambda z: op_jitter(z, s)), target, 1e-3, 3.0)
    ops = {"none": None, "yaw180": op_yaw,
           "hax180": make_axis_op((1, 0, 0), 180.0), "so3_180": make_so3_angle_op(180.0),
           "hax_matched": make_axis_op((1, 0, 0), d_ax), "so3_matched": make_so3_angle_op(d_so3),
           "jitter_matched": (lambda z: op_jitter(z, sig))}
    cal = {"target_mag": target, "hax_deg": d_ax, "so3_deg": d_so3, "jitter_sigma": sig}
    json.dump(cal, open(os.path.join(REV_DIR, "tables", "augspec_calibration.json"), "w"), indent=2)
    _p(f"  [augspec] calibration {cal}")
    for src, tgt in pairs:
        Xs, ys, Xt, yt, labels, fp, _, _ = pair_data(src, tgt, "canon")
        for name, op in ops.items():
            _run_cell(log, "augspec", src, tgt, dict(kind="canon", obj="yaw", aug=op, name=name),
                      Xs, ys, Xt, yt, labels, src, "merged", pre_seeds, ft_seeds, data_fp=fp,
                      extra=cal)
    return log.df()


# -----------------------------------------------------------------------------
#  E-POS  Shoaib body-position transfer (real mounting changes, same subjects)
# -----------------------------------------------------------------------------
def exp_positions(arms=("A_ssl", "B_grav", "C_yaw", "D_gravyaw", "G_pca", "H_mizell"),
                  pre_seeds=(42,), ft_seeds=(42, 43, 44)):
    log = RunLog("positions")
    Xr, y, g, meta, fp = get_ds("shoaib")
    pos = meta["position"]
    for arm in arms:
        Xk, _, _, _, _ = prep_cached("shoaib", "merged", ARMS_REV[arm]["kind"])
        for ps_ in sorted(np.unique(pos)):
            for pt_ in sorted(np.unique(pos)):
                if ps_ == pt_:
                    continue
                si, ti = pos == ps_, pos == pt_
                labels = sorted(int(c) for c in set(np.unique(y[si])) & set(np.unique(y[ti])))
                lmap = {c: i for i, c in enumerate(labels)}
                si &= np.isin(y, labels); ti &= np.isin(y, labels)
                _run_cell(log, "positions", f"shoaib:{SHOAIB_POSITIONS[ps_]}",
                          f"shoaib:{SHOAIB_POSITIONS[pt_]}", arm, Xk[si],
                          np.array([lmap[v] for v in y[si]]), Xk[ti], y[ti], labels, "shoaib",
                          "merged", pre_seeds, ft_seeds, data_fp=fp)
    return log.df()


# -----------------------------------------------------------------------------
#  E-LABEL  label efficiency (c65), optional: k labelled target windows per class
#  added to the source set; evaluated on the remaining target windows.
# -----------------------------------------------------------------------------
def exp_label_efficiency(pairs=(("hhar", "uci"), ("shoaib", "motion")), ks=(0, 5, 10, 25, 50, 100),
                         arms=("A_ssl", "D_gravyaw"), pre_seeds=(42,), ft_seeds=(42, 43, 44, 45)):
    log = RunLog("labeleff")
    for src, tgt in pairs:
        for arm in arms:
            a = ARMS_REV[arm]
            Xs, ys, Xt, yt, labels, fp, _, _ = pair_data(src, tgt, a["kind"])
            lmap = {c: i for i, c in enumerate(labels)}
            for k in ks:
                for ps in pre_seeds:
                    enc = None
                    for fs in ft_seeds:
                        if log.has(exp="labeleff", source=src, target=tgt, arm=arm, pre_seed=ps,
                                   ft_seed=fs, perturb="k", angle=k):
                            continue
                        enc = enc or get_encoder_rev(src, a["kind"], a["obj"], ps)
                        rng = np.random.default_rng(fs)
                        if k > 0:
                            sel = np.concatenate([rng.choice(np.where(yt == c)[0], min(k, int((yt == c).sum())),
                                                             replace=False) for c in labels])
                            Xtr = np.concatenate([Xs, Xt[sel]])
                            ytr = np.concatenate([ys, [lmap[v] for v in yt[sel]]])
                            hold = np.setdiff1d(np.arange(len(yt)), sel)
                        else:
                            Xtr, ytr, hold = Xs, ys, np.arange(len(yt))
                        m = finetune_rev(enc, Xtr, ytr, len(labels), aug=a["aug"], seed=fs)
                        pred = np.array([labels[p] for p in predict_rev(m, Xt[hold])])
                        r = metrics_rev(yt[hold], pred, labels)
                        log.add(dict(exp="labeleff", source=src, target=tgt, arm=arm, pre_seed=ps,
                                     ft_seed=fs, perturb="k", angle=k, labels=json.dumps(labels),
                                     n_train=len(Xtr), n_test=len(hold), data_fp=fp, **r))
    return log.df()


# -----------------------------------------------------------------------------
#  E-AUTOAUG  AutoAugHAR on ALL 12 pairs (c47), optional
#    autoaug_raw  : the published setting -- device-frame LIMU-BERT + bilevel search
#    autoaug_ours : the draft's setting -- our canonicalized yaw-consistent encoder
#  Uses the pipeline's own search implementation (train_classifier 'search').
# -----------------------------------------------------------------------------
def exp_autoaug(pairs=None, settings=(("autoaug_raw", "raw", "mlm"), ("autoaug_ours", "canon", "yaw")),
                pre_seed=42, ft_seeds=(42, 43, 44), search_epochs=15):
    pairs = pairs or PAIRS
    log = RunLog("autoaug")
    for src, tgt in pairs:
        for name, kind, obj in settings:
            if all(log.has(exp="autoaug", source=src, target=tgt, arm=name, pre_seed=pre_seed,
                           ft_seed=fs) for fs in ft_seeds):
                continue
            Xs, ys, Xt, yt, labels, fp, _, _ = pair_data(src, tgt, kind)
            enc = get_encoder_rev(src, kind, obj, pre_seed)
            t0 = time.time()
            _, pol = train_classifier(enc, Xs, ys, len(labels), aug_mode="search",
                                      epochs=search_epochs, freeze=False, return_policy=True,
                                      seed=pre_seed)
            mins = (time.time() - t0) / 60
            names = [n for n, _ in LABEL_PRESERVING_OPS]
            top = sorted(zip(names, pol), key=lambda t: -t[1])[:5]
            for fs in ft_seeds:
                if log.has(exp="autoaug", source=src, target=tgt, arm=name, pre_seed=pre_seed, ft_seed=fs):
                    continue
                m = train_classifier(enc, Xs, ys, len(labels), aug_mode="fixed", fixed_policy=pol, seed=fs)
                m.eval()
                with torch.no_grad():
                    pr = []
                    for i in range(0, len(Xt), 1024):
                        pr.append(m(torch.tensor(Xt[i:i + 1024], dtype=torch.float32, device=DEVICE))
                                  .argmax(1).cpu())
                pred = np.array([labels[p] for p in torch.cat(pr).numpy()])
                r = metrics_rev(yt, pred, labels)
                log.add(dict(exp="autoaug", source=src, target=tgt, arm=name, pre_seed=pre_seed,
                             ft_seed=fs, labels=json.dumps(labels), n_train=len(Xs), n_test=len(Xt),
                             data_fp=fp, extra=json.dumps({"search_min": mins, "peak": float(np.max(pol)),
                                                           "top5": [(n, round(float(v), 4)) for n, v in top]}),
                             **r))
    return log.df()


# -----------------------------------------------------------------------------
#  E-UNIMTS  a published cross-dataset method (c46 item 3)
#  UniMTS (NeurIPS 2024) released checkpoint, fine-tuned on the SOURCE with the
#  authors' recipe (Adam 1e-4, CE, best train loss), evaluated on the target. Run
#  with the raw device frame (as published) and with our canonicalization in
#  front of it (does physics canonicalization still help a foundation model?).
#  Joint placement follows the authors' data.py: waist -> 9 (UCI-HAR; HHAR waist
#  pouch assumed identical), MotionSense pocket -> joints 1 and 5, Shoaib
#  positions -> [1, 5, 21, 20, 0].
# -----------------------------------------------------------------------------
UNIMTS_DIR = os.environ.get("UNIMTS_DIR", "/content/UniMTS")
UNIMTS_JOINTS = {"hhar": [9], "uci": [9], "motion": [1, 5],
                 "shoaib": {0: [1], 1: [5], 2: [21], 3: [20], 4: [0]}}


def setup_unimts():
    """Clone the official repo, install CLIP, fetch the released checkpoint."""
    if not os.path.isdir(UNIMTS_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xiyuanzh/UniMTS.git",
                        UNIMTS_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/openai/CLIP.git", "huggingface_hub", "ftfy", "regex"],
                   check=True)
    from huggingface_hub import hf_hub_download
    ck = os.path.join(UNIMTS_DIR, "checkpoint", "UniMTS.pth")
    if not os.path.exists(ck):
        hf_hub_download(repo_id="xiyuanz/UniMTS", filename="checkpoint/UniMTS.pth", local_dir=UNIMTS_DIR)
    if UNIMTS_DIR not in sys.path:
        sys.path.insert(0, UNIMTS_DIR)
    return ck


def unimts_batch(Xp, joints, gyro=False, pad_to=200):
    """[B,51,6] (acc g, gyro rad/s, 20 Hz) -> UniMTS input [B, C, 200, 22, 1].
    Our windows are already at 20 Hz, so the authors' resampling is the identity;
    padding is 'wrap' as in their load_custom_data."""
    B, L, _ = Xp.shape
    allX = np.zeros((B, L, 22, 6), np.float32)
    acc = Xp[:, :, 0:3] * G_REV; gyr = Xp[:, :, 3:6]
    for i in range(B):
        for j in (joints[i] if isinstance(joints[0], (list, tuple)) else joints):
            allX[i, :, j, 0:3] = acc[i]; allX[i, :, j, 3:6] = gyr[i]
    if L < pad_to:
        allX = np.pad(allX, ((0, 0), (0, pad_to - L), (0, 0), (0, 0)), mode="wrap")
    allX = allX[:, :pad_to]
    if not gyro:
        allX = allX[..., 0:3]
    return torch.from_numpy(allX).permute(0, 3, 1, 2).unsqueeze(-1).contiguous()


def exp_unimts(pairs=None, frames=("raw", "canon"), epochs=10, ft_seeds=(42, 43), gyro=False,
               bs=64, lr=1e-4, max_train=None):
    from types import SimpleNamespace
    ck = setup_unimts()
    from contrastive import ContrastiveModule
    pairs = pairs or PAIRS
    log = RunLog("unimts")
    for src, tgt in pairs:
        for frame in frames:
            arm = f"UniMTS_{frame}"
            Xs_r, ys_all, _, ms_meta, fps = get_ds(src)
            Xt_r, yt_all, _, mt_meta, fpt = get_ds(tgt)
            labels = sorted(int(c) for c in set(np.unique(ys_all)) & set(np.unique(yt_all)))
            a, b = np.isin(ys_all, labels), np.isin(yt_all, labels)
            Xs, Xt = prep(Xs_r[a], frame), prep(Xt_r[b], frame)
            js = UNIMTS_JOINTS[src]; jt = UNIMTS_JOINTS[tgt]
            js = [js[int(p)] for p in ms_meta["position"][a]] if isinstance(js, dict) else js
            jt = [jt[int(p)] for p in mt_meta["position"][b]] if isinstance(jt, dict) else jt
            lmap = {c: i for i, c in enumerate(labels)}
            ys = np.array([lmap[v] for v in ys_all[a]]); yt = yt_all[b]
            for fs in ft_seeds:
                if log.has(exp="unimts", source=src, target=tgt, arm=arm, pre_seed=0, ft_seed=fs):
                    continue
                set_seed(fs)
                args = SimpleNamespace(gyro=int(gyro), stft=0, stage="finetune", num_class=len(labels))
                model = ContrastiveModule(args).to(DEVICE)
                model.model.load_state_dict(torch.load(ck, map_location=DEVICE))
                opt = torch.optim.Adam(model.parameters(), lr=lr)
                tr_idx = np.arange(len(Xs))
                if max_train and len(tr_idx) > max_train:
                    tr_idx = np.random.default_rng(fs).choice(tr_idx, max_train, replace=False)
                best, best_state = None, None
                for ep in range(epochs):
                    model.train(); perm = np.random.permutation(tr_idx); tot = 0.0
                    for i in range(0, len(perm), bs):
                        ii = perm[i:i + bs]
                        jj = [js[k] for k in ii] if isinstance(js, list) and isinstance(js[0], list) else js
                        xb = unimts_batch(Xs[ii], jj, gyro).to(DEVICE)
                        yb = torch.tensor(ys[ii], dtype=torch.long, device=DEVICE)
                        loss = F.cross_entropy(model.classifier(xb).float(), yb)
                        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ii)
                    if best is None or tot < best:
                        best = tot
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    _p(f"  [unimts] {src}->{tgt} {frame} seed {fs} ep {ep} loss {tot / len(perm):.4f}")
                model.load_state_dict(best_state); model.eval()
                pr = []
                with torch.no_grad():
                    for i in range(0, len(Xt), 256):
                        jj = (jt[i:i + 256] if isinstance(jt, list) and isinstance(jt[0], list) else jt)
                        pr.append(model.classifier(unimts_batch(Xt[i:i + 256], jj, gyro).to(DEVICE))
                                  .argmax(1).cpu())
                pred = np.array([labels[p] for p in torch.cat(pr).numpy()])
                r = metrics_rev(yt, pred, labels)
                log.add(dict(exp="unimts", source=src, target=tgt, arm=arm, pre_seed=0, ft_seed=fs,
                             labels=json.dumps(labels), n_train=len(tr_idx), n_test=len(Xt),
                             data_fp=f"{fps}|{fpt}", extra=json.dumps({"epochs": epochs, "gyro": gyro}),
                             **r))
                _p(f"  [unimts] {src}->{tgt} {arm} seed {fs}  F1 {r['macro_f1']:.3f}")
                del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return log.df()


# =============================================================================
#  7. ANALYSES WITHOUT TRAINING (minutes on CPU)
# =============================================================================
def audit_motionsense_sign(n=4000):
    """c33. Pose statistics of MotionSense with and without the iOS->Android sign
    conversion, and what the error does after canonicalization: the vertical
    channel is identical, the horizontal plane is MIRRORED (an improper
    transform that no yaw rotation can undo), and in the raw device frame the
    whole accelerometer is negated relative to the other three datasets."""
    old = REV["ios_sign_fix"]
    try:
        REV["ios_sign_fix"] = True; Xf = get_ds("motion")[0]
        REV["ios_sign_fix"] = False; Xo, y = get_ds("motion")[0:2]
    finally:
        REV["ios_sign_fix"] = old
    idx = np.random.default_rng(0).choice(len(Xf), min(n, len(Xf)), replace=False)
    rows = []
    for tag, X in (("as loaded in draft 2 (iOS sign)", Xo), ("converted to Android sign", Xf)):
        g, nrm = gravity_dirs(X[idx])
        rows.append({"version": tag, "median_gz": float(np.median(g[:, 2])),
                     "frac_gz<0 ('inverted')": float((g[:, 2] < 0).mean()),
                     "frac_gz<-0.9": float((g[:, 2] < -0.9).mean()),
                     "median_|a_bar|_g": float(np.median(nrm))})
    ca, cb = canonicalize(Xo[idx]), canonicalize(Xf[idx])
    vert = float(np.abs(ca[:, :, 2] - cb[:, :, 2]).max())
    hm = float(np.abs(np.linalg.norm(ca[:, :, 0:2], axis=2) - np.linalg.norm(cb[:, :, 0:2], axis=2)).max())
    # det of the 2x2 map between the horizontal planes (least squares, per window)
    dets = []
    for i in range(min(500, len(idx))):
        A, B = cb[i, :, 0:2], ca[i, :, 0:2]
        M, *_ = np.linalg.lstsq(A, B, rcond=None)
        dets.append(np.linalg.det(M))
    df = pd.DataFrame(rows)
    _p("\n=== c33 MotionSense sign audit ===")
    _p(df.round(4).to_string(index=False))
    _p(f"  after canonicalization: max |vertical acc difference| = {vert:.2e} g, "
       f"max |horizontal norm difference| = {hm:.2e} g")
    _p(f"  median det of horizontal map old->fixed = {np.median(dets):+.3f} "
       f"(-1 = mirror image, +1 = rotation)")
    df.to_csv(os.path.join(REV_DIR, "tables", "c33_motionsense_sign.csv"), index=False)
    return df


def geometry_identities(n=2000, seed=0):
    """c36 c37 c51. Numerical check of four statements whose proofs go in the
    paper (Sec. 3.2.3):
      P1  S(Q) is a rotation about z for every Q and g      (Proposition 1)
      L1  a rotation about the DEVICE z axis passes through unchanged:
          S(Rz(t)) = Rz(t) for every g  (so 'yaw reproduces itself' is an identity)
      L2  a rotation about the window's own gravity by t leaves psi = t exactly
      L3  a physical tilt leaves psi = 0 only when g = +z; otherwise psi grows
          with the angle between g and +z and diverges near g = -z
          (the gauge of the minimal rotation)."""
    rng = np.random.default_rng(seed)
    g = rng.normal(size=(n, 3)); g /= np.linalg.norm(g, axis=1, keepdims=True)
    Q = np.stack([_haar_np(rng) for _ in range(n)])
    psi, S = residual_heading(g, Q)
    tilt_axis = np.degrees(np.arccos(np.clip(np.abs(S[:, 2, 2]), 0, 1)))
    out = {"P1_max_axis_tilt_deg": float(tilt_axis.max())}
    for t in (30.0, 90.0, 150.0):
        psi_z, _ = residual_heading(g, np.broadcast_to(axis_angle_np((0, 0, 1), t)[0], (n, 3, 3)))
        psi_g, _ = residual_heading(g, axis_angle_np(g, t))
        out[f"L1_device_z_{int(t)}_max_err"] = float(np.abs(psi_z - t).max())
        out[f"L2_about_g_{int(t)}_max_err"] = float(np.abs(psi_g - t).max())
    ang_from_up = np.degrees(np.arccos(np.clip(g[:, 2], -1, 1)))
    h = np.cross(g, rng.normal(size=(n, 3)))
    psi_t, _ = residual_heading(g, axis_angle_np(h, 45.0))
    rows = []
    for lo, hi in ((0, 30), (30, 60), (60, 90), (90, 120), (120, 150), (150, 170), (170, 180.01)):
        m = (ang_from_up >= lo) & (ang_from_up < hi)
        if m.any():
            rows.append({"angle(g,+z) deg": f"{lo}-{int(hi)}", "n": int(m.sum()),
                         "median |psi| after 45 deg tilt": float(np.median(np.abs(psi_t[m]))),
                         "p95 |psi|": float(np.percentile(np.abs(psi_t[m]), 95))})
    _p("\n=== geometry identities (should be ~0 errors) ===")
    for k, v in out.items():
        _p(f"  {k:32s} {v:.2e}")
    _p("\n=== L3: residual heading left by a pure 45 deg physical tilt, by device pose ===")
    tab = pd.DataFrame(rows)
    _p(tab.round(2).to_string(index=False))
    tab.to_csv(os.path.join(REV_DIR, "tables", "geometry_L3_tilt_gauge.csv"), index=False)
    json.dump(out, open(os.path.join(REV_DIR, "tables", "geometry_identities.json"), "w"), indent=2)
    return out, tab


def rotation_axis_decomposition(datasets=None, degs=(30, 90), n=3000):
    """c37 c50 c51, data-driven. For each dataset and device-axis rotation:
    the angle between the rotation axis and the window's gravity (0 = the rotation
    is a physical heading change, 90 = a pure tilt), the gravity displacement it
    causes in the RAW frame (what the device-frame baseline sees), and the residual
    heading after canonicalization. Also the fraction of dynamic acc/gyro energy on
    each device axis: a rotation leaves the component along its own axis
    untouched, which is the valid explanation of why the raw baseline is more
    robust to one axis than another (c50)."""
    datasets = datasets or DATASETS_REV
    rows, energy = [], []
    for d in datasets:
        X = get_ds(d)[0]
        idx = np.random.default_rng(0).choice(len(X), min(n, len(X)), replace=False)
        X = X[idx]; g, _ = gravity_dirs(X)
        dyn = X - X.mean(1, keepdims=True)
        ea = (dyn[:, :, 0:3] ** 2).mean((0, 1)); eg = (dyn[:, :, 3:6] ** 2).mean((0, 1))
        energy.append({"dataset": d, **{f"acc_dyn_energy_{a}": float(v / ea.sum()) for a, v in zip("xyz", ea)},
                       **{f"gyro_energy_{a}": float(v / eg.sum()) for a, v in zip("xyz", eg)}})
        for pt in ("device_x", "device_y", "device_z"):
            ax = {"device_x": 0, "device_y": 1, "device_z": 2}[pt]
            ang = np.degrees(np.arccos(np.clip(np.abs(g[:, ax]), 0, 1)))
            for deg in degs:
                Q = make_perturbation(X, pt, deg)
                g1 = np.einsum("nij,nj->ni", Q, g)
                disp = np.degrees(np.arccos(np.clip((g * g1).sum(1), -1, 1)))
                psi, _ = residual_heading(g, Q)
                rows.append({"dataset": d, "rotation": pt, "deg": deg,
                             "median angle(axis, g)": float(np.median(ang)),
                             "median gravity displacement (raw frame)": float(np.median(disp)),
                             "median |psi| after canon.": float(np.median(np.abs(psi)))})
    df, en = pd.DataFrame(rows), pd.DataFrame(energy)
    _p("\n=== device-axis rotations, decomposed (c37 c50 c51) ===")
    _p(df.round(1).to_string(index=False))
    _p("\n=== share of dynamic energy per device axis (raw frame) ===")
    _p(en.round(3).to_string(index=False))
    df.to_csv(os.path.join(REV_DIR, "tables", "rotation_axis_decomposition.csv"), index=False)
    en.to_csv(os.path.join(REV_DIR, "tables", "axis_energy.csv"), index=False)
    return df, en


def antipode_report(datasets=None, deltas=(5, 10, 15, 25), noise_deg=1.0, n=3000):
    """c38. The neighbourhood of the singularity is defined by the angle between
    g_hat and -z. Within it, a small error in g_hat produces a large change of the
    canonical heading: we measure that amplification directly (heading change
    caused by a `noise_deg` perturbation of g_hat) as a function of the distance to
    the antipode. Exactly at g = -z the code returns diag(1,-1,-1)."""
    datasets = datasets or DATASETS_REV
    rows = []
    rng = np.random.default_rng(0)
    for d in datasets:
        X = get_ds(d)[0]
        g, nrm = gravity_dirs(X)
        dist = np.degrees(np.arccos(np.clip(-g[:, 2], -1, 1)))
        r = {"dataset": d, "n": len(g), "gate_fail_frac": float((nrm < REV["gate_g"]).mean())}
        for dl in deltas:
            r[f"within_{dl}deg_of_antipode"] = float((dist < dl).mean())
        idx = rng.choice(len(g), min(n, len(g)), replace=False)
        gi = g[idx]
        pert = gi + np.radians(noise_deg) * rng.normal(size=gi.shape)
        pert /= np.linalg.norm(pert, axis=1, keepdims=True)
        R0, R1 = rot_to_vertical_batch(gi), rot_to_vertical_batch(pert)
        D = R1 @ np.transpose(R0, (0, 2, 1))
        dh = np.abs(np.degrees(np.arctan2(D[:, 1, 0], D[:, 0, 0])))
        r["median heading jitter (deg) per 1 deg g error, far (>90 from antipode)"] = float(
            np.median(dh[dist[idx] > 90])) if (dist[idx] > 90).any() else np.nan
        r["... near (<15 from antipode)"] = float(np.median(dh[dist[idx] < 15])) if (dist[idx] < 15).any() else np.nan
        rows.append(r)
    df = pd.DataFrame(rows)
    _p("\n=== c38 antipodal neighbourhood ===")
    _p(df.round(4).to_string(index=False))
    df.to_csv(os.path.join(REV_DIR, "tables", "c38_antipode.csv"), index=False)
    return df


def heading_profile_v2(datasets=None):
    """c32. theta_h = axis (mod 180) of the principal horizontal acceleration of a
    dynamic window after canonicalization, measured from the canonical x axis
    (x_c = R(g_hat) applied to the device frame, i.e. the gauge fixed by Eq. 3):
        C = sum_t h_t h_t^T, h_t = [a_x,t a_y,t]^T - mean,  theta_h = atan2(v2, v1) mod 180
    with v the leading eigenvector of C. Concentration R = |mean exp(2 i theta_h)| in
    [0,1] (0 uniform, 1 identical)."""
    datasets = datasets or DATASETS_REV
    rows = []
    for d in datasets:
        X, y = get_ds(d)[0:2]
        m = np.isin(y, (2, 3, 4, 5))
        Xc = canonicalize(X[m])
        H = Xc[:, :, 0:2].astype(np.float64); H = H - H.mean(1, keepdims=True)
        C = np.einsum("nti,ntj->nij", H, H)
        _, V = np.linalg.eigh(C)
        th = np.degrees(np.arctan2(V[:, 1, 1], V[:, 0, 1])) % 180.0
        z = np.exp(1j * np.radians(2 * th))
        rows.append({"dataset": d, "n_dynamic": int(m.sum()), "R_bar": float(abs(z.mean())),
                     "mean_theta_deg": float((np.degrees(np.angle(z.mean())) / 2) % 180)})
    df = pd.DataFrame(rows)
    _p("\n=== c32 residual-heading profile (dynamic windows) ===")
    _p(df.round(3).to_string(index=False))
    df.to_csv(os.path.join(REV_DIR, "tables", "c32_heading_profile.csv"), index=False)
    return df


def cadence_energy_table(datasets=None):
    """c66. Dominant walking cadence (FFT of the acc magnitude, walk windows) and
    acc/gyro energy per dataset, relative to HHAR."""
    datasets = datasets or DATASETS_REV
    rows = []
    for d in datasets:
        X, y = get_ds(d)[0:2]
        w = X[y == 4]
        if not len(w):
            continue
        mag = np.linalg.norm(w[:, :, 0:3], axis=2); mag = mag - mag.mean(1, keepdims=True)
        sp = np.abs(np.fft.rfft(mag, axis=1)); fr = np.fft.rfftfreq(mag.shape[1], d=1.0 / TARGET_HZ)
        dom = fr[1 + sp[:, 1:].argmax(1)]
        rows.append({"dataset": d, "n_walk": len(w), "cadence_median_hz": float(np.median(dom)),
                     "cadence_iqr": f"{np.percentile(dom, 25):.2f}-{np.percentile(dom, 75):.2f}",
                     "freq_resolution_hz": float(fr[1]),
                     "acc_dyn_rms_g": float(np.median(mag.std(1))),
                     "gyro_rms_rad_s": float(np.median(np.linalg.norm(w[:, :, 3:6], axis=2).std(1)))})
    df = pd.DataFrame(rows)
    if "hhar" in set(df.dataset):
        ref = df.set_index("dataset").loc["hhar"]
        df["acc_energy_vs_hhar"] = df["acc_dyn_rms_g"] / ref["acc_dyn_rms_g"]
        df["gyro_energy_vs_hhar"] = df["gyro_rms_rad_s"] / ref["gyro_rms_rad_s"]
    _p("\n=== c66 walking cadence and energy ===")
    _p(df.round(3).to_string(index=False))
    _p("  note: 2.56 s windows give a 0.39 Hz frequency resolution; report it with the cadence.")
    df.to_csv(os.path.join(REV_DIR, "tables", "c66_cadence_energy.csv"), index=False)
    return df


def preprocessing_table(datasets=None):
    """c45 (+c44): what each dataset contributes after preprocessing."""
    datasets = datasets or DATASETS_REV
    rows = []
    for d in datasets:
        raw = load_raw_v2(d)
        X, y, g, meta, _ = get_ds(d)
        _, yp, _, _, _ = get_ds(d, "posture")
        r = {"dataset": d, "subjects": len(np.unique(g)), "windows (merged labels)": len(X),
             "windows (posture labels)": len(yp), **raw["info"],
             "classes (merged)": ", ".join(f"{CLS[c]}:{int((y == c).sum())}" for c in np.unique(y)),
             "classes (posture)": ", ".join(f"{CLS[c]}:{int((yp == c).sum())}" for c in np.unique(yp))}
        if d == "hhar":
            mdl, hz = meta["model"], meta["native_hz"]
            r["devices"] = "; ".join(f"{m}: {np.median(hz[mdl == m]):.0f} Hz"
                                     for m in sorted(np.unique(mdl)))
        rows.append(r)
    df = pd.DataFrame(rows)
    _p(df.T.to_string())
    df.to_csv(os.path.join(REV_DIR, "tables", "c45_preprocessing.csv"), index=False)
    return df


def pair_label_table(pairs=None, label_mode="merged"):
    """c44: the classes each of the 12 pairs uses."""
    pairs = pairs or PAIRS
    rows = []
    for s, t in pairs:
        ys, yt = get_ds(s, label_mode)[1], get_ds(t, label_mode)[1]
        sh = sorted(int(c) for c in set(np.unique(ys)) & set(np.unique(yt)))
        rows.append({"pair": f"{s}->{t}", "classes": ", ".join(CLS[c] for c in sh),
                     "n_source": int(np.isin(ys, sh).sum()), "n_target": int(np.isin(yt, sh).sum())})
    df = pd.DataFrame(rows)
    _p(df.to_string(index=False))
    df.to_csv(os.path.join(REV_DIR, "tables", f"c44_pair_labels_{label_mode}.csv"), index=False)
    return df


def param_breakdown(n_classes=4):
    """c42."""
    rows = []
    for kind, s_dim in (("6-ch (raw/canon/pca)", 6), ("Mizell 4-ch", 4), ("OIT 18-ch", 18)):
        m = HARModelRev(n_classes, s_dim)
        e = sum(p.numel() for p in m.encoder.parameters())
        h = sum(p.numel() for p in m.head.parameters())
        rows.append({"input": kind, "LIMU-BERT encoder": e, "GRU head": h, "total": e + h})
    dec = sum(p.numel() for p in LimuBERT().decoder.parameters())
    df = pd.DataFrame(rows)
    _p(df.to_string(index=False))
    _p(f"  pretraining-only decoder (discarded after pretraining): {dec:,} parameters")
    df.to_csv(os.path.join(REV_DIR, "tables", "c42_parameters.csv"), index=False)
    return df


def inference_benchmark(n=2048, reps=20, export=True):
    """c27 c56. CPU model, threads and batch size are recorded with every number;
    single-thread batch-1 latency is the figure closest to a phone. Exports
    TorchScript/ONNX so the same model can be timed on a device."""
    X = get_ds("hhar")[0][:n]
    try:
        cpu = subprocess.run(["bash", "-c", "lscpu | grep 'Model name' | head -1"],
                             capture_output=True, text=True).stdout.split(":")[-1].strip()
    except Exception:
        cpu = platform.processor()
    rows = []
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); canonicalize(X); t.append(time.perf_counter() - t0)
    rows.append({"component": "canonicalization (numpy, vectorized)", "batch": n, "threads": "numpy default",
                 "ms_per_window": 1000 * np.median(t) / n})
    t = []
    for i in range(min(n, 500)):
        t0 = time.perf_counter(); canonicalize(X[i:i + 1]); t.append(time.perf_counter() - t0)
    rows.append({"component": "canonicalization (numpy)", "batch": 1, "threads": "numpy default",
                 "ms_per_window": 1000 * np.median(t)})
    enc = get_encoder_rev("hhar", "canon", "yaw", 42)
    model = HARModelRev(4).cpu(); model.encoder.load_state_dict(enc); model.eval()
    nt0 = torch.get_num_threads()
    for threads in (1, nt0):
        torch.set_num_threads(threads)
        for bsz in (1, 256):
            xb = torch.tensor(canonicalize(X[:bsz]), dtype=torch.float32)
            with torch.no_grad():
                for _ in range(3):
                    model(xb)
                t = []
                for _ in range(reps):
                    t0 = time.perf_counter(); model(xb); t.append(time.perf_counter() - t0)
            rows.append({"component": "encoder + GRU head (PyTorch CPU)", "batch": bsz, "threads": threads,
                         "ms_per_window": 1000 * np.median(t) / bsz})
    torch.set_num_threads(nt0)
    df = pd.DataFrame(rows)
    df["cpu"] = cpu; df["torch"] = torch.__version__
    df["fraction_of_2.56s_window_%"] = 100 * df["ms_per_window"] / 2560.0
    _p(df.to_string(index=False))
    df.to_csv(os.path.join(REV_DIR, "tables", "c56_inference.csv"), index=False)
    if export:
        xb = torch.zeros(1, XDOMAIN_SEQ_LEN, 6)
        ts = torch.jit.trace(model, (xb,))
        ts.save(os.path.join(REV_DIR, "tables", "gravihar_hhar_4cls.pt"))
        try:
            torch.onnx.export(model, (xb,), os.path.join(REV_DIR, "tables", "gravihar_hhar_4cls.onnx"),
                              input_names=["window"], output_names=["logits"], opset_version=17)
        except Exception as e:
            _p(f"  [onnx] export skipped: {e}")
    return df


# =============================================================================
#  8. COST ESTIMATE + DRIVER
# =============================================================================
def _timeit(fn, n=30, warm=5):
    for _ in range(warm):
        fn()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n


def _measure_rates(unimts=False):
    """Per-step times of the real training code on THIS runtime (about a minute)."""
    X = prep_cached("hhar", "merged", "canon")[0]
    xf = torch.tensor(X[:REV["ft_bs"]], dtype=torch.float32, device=DEVICE)
    yf = torch.randint(0, 5, (len(xf),), device=DEVICE)
    m = HARModelRev(5).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)

    def ft_step():
        x = op_yaw(xf) if random.random() >= REV["floor"] else xf
        loss = F.cross_entropy(m(x), yf)
        opt.zero_grad(); loss.backward(); opt.step()

    xp = torch.tensor(X[:REV["pre_bs"]], dtype=torch.float32, device=DEVICE)
    lb = LimuBERT().to(DEVICE); opt2 = torch.optim.Adam(lb.parameters(), lr=1e-3)

    def pre_step(cons):
        xm, mask = span_mask(xp)
        loss = F.mse_loss(lb(xm)[mask], xp[mask])
        if cons:
            e1 = lb.encoder(op_yaw(op_jitter(xp, 0.03))).mean(1)
            e2 = lb.encoder(op_yaw(op_jitter(xp, 0.03))).mean(1)
            loss = loss + 0.1 * (1 - F.cosine_similarity(e1, e2, dim=1).mean())
        opt2.zero_grad(); loss.backward(); opt2.step()

    r = {"ft_step": _timeit(ft_step), "pre_step": _timeit(lambda: pre_step(False)),
         "pre_step_cons": _timeit(lambda: pre_step(True))}
    Xr = get_ds("hhar")[0][:4096]

    def ev():
        Q = make_perturbation(Xr, "phys_tilt", 45.0)
        Xp = prep(rotate_windows(Xr, Q), "canon")
        predict_rev(m, Xp); residual_heading(gravity_dirs(Xr)[0], Q)
    r["eval_per_window"] = _timeit(ev, n=3, warm=1) / len(Xr)
    t0 = time.perf_counter(); _features_df_from_raw(get_ds("hhar")[0][:200])
    r["xgb_feat_per_window"] = (time.perf_counter() - t0) / 200
    rng = np.random.default_rng(0)
    Fx, Fy = rng.normal(size=(4000, 110)), rng.integers(0, 4, 4000)
    t0 = time.perf_counter(); make_xgb(4).fit(Fx, Fy)
    r["xgb_fit_per_sample_class"] = (time.perf_counter() - t0) / (4000 * 4)
    r["unimts_measured"] = False
    r["unimts_step"] = 0.06 if (DEVICE.type == "cuda" and "A100" in torch.cuda.get_device_name(0)) else 0.25
    if unimts:
        from types import SimpleNamespace
        ck = setup_unimts()
        from contrastive import ContrastiveModule
        um = ContrastiveModule(SimpleNamespace(gyro=0, stft=0, stage="finetune", num_class=5)).to(DEVICE)
        um.model.load_state_dict(torch.load(ck, map_location=DEVICE))
        uo = torch.optim.Adam(um.parameters(), lr=1e-4)
        xb = torch.randn(64, 3, 200, 22, 1, device=DEVICE); yb = torch.randint(0, 5, (64,), device=DEVICE)

        def u_step():
            loss = F.cross_entropy(um.classifier(xb).float(), yb)
            uo.zero_grad(); loss.backward(); uo.step()
        r["unimts_step"] = _timeit(u_step, n=10, warm=2) + 0.01   # + host-side batch building
        r["unimts_measured"] = True
        del um; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return r


def _stage_work(pre_epochs_assumed):
    """Exact amount of work per stage, from the real dataset sizes and budgets.
    Returns {stage: dict(tier, ft_steps, ft_runs, encoders, eval_windows, xgb_feat,
    xgb_fit_units, unimts_steps, log, done_filter)}."""
    ceil_ = lambda a, b: -(-a // b)
    ft = lambda n: ceil_(int(n), REV["ft_bs"]) * REV["ft_epochs"]
    nb = len(REV["budget"]["pre"]) * len(REV["budget"]["ft"])
    nbb = len(REV["budget_baselines"]["pre"]) * len(REV["budget_baselines"]["ft"])
    sizes, n_all, n_post = {}, {}, {}
    for d in DATASETS_REV:
        n_all[d] = len(get_ds(d)[1]); n_post[d] = len(get_ds(d, "posture")[1])
    for s_, t_ in PAIRS:
        ys, yt = get_ds(s_)[1], get_ds(t_)[1]
        lab = sorted(set(np.unique(ys)) & set(np.unique(yt)))
        sizes[(s_, t_)] = (int(np.isin(ys, lab).sum()), int(np.isin(yt, lab).sum()), len(lab), tuple(lab))
    src_steps = sum(ft(v[0]) for v in sizes.values())
    W = {}
    W["xgb"] = dict(tier=1, ft_runs=0, ft_steps=0, encoders=[], eval_windows=0,
                    xgb_feat=[(d, k) for d in DATASETS_REV for k in ("raw", "canon")],
                    xgb_fit_units=sum(v[0] * v[2] for v in sizes.values()) * 2 * 3, unimts_steps=0,
                    log="xgb", arms=["XGB_raw", "XGB_canon"], expected=12 * 2 * 3)
    enc_ab = [(d, k, o, ps, "merged", None) for d in DATASETS_REV for ps in REV["budget"]["pre"]
              for k, o in (("raw", "mlm"), ("canon", "mlm"), ("raw", "yaw"), ("canon", "yaw"))]
    W["ablation"] = dict(tier=1, ft_runs=4 * nb * 12, ft_steps=4 * nb * src_steps, encoders=enc_ab,
                         eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0,
                         log="main", arms=ABLATION_ARMS, expected=4 * nb * 12)
    enc_b = [(d, k, o, ps, "merged", None) for d in DATASETS_REV for ps in REV["budget_baselines"]["pre"]
             for k, o in (("raw", "mlm"), ("raw", "so3"), ("pca", "mlm"), ("mizell", "mlm"), ("oit", "mlm"))]
    W["baselines"] = dict(tier=1, ft_runs=5 * nbb * 12, ft_steps=5 * nbb * src_steps, encoders=enc_b,
                          eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0,
                          log="main", arms=BASELINE_ARMS, expected=5 * nbb * 12)
    combos = {}
    for (s_, t_), v in sizes.items():
        combos[(t_, v[3])] = v[1]
    W["ceiling"] = dict(tier=1, ft_runs=len(combos) * 3 * 3 * 2,
                        ft_steps=sum(3 * 3 * 2 * ft(n * 2 / 3) for n in combos.values()),
                        encoders=[(d, k, o, 42, "merged", None) for d in DATASETS_REV
                                  for k, o in (("raw", "mlm"), ("canon", "mlm"), ("canon", "yaw"))],
                        eval_windows=0, xgb_feat=[],
                        xgb_fit_units=sum(3 * 2 * (n * 2 / 3) * len(l) for (t_, l), n in combos.items()),
                        unimts_steps=0, log="ceiling", arms=None, expected=len(combos) * 3 * (3 * 2 + 2))
    rot_arms = ABLATION_ARMS + ["E_so3aug", "G_pca", "H_mizell"]
    n_angles = sum(len(v) for v in ROT_PERTURB.values())
    W["rotation"] = dict(tier=1, ft_runs=len(rot_arms) * 3 * 2 * 4,
                         ft_steps=sum(len(rot_arms) * 3 * 2 * ft(n_all[d] * 2 / 3) for d in DATASETS_REV),
                         encoders=[(d, ARMS_REV[a]["kind"], ARMS_REV[a]["obj"], 42, "merged", None)
                                   for d in DATASETS_REV for a in rot_arms],
                         eval_windows=sum(len(rot_arms) * 2 * n_all[d] * n_angles for d in DATASETS_REV),
                         xgb_feat=[], xgb_fit_units=0, unimts_steps=0, log="rotation", arms=None,
                         expected=sum(len(rot_arms) * 3 * 2 * n_angles for d in DATASETS_REV))
    W["unimts"] = dict(tier=1, ft_runs=12 * 2 * 2, ft_steps=0, encoders=[], eval_windows=0, xgb_feat=[],
                       xgb_fit_units=0,
                       # per (frame, seed): 10 epochs of training + one forward pass over the target
                       unimts_steps=sum(2 * 2 * (10 * ceil_(v[0], 64) + ceil_(v[1], 64) / 3)
                                        for v in sizes.values()),
                       log="unimts", arms=None, expected=12 * 2 * 2)
    W["site"] = dict(tier=2, ft_runs=2 * nb * 12, ft_steps=2 * nb * src_steps, encoders=enc_ab,
                     eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0,
                     log="main", arms=SITE_ARMS, expected=2 * nb * 12)
    post_cross = (("uci", "motion"), ("motion", "uci"), ("uci", "shoaib"), ("shoaib", "uci"),
                  ("motion", "shoaib"), ("shoaib", "motion"))
    pw = [d for d in ("uci", "motion", "shoaib", "hhar")]
    W["posture"] = dict(tier=2, ft_runs=4 * 2 * (3 * len(pw) + len(post_cross)),
                        ft_steps=4 * 2 * (sum(3 * ft(n_post[d] * 2 / 3) for d in pw)
                                          + sum(ft(n_post[s_]) for s_, _ in post_cross)),
                        encoders=[(d, k, o, 42, "posture", None) for d in pw
                                  for k, o in (("raw", "mlm"), ("canon", "mlm"), ("canon", "yaw"))],
                        eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0, log="posture",
                        arms=None, expected=4 * 2 * (3 * len(pw) + len(post_cross)))
    W["hparam"] = dict(tier=2, ft_runs=9 * 2 * (4 + 12),
                       ft_steps=9 * 2 * (sum(ft(0.8 * n_all[d]) for d in DATASETS_REV) + src_steps),
                       encoders=[(d, "canon", "yaw", 42, "merged", lam) for d in DATASETS_REV
                                 for lam in (0.03, 0.1, 0.3)],
                       eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0, log="hparam",
                       arms=None, expected=9 * 2 * (4 + 12))
    W["augspec"] = dict(tier=3, ft_runs=7 * 4 * 12, ft_steps=7 * 4 * src_steps,
                        encoders=[(d, "canon", "yaw", ps, "merged", None) for d in DATASETS_REV
                                  for ps in (42, 43)],
                        eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0, log="augspec",
                        arms=None, expected=7 * 4 * 12)
    n_pos = n_all["shoaib"] / 5
    W["positions"] = dict(tier=3, ft_runs=6 * 20 * 3, ft_steps=6 * 20 * 3 * ft(n_pos),
                          encoders=[("shoaib", ARMS_REV[a]["kind"], ARMS_REV[a]["obj"], 42, "merged", None)
                                    for a in ("A_ssl", "B_grav", "C_yaw", "D_gravyaw", "G_pca", "H_mizell")],
                          eval_windows=0, xgb_feat=[], xgb_fit_units=0, unimts_steps=0, log="positions",
                          arms=None, expected=6 * 20 * 3)
    return W


def estimate_cost(pre_epochs_assumed=60, unimts=False):
    """Measure per-step times of the real training code on this runtime (about a
    minute; pass unimts=True to also time UniMTS, which clones it and downloads the
    checkpoint), then multiply by the exact amount of work of every stage, computed
    from the dataset sizes, pair label sets and seed budgets. Runs already in the
    CSVs and encoders already cached are subtracted, so calling this again later
    gives the REMAINING time. Pretraining stops early between 30 and 80 epochs; the
    estimate assumes `pre_epochs_assumed` and also prints the 30/80 bounds."""
    r = _measure_rates(unimts)
    W = _stage_work(pre_epochs_assumed)
    seen_enc, seen_feat = set(), set()
    rows = []
    for stage, w in W.items():
        # encoders that this stage would have to pretrain (not cached, not counted earlier)
        pre_h = 0.0; n_new = 0
        for (d, kind, obj, seed, lm, lam) in w["encoders"]:
            lam = REV["lam"] if lam is None else lam
            fp = get_ds(d, lm)[4]
            _, path = _enc_path(d, kind, obj, seed, fp, lam, REV["warm"])
            if path in seen_enc or os.path.exists(path):
                continue
            seen_enc.add(path); n_new += 1
            steps = -(-len(get_ds(d, lm)[1]) // REV["pre_bs"]) * pre_epochs_assumed
            pre_h += steps * (r["pre_step"] if obj == "mlm" else r["pre_step_cons"]) / 3600
        feat_h = 0.0
        for (d, kind) in w["xgb_feat"]:
            fp = get_ds(d)[4]
            if (d, kind) in seen_feat or os.path.exists(
                    os.path.join(REV_DIR, "cache", f"xgbfeat_{d}_{fp}_{kind}.pkl")):
                continue
            seen_feat.add((d, kind))
            feat_h += len(get_ds(d)[1]) * r["xgb_feat_per_window"] / 3600
        done = 0
        lp = os.path.join(REV_DIR, "runs", f"{w['log']}.csv")
        if os.path.exists(lp):
            lg = pd.read_csv(lp)
            if w["arms"] is not None:
                lg = lg[lg.arm.isin(w["arms"])]
            done = len(lg)
        left = max(0.0, 1.0 - done / max(w["expected"], 1))
        work_h = left * (w["ft_steps"] * r["ft_step"] + w["eval_windows"] * r["eval_per_window"]
                         + w["xgb_fit_units"] * r["xgb_fit_per_sample_class"]
                         + w["unimts_steps"] * r["unimts_step"]) / 3600
        rows.append({"stage": stage, "tier": w["tier"], "fine-tuning runs": w["ft_runs"],
                     "done": f"{100 * (1 - left):.0f}%", "new encoders": n_new,
                     "pretrain_h": pre_h, "train+eval_h": work_h + feat_h, "total_h": pre_h + work_h + feat_h})
    df = pd.DataFrame(rows)
    gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else str(DEVICE)
    _p(f"[cost] {gpu}: fine-tune step {1000 * r['ft_step']:.1f} ms (batch {REV['ft_bs']}), "
       f"pretrain step {1000 * r['pre_step']:.1f} ms / {1000 * r['pre_step_cons']:.1f} ms with consistency "
       f"(batch {REV['pre_bs']}), UniMTS step {1000 * r['unimts_step']:.0f} ms "
       f"({'measured' if r['unimts_measured'] else 'ASSUMED -- estimate_cost(unimts=True) measures it'})")
    _p(df.round(1).to_string(index=False))
    for t in (1, 2, 3):
        d = df[df.tier == t]
        pre = d.pretrain_h.sum()
        lo = d.total_h.sum() - pre + pre * 30 / pre_epochs_assumed
        hi = d.total_h.sum() - pre + pre * 80 / pre_epochs_assumed
        _p(f"  Tier {t}: {d.total_h.sum():.1f} h remaining (pretraining stopping at 30-80 epochs: "
           f"{lo:.1f}-{hi:.1f} h)")
    return df


def run_stage(name):
    """Stages in priority order (see REVISION_PLAN.md)."""
    S = {
        "audit": lambda: (audit_motionsense_sign(), geometry_identities(), rotation_axis_decomposition(),
                          antipode_report(), heading_profile_v2(), cadence_energy_table(),
                          preprocessing_table(), pair_label_table(), pair_label_table(label_mode="posture"),
                          param_breakdown()),
        "xgb": exp_xgb,
        "ablation": lambda: exp_main(arms=ABLATION_ARMS),
        "site": lambda: exp_main(arms=SITE_ARMS),
        "baselines": lambda: exp_main(arms=BASELINE_ARMS, budget=REV["budget_baselines"]),
        "ceiling": exp_ceiling,
        "rotation": exp_rotation,
        "unimts": exp_unimts,
        "posture": exp_posture,
        "hparam": exp_hparam,
        "augspec": exp_aug_specificity,
        "positions": exp_positions,
        "inference": inference_benchmark,
        "labeleff": exp_label_efficiency,
        "autoaug": exp_autoaug,
    }
    if name not in S:
        raise ValueError(f"unknown stage {name}; choose from {list(S)}")
    _p(f"\n######## stage {name} ########")
    return S[name]()


_p(f"""
[revision] ready. results -> {REV_DIR}
  run_stage("audit")      no training: c33 sign fix, geometry identities, tables  (minutes)
  estimate_cost()         time one epoch, project hours per stage
  run_stage("xgb")        engineered-feature baselines, raw + canon, 12 pairs     (CPU)
  run_stage("ablation")   A/B/C/D on 12 pairs (re-run on corrected data, per-run log)
  run_stage("site")       where yaw invariance must live (c39 c41)
  run_stage("baselines")  SO(3) aug, learned SO(3), PCA heading, Mizell, OIT (c46, e-mail)
  run_stage("ceiling")    in-domain ceilings on each pair's label set (c29 c64)
  run_stage("rotation")   controlled device- and gravity-frame rotations, 4 datasets (RQ3)
  run_stage("unimts")     published method, raw vs canonicalized input (c46.3)
  run_stage("posture")    sit/stand/lie cost of canonicalization (c44)
  run_stage("hparam")     source-only selection + target sensitivity (c40)
  optional: "augspec" "positions" "inference" "labeleff" "autoaug"
Then build the tables with make_all_tables() from revision_analysis (Part 3 of the single notebook).
""")

## Part 3: analysis
Tables, paired tests (Holm), hierarchical bootstrap, leave-one-dataset-out, gap decomposition, rotation summaries, headline numbers.

In [ ]:
# =============================================================================
#  REVISION ANALYSIS -- every number in the paper comes from here.
#
#  Reads the long-format run logs written by revision_experiments.py
#  (SAVE_PATH/revision/runs/*.csv) and writes tables, figures and a JSON of the
#  headline numbers to SAVE_PATH/revision/tables/. Needs numpy/pandas/scipy only
#  (matplotlib for figures), so it also runs on a laptop with the CSVs copied over:
#      python revision/revision_analysis.py --rev-dir /path/to/revision
#  or in Colab after the experiments:  %run -i -n revision/revision_analysis.py
#                                      make_all_tables()
# =============================================================================
import os, json, math, argparse, itertools
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr

try:
    REV_DIR  # noqa: F821  (defined when run with %run -i after the experiments)
except NameError:
    REV_DIR = os.environ.get("REV_DIR", os.path.join(os.getcwd(), "revision_results"))

ANA_DATASETS = ["hhar", "uci", "motion", "shoaib"]
DNAME = {"hhar": "HHAR", "uci": "UCI-HAR", "motion": "MotionSense", "shoaib": "Shoaib"}
ARM_LABEL = {
    "XGB_raw": "XGBoost (engineered features)", "XGB_canon": "XGBoost + gravity canon.",
    "A_ssl": "Baseline SSL (LIMU-BERT)", "B_grav": "+ Gravity canon.", "C_yaw": "+ Yaw inv. only",
    "D_gravyaw": "Gravity + yaw inv. (ours)", "D_pre": "Gravity + yaw consistency only",
    "D_ft": "Gravity + yaw augmentation only", "E_so3aug": "LIMU-BERT + SO(3) aug.",
    "F_so3inv": "Learned SO(3) invariance", "G_pca": "Gravity + PCA heading",
    "H_mizell": "Vertical/horizontal [Mizell]", "I_oit": "OIT [Yurtman & Barshan]",
    "UniMTS_raw": "UniMTS (fine-tuned)", "UniMTS_canon": "UniMTS + gravity canon.",
    "autoaug_raw": "AutoAugHAR", "autoaug_ours": "AutoAugHAR on our encoder",
}
MAIN_ORDER = ["XGB_raw", "A_ssl", "E_so3aug", "F_so3inv", "H_mizell", "I_oit", "G_pca",
              "UniMTS_raw", "UniMTS_canon", "XGB_canon", "D_gravyaw"]


def _p(*a):
    print(*a, flush=True)


def _tab_dir():
    d = os.path.join(REV_DIR, "tables"); os.makedirs(d, exist_ok=True); return d


def load_runs(*names):
    dfs = []
    for n in names:
        p = os.path.join(REV_DIR, "runs", f"{n}.csv")
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df["pair"] = df["source"].astype(str) + "->" + df["target"].astype(str)
    return df


def cross_runs():
    """All cross-dataset runs of the main comparison in one frame."""
    return load_runs("main", "xgb", "unimts", "autoaug")


# =============================================================================
#  1. CELL STATISTICS
# =============================================================================
def cell_means(df, metric="macro_f1"):
    g = df.groupby(["pair", "arm"])[metric]
    out = g.agg(["mean", "std", "count"]).reset_index()
    out["ci95"] = [1.96 * s / math.sqrt(n) if n > 1 else np.nan for s, n in zip(out["std"], out["count"])]
    return out


def pivot(df, metric="macro_f1", arms=None):
    m = cell_means(df, metric).pivot(index="pair", columns="arm", values="mean")
    if arms:
        m = m[[a for a in arms if a in m.columns]]
    return m


def _runs_by_seed(df, pair, arm, metric):
    """{pre_seed: np.array(run values)} -- the two-level structure of a cell."""
    s = df[(df.pair == pair) & (df.arm == arm)]
    return {k: g[metric].dropna().values for k, g in s.groupby("pre_seed")}


def _resample_cell(cell, rng):
    keys = list(cell)
    if not keys:
        return np.nan
    pick = rng.choice(len(keys), len(keys), replace=True)
    vals = []
    for i in pick:
        v = cell[keys[i]]
        vals.append(v[rng.integers(0, len(v), len(v))].mean())
    return float(np.mean(vals))


def cell_difference_ci(df, pair, arm_hi, arm_lo, metric="macro_f1", n_boot=2000, seed=0):
    """Two-level bootstrap CI of (arm_hi - arm_lo) inside ONE pair (c53)."""
    a, b = _runs_by_seed(df, pair, arm_hi, metric), _runs_by_seed(df, pair, arm_lo, metric)
    if not a or not b:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    d = np.array([_resample_cell(a, rng) - _resample_cell(b, rng) for _ in range(n_boot)])
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


# =============================================================================
#  2. TESTS ACROSS PAIRS  (c26 c49 c53)
# =============================================================================
def holm(p):
    p = np.asarray(p, float); k = len(p); order = np.argsort(p)
    adj = np.empty(k); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (k - r) * p[i]); adj[i] = min(1.0, run)
    return adj


def hier_bootstrap(df, arm_hi, arm_lo, metric="macro_f1", n_boot=2000, seed=0):
    """c49. The 12 pairs share datasets (each appears in 6) and each cell has a
    pretraining x fine-tuning seed structure. Level 1 resamples the 4 ANA_DATASETS with
    replacement and keeps every directed pair between distinct draws (with
    multiplicity); level 2 resamples pretraining seeds, then fine-tuning seeds,
    inside each cell. Returns the mean difference, its 95% CI and a two-sided
    bootstrap p. With only four datasets this is deliberately conservative."""
    pairs = sorted(set(df[df.arm == arm_hi].pair) & set(df[df.arm == arm_lo].pair))
    cells = {p: (_runs_by_seed(df, p, arm_hi, metric), _runs_by_seed(df, p, arm_lo, metric))
             for p in pairs}
    rng = np.random.default_rng(seed)
    stats, skipped = [], 0
    while len(stats) < n_boot:
        draw = rng.choice(ANA_DATASETS, len(ANA_DATASETS), replace=True)
        sel = [f"{s}->{t}" for s, t in itertools.product(draw, draw) if s != t]
        sel = [p for p in sel if p in cells]
        if not sel:
            skipped += 1
            if skipped > 10 * n_boot:
                break
            continue
        stats.append(np.mean([_resample_cell(cells[p][0], rng) - _resample_cell(cells[p][1], rng)
                              for p in sel]))
    stats = np.array(stats)
    obs = np.mean([np.mean(np.concatenate(list(cells[p][0].values())))
                   - np.mean(np.concatenate(list(cells[p][1].values()))) for p in pairs]) if pairs else np.nan
    if len(stats) < 100:                     # too few pairs for a dataset-level bootstrap
        return {"mean_delta": float(obs), "ci_lo": np.nan, "ci_hi": np.nan, "p_boot": np.nan,
                "n_pairs": len(pairs), "degenerate_draws_skipped": skipped}
    p2 = 2 * min((stats <= 0).mean(), (stats >= 0).mean())
    return {"mean_delta": float(obs), "ci_lo": float(np.percentile(stats, 2.5)),
            "ci_hi": float(np.percentile(stats, 97.5)), "p_boot": float(min(1.0, p2)),
            "n_pairs": len(pairs), "degenerate_draws_skipped": skipped}


def lodo(df, arm_hi, arm_lo, metric="macro_f1"):
    """Leave-one-dataset-out: the mean difference recomputed without every pair
    that involves one dataset (6 pairs remain). Shows whether one dataset drives it."""
    P = pivot(df, metric, [arm_hi, arm_lo]).dropna()
    out = {}
    for d in ANA_DATASETS:
        keep = [p for p in P.index if d not in p.split("->")]
        dd = (P.loc[keep, arm_hi] - P.loc[keep, arm_lo])
        out[f"without_{d}"] = float(dd.mean()) if len(dd) else np.nan
    return out


def paired_report(df, contrasts, metric="macro_f1", family="main"):
    """Wilcoxon signed-rank over pairs (cell means), Holm across the declared family,
    plus the hierarchical bootstrap and LODO for every contrast."""
    P = pivot(df, metric)
    rows = []
    for hi, lo in contrasts:
        if hi not in P or lo not in P:
            continue
        j = P[[hi, lo]].dropna()
        d = j[hi] - j[lo]
        try:
            pw = wilcoxon(j[hi], j[lo])[1]
        except ValueError:
            pw = np.nan
        hb = hier_bootstrap(df, hi, lo, metric)
        rows.append({"family": family, "metric": metric, "contrast": f"{hi} - {lo}",
                     "n_pairs": len(j), "mean_hi": j[hi].mean(), "mean_lo": j[lo].mean(),
                     "mean_delta": d.mean(), "median_delta": d.median(),
                     "wins": f"{int((d > 0).sum())}/{len(d)}", "wilcoxon_p": pw,
                     "boot_ci": f"[{hb['ci_lo']:+.3f}, {hb['ci_hi']:+.3f}]", "boot_p": hb["p_boot"],
                     **{k: round(v, 3) for k, v in lodo(df, hi, lo, metric).items()}})
    out = pd.DataFrame(rows)
    if len(out):
        out["wilcoxon_p_holm"] = holm(out["wilcoxon_p"].fillna(1.0).values)
    return out


# =============================================================================
#  3. TABLES
# =============================================================================
def _fmt(v, nd=3):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{nd}f}".lstrip("0").replace("-0.", "-.")


def main_table(df=None, arms=None, metric_pair=("macro_f1", "stairs_f1")):
    """Table 2 replacement (c46 c52 c53 c54): every method, every pair, "macro / stairs"
    from the same predictions, averages over the SAME pairs for every method. Bold = best cell
    mean; '~' marks methods whose two-level bootstrap CI of the difference to the
    best includes 0 (statistically indistinguishable from the best in that pair)."""
    df = cross_runs() if df is None else df
    if df.empty:
        _p("no cross-dataset runs yet"); return None
    arms = [a for a in (arms or MAIN_ORDER) if a in set(df.arm)]
    M = pivot(df, metric_pair[0], arms); S = pivot(df, metric_pair[1], arms)
    rows = []
    for pair in M.index:
        best = M.loc[pair].idxmax()
        r = {"pair": pair}
        for a in arms:
            m, s = M.loc[pair, a], S.loc[pair, a]
            tag = ""
            if a == best:
                tag = "**"
            elif not np.isnan(m):
                _, lo, hi = cell_difference_ci(df, pair, best, a)
                tag = "~" if (not np.isnan(lo) and lo <= 0 <= hi) else ""
            r[a] = f"{tag}{_fmt(m)}{tag if tag == '**' else ''} / {_fmt(s)}"
        rows.append(r)
    complete = M.dropna()
    avg = {"pair": f"mean over the {len(complete)} pairs every method has"}
    own = {"pair": "mean over the pairs each method has (n)"}
    for a in arms:
        avg[a] = f"{_fmt(complete[a].mean())} / {_fmt(S.loc[complete.index, a].mean())}"
        have = M[a].dropna()
        own[a] = f"{_fmt(have.mean())} / {_fmt(S.loc[have.index, a].mean())} (n={len(have)})"
    rows += [avg, own]
    out = pd.DataFrame(rows).rename(columns={a: ARM_LABEL.get(a, a) for a in arms})
    out.to_csv(os.path.join(_tab_dir(), "T_main.csv"), index=False)
    return out


def ablation_table(df=None):
    """Table 4 replacement (c41 c57 c62 c63): the 2x2 plus the yaw-site arms. The
    interaction column has a run-level bootstrap CI; the draft's rho=-0.937 analysis
    is removed (c58 c59 c68)."""
    df = load_runs("main") if df is None else df
    arms = ["A_ssl", "B_grav", "C_yaw", "D_gravyaw", "D_pre", "D_ft"]
    P = pivot(df, "macro_f1", arms)
    if not set(arms[:4]) <= set(P.columns):
        _p("ablation incomplete"); return None
    out = P.copy()
    out["dC (B-A)"] = P.B_grav - P.A_ssl
    out["dY (C-A)"] = P.C_yaw - P.A_ssl
    out["dBoth (D-A)"] = P.D_gravyaw - P.A_ssl
    out["yaw on top of canon (D-B)"] = P.D_gravyaw - P.B_grav
    inter = P.D_gravyaw - P.B_grav - P.C_yaw + P.A_ssl
    cis = []
    rng = np.random.default_rng(0)
    for pair in P.index:
        cells = [_runs_by_seed(df, pair, a, "macro_f1") for a in arms[:4]]
        bs = [(_resample_cell(cells[3], rng) - _resample_cell(cells[1], rng)
               - _resample_cell(cells[2], rng) + _resample_cell(cells[0], rng)) for _ in range(2000)]
        cis.append(f"{inter[pair]:+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    out["interaction [95% CI]"] = cis
    out.loc["mean"] = list(P.mean().reindex(P.columns)) + [out[c].mean() for c in
                                                           ["dC (B-A)", "dY (C-A)", "dBoth (D-A)",
                                                            "yaw on top of canon (D-B)"]] + [f"{inter.mean():+.3f}"]
    out.round(3).to_csv(os.path.join(_tab_dir(), "T_ablation.csv"))
    return out


def family_frame_table(df=None):
    """Table 5 replacement (c62 c63 c64): model family x input frame, with GraviHAR
    as its own row so both families are compared under the same treatment."""
    df = cross_runs() if df is None else df
    rows = []
    for fam, raw, canon in (("XGBoost (engineered features)", "XGB_raw", "XGB_canon"),
                            ("LIMU-BERT (sequence encoder)", "A_ssl", "B_grav")):
        for metric in ("macro_f1", "stairs_f1"):
            P = pivot(df, metric, [raw, canon]).dropna()
            if len(P.columns) < 2:
                continue
            rows.append({"family": fam, "metric": metric, "device frame": P[raw].mean(),
                         "gravity-aligned frame": P[canon].mean(), "gain": (P[canon] - P[raw]).mean(),
                         "n_pairs": len(P)})
    for metric in ("macro_f1", "stairs_f1"):
        P = pivot(df, metric, ["A_ssl", "D_gravyaw"]).dropna()
        if len(P.columns) == 2:
            rows.append({"family": "GraviHAR (gravity + yaw invariance)", "metric": metric,
                         "device frame": P["A_ssl"].mean(), "gravity-aligned frame": P["D_gravyaw"].mean(),
                         "gain": (P["D_gravyaw"] - P["A_ssl"]).mean(), "n_pairs": len(P)})
    out = pd.DataFrame(rows)
    out.round(3).to_csv(os.path.join(_tab_dir(), "T_family_frame.csv"), index=False)
    return out


def gap_table(df=None):
    """c29 c64 c74. For each pair: the in-domain ceiling of the target on the pair's
    own label set (same method, subject-disjoint CV), the cross-dataset score, the
    gap, and the share of the baseline's gap that orientation treatment closes:
        share = (cross_D - cross_A) / (ceil_D - cross_A)     (LIMU-BERT family)
        share = (XGB_canon - XGB_raw) / (ceil_XGBcanon - XGB_raw)  (feature family)"""
    df = cross_runs() if df is None else df
    ce = load_runs("ceiling")
    if ce.empty or df.empty:
        _p("ceiling runs missing"); return None
    ce["tgt"] = ce["target"].str.split("[").str[0]
    ce["labset"] = ce["labels"].astype(str)
    C = ce.groupby(["tgt", "labset", "arm"])["macro_f1"].mean()
    X = pivot(df, "macro_f1")
    lab = df.groupby("pair")["labels"].first().astype(str)
    rows = []
    for pair in X.index:
        s, t = pair.split("->")
        r = {"pair": pair}
        for arm in ("A_ssl", "B_grav", "D_gravyaw", "XGB_raw", "XGB_canon"):
            r[f"cross_{arm}"] = X.loc[pair].get(arm, np.nan)
            r[f"ceil_{arm}"] = C.get((t, lab.get(pair, ""), arm), np.nan)
        if not np.isnan(r["ceil_D_gravyaw"]):
            r["gap_A"] = r["ceil_D_gravyaw"] - r["cross_A_ssl"]
            r["gap_D"] = r["ceil_D_gravyaw"] - r["cross_D_gravyaw"]
            r["share_closed_ssl"] = (r["cross_D_gravyaw"] - r["cross_A_ssl"]) / max(r["gap_A"], 1e-9)
            r["share_closed_by_canon_only"] = (r["cross_B_grav"] - r["cross_A_ssl"]) / max(r["gap_A"], 1e-9)
        if not np.isnan(r.get("ceil_XGB_canon", np.nan)):
            g = r["ceil_XGB_canon"] - r["cross_XGB_raw"]
            r["share_closed_xgb"] = (r["cross_XGB_canon"] - r["cross_XGB_raw"]) / max(g, 1e-9)
        rows.append(r)
    out = pd.DataFrame(rows)
    num = out.select_dtypes("number")
    out = pd.concat([out, pd.DataFrame([{"pair": "mean", **num.mean().to_dict()}])], ignore_index=True)
    out.round(3).to_csv(os.path.join(_tab_dir(), "T_gap.csv"), index=False)
    return out


def rotation_tables(make_figs=True):
    """RQ3. Curves, ORS = mean_{theta>0} F1(theta)/F1(0), worst-case drop, per
    dataset, arm and perturbation family; and accuracy as a function of the
    residual heading |psi| pooled over every perturbation (canonicalized arms)."""
    df = load_runs("rotation")
    if df.empty:
        _p("no rotation runs"); return None
    cur = df.groupby(["source", "arm", "perturb", "angle"])["macro_f1"].mean().reset_index()
    rows = []
    for (d, a, pt), g in cur.groupby(["source", "arm", "perturb"]):
        g = g.set_index("angle")["macro_f1"]
        if 0.0 not in g.index:
            continue
        f0, rest = g.loc[0.0], g.drop(0.0)
        rows.append({"dataset": d, "arm": a, "perturb": pt, "F1(0)": f0,
                     "ORS": rest.mean() / max(f0, 1e-9), "worst_drop": f0 - rest.min()})
    ors = pd.DataFrame(rows)
    ors.round(3).to_csv(os.path.join(_tab_dir(), "T_rotation_ORS.csv"), index=False)
    piv = ors.pivot_table(index=["dataset", "arm"], columns="perturb", values="ORS")
    piv.round(3).to_csv(os.path.join(_tab_dir(), "T_rotation_ORS_wide.csv"))
    # accuracy vs |psi|
    acc = {}
    for _, r in df.iterrows():
        try:
            bins = json.loads(r["extra"]).get("psi_bins", {})
        except Exception:
            continue
        for b, (c, n) in bins.items():
            k = (r["source"], r["arm"], float(b))
            cc, nn = acc.get(k, (0, 0)); acc[k] = (cc + c, nn + n)
    psi = pd.DataFrame([{"dataset": k[0], "arm": k[1], "psi_bin_lo": k[2], "acc": v[0] / max(v[1], 1),
                         "n": v[1]} for k, v in acc.items()])
    psi.round(4).to_csv(os.path.join(_tab_dir(), "T_rotation_psi.csv"), index=False)
    if make_figs:
        _rotation_figs(cur, psi)
    return ors, piv, psi


def _rotation_figs(cur, psi):
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    except Exception:
        return
    arms = [a for a in ["A_ssl", "B_grav", "C_yaw", "D_gravyaw", "G_pca", "H_mizell", "E_so3aug"]
            if a in set(cur.arm)]
    perts = [p for p in ["device_x", "device_y", "device_z", "phys_tilt", "phys_heading"]
             if p in set(cur.perturb)]
    ds = [d for d in ANA_DATASETS if d in set(cur.source)]
    fig, ax = plt.subplots(len(ds), len(perts), figsize=(3.0 * len(perts), 2.4 * len(ds)),
                           squeeze=False, sharey=True)
    for i, d in enumerate(ds):
        for j, pt in enumerate(perts):
            for a in arms:
                g = cur[(cur.source == d) & (cur.arm == a) & (cur.perturb == pt)].sort_values("angle")
                if len(g):
                    ax[i, j].plot(g.angle, g.macro_f1, marker="o", ms=3, label=ARM_LABEL.get(a, a))
            ax[i, j].set_title(f"{DNAME[d]} / {pt}", fontsize=8); ax[i, j].set_ylim(0, 1)
    ax[0, -1].legend(fontsize=6, loc="lower left")
    fig.tight_layout(); fig.savefig(os.path.join(_tab_dir(), "F_rotation_curves.pdf")); plt.close(fig)
    if len(psi):
        fig, ax = plt.subplots(1, len(ds), figsize=(3.2 * len(ds), 2.6), squeeze=False, sharey=True)
        for j, d in enumerate(ds):
            for a in [x for x in ["B_grav", "D_gravyaw", "G_pca"] if x in set(psi.arm)]:
                g = psi[(psi.dataset == d) & (psi.arm == a)].sort_values("psi_bin_lo")
                ax[0, j].plot(g.psi_bin_lo, g.acc, marker="o", label=ARM_LABEL.get(a, a))
            ax[0, j].set_title(DNAME[d], fontsize=9); ax[0, j].set_xlabel("|residual heading| (deg)")
        ax[0, 0].set_ylabel("accuracy"); ax[0, -1].legend(fontsize=7)
        fig.tight_layout(); fig.savefig(os.path.join(_tab_dir(), "F_rotation_psi.pdf")); plt.close(fig)


def posture_table():
    """c44: per-class F1 of the posture classes by arm, within and across datasets."""
    df = load_runs("posture")
    if df.empty:
        _p("no posture runs"); return None
    df["setting"] = np.where(df.source == df.target, "within", "cross")
    rows = []
    for (st, pair, arm), g in df.groupby(["setting", "pair", "arm"]):
        pcs = pd.DataFrame([json.loads(x) for x in g.per_class]).mean()
        rows.append({"setting": st, "pair": pair, "arm": arm, "macro_f1": g.macro_f1.mean(),
                     **{f"F1_{n}": pcs.get(str(c), np.nan) for c, n in ((8, "sit"), (9, "stand"), (10, "lie"))}})
    out = pd.DataFrame(rows)
    out.round(3).to_csv(os.path.join(_tab_dir(), "T_posture.csv"), index=False)
    return out


def hparam_tables():
    """c40: which (lambda, floor) the SOURCE-ONLY criterion picks per source, the full
    grid on the targets, and the selected vs default configuration."""
    df = load_runs("hparam")
    if df.empty:
        _p("no hparam runs"); return None
    val = df[df.exp == "hparam_val"].groupby(["source", "arm"])["macro_f1"].mean().reset_index()
    sel = val.loc[val.groupby("source")["macro_f1"].idxmax()].rename(columns={"arm": "selected",
                                                                             "macro_f1": "rotated_val_f1"})
    tgt = df[df.exp == "hparam"].groupby(["pair", "arm"])["macro_f1"].mean().unstack()
    tgt["selected"] = [sel.set_index("source").loc[p.split("->")[0], "selected"]
                       if p.split("->")[0] in set(sel.source) else None for p in tgt.index]
    tgt["F1_selected"] = [tgt.loc[p, s] if s in tgt.columns else np.nan for p, s in zip(tgt.index, tgt.selected)]
    default = "D_l0.1_f0.3"
    if default in tgt.columns:
        tgt["F1_default"] = tgt[default]
    grid_range = tgt.drop(columns=[c for c in ["selected", "F1_selected", "F1_default"] if c in tgt]).agg(
        ["min", "max"], axis=1)
    tgt["grid_min"], tgt["grid_max"] = grid_range["min"], grid_range["max"]
    sel.round(4).to_csv(os.path.join(_tab_dir(), "T_hparam_selection.csv"), index=False)
    tgt.round(3).to_csv(os.path.join(_tab_dir(), "T_hparam_targets.csv"))
    return sel, tgt


def simple_table(log, index="pair", col="arm", metric="macro_f1", name=None):
    df = load_runs(log)
    if df.empty:
        return None
    t = df.groupby([index, col])[metric].mean().unstack()
    t.loc["mean"] = t.mean()
    t.round(3).to_csv(os.path.join(_tab_dir(), f"T_{name or log}.csv"))
    return t


# =============================================================================
#  4. HEADLINE NUMBERS (abstract / contributions / conclusion use ONLY these)
# =============================================================================
def headline_numbers(df=None):
    df = cross_runs() if df is None else df
    out = {}
    for metric in ("macro_f1", "stairs_f1"):
        P = pivot(df, metric).dropna(axis=1, how="all")
        full = P.dropna(axis=1)
        out[metric] = {a: round(float(full[a].mean()), 3) for a in full.columns}
        out[metric]["n_pairs"] = int(len(full))
        if "D_gravyaw" in full:
            base = [a for a in full.columns if a not in ("D_gravyaw", "D_pre", "D_ft", "B_grav", "C_yaw")]
            if base:
                best = max(base, key=lambda a: full[a].mean())
                out[metric]["best_baseline"] = best
                out[metric]["ours_minus_best_baseline_points"] = round(
                    float((full["D_gravyaw"] - full[best]).mean()), 3)
    json.dump(out, open(os.path.join(_tab_dir(), "headline_numbers.json"), "w"), indent=2)
    return out


# =============================================================================
#  5. ONE CALL FOR EVERYTHING
# =============================================================================
MAIN_CONTRASTS = [("B_grav", "A_ssl"), ("C_yaw", "A_ssl"), ("D_gravyaw", "A_ssl"),
                  ("D_gravyaw", "B_grav"), ("D_gravyaw", "C_yaw")]
SITE_CONTRASTS = [("D_gravyaw", "D_ft"), ("D_gravyaw", "D_pre"), ("D_ft", "B_grav")]
BASELINE_CONTRASTS = [("D_gravyaw", "XGB_canon"), ("D_gravyaw", "XGB_raw"), ("D_gravyaw", "E_so3aug"),
                      ("D_gravyaw", "F_so3inv"), ("D_gravyaw", "G_pca"), ("D_gravyaw", "H_mizell"),
                      ("D_gravyaw", "I_oit"), ("D_gravyaw", "UniMTS_raw"), ("UniMTS_canon", "UniMTS_raw"),
                      ("XGB_canon", "XGB_raw")]


def make_all_tables():
    df = cross_runs()
    md = ["# Revision tables (generated by revision_analysis.py -- do not edit by hand)\n"]

    def add(title, t):
        if t is None or (hasattr(t, "empty") and t.empty):
            md.append(f"\n## {title}\n\n_not available yet_\n"); return
        try:
            body = t.to_markdown(floatfmt=".3f")
        except ImportError:                      # tabulate not installed
            body = "```\n" + t.to_string() + "\n```"
        md.append(f"\n## {title}\n\n" + body + "\n")

    if not df.empty:
        add("Main cross-dataset results (macro-F1 / stairs-F1)", main_table(df))
        add("Component ablation", ablation_table(load_runs("main")))
        add("Model family x input frame", family_frame_table(df))
        tests = []
        for fam, con in (("ablation", MAIN_CONTRASTS), ("yaw site", SITE_CONTRASTS),
                         ("baselines", BASELINE_CONTRASTS)):
            for metric in ("macro_f1", "stairs_f1"):
                tests.append(paired_report(df, con, metric, fam))
        tests = pd.concat([t for t in tests if len(t)], ignore_index=True) if any(len(t) for t in tests) else None
        if tests is not None:
            tests.round(4).to_csv(os.path.join(_tab_dir(), "T_tests.csv"), index=False)
        add("Paired tests over pairs (Holm within family), hierarchical bootstrap, LODO", tests)
        add("Headline numbers", pd.DataFrame(headline_numbers(df)))
    add("In-domain ceiling and orientation share of the gap", gap_table(df) if not df.empty else None)
    r = rotation_tables()
    add("Controlled rotations: ORS", None if r is None else r[1])
    add("Posture classes (c44)", posture_table())
    h = hparam_tables()
    add("Hyperparameters: source-only selection (c40)", None if h is None else h[0])
    add("Hyperparameters: target sensitivity grid (c40)", None if h is None else h[1])
    add("Augmentation specificity", simple_table("augspec"))
    add("Shoaib body-position transfer", simple_table("positions"))
    add("Label efficiency", simple_table("labeleff", index="pair", col="angle", name="labeleff"))
    path = os.path.join(_tab_dir(), "paper_tables.md")
    open(path, "w").write("\n".join(md))
    _p(f"[analysis] wrote {path}")
    return path

---
## Stage 0: audit (no training, minutes)
MotionSense sign (c33), geometry identities (c36 c37 c51), device-axis decomposition (c37 c50 c51), antipode (c38), heading definition (c32), cadence and energy (c66), preprocessing and label tables (c44 c45), parameter split (c42).

In [ ]:
run_stage("audit")

Time one epoch on this GPU and project the hours per stage.

In [ ]:
estimate_cost()

## Tier 1 (minimum for the resubmission)

In [ ]:
run_stage("xgb")        # engineered features, raw + canonicalized, 12 pairs (CPU; cached after the first run)

In [ ]:
run_stage("ablation")   # A/B/C/D on 12 pairs, 2 pretraining x 4 fine-tuning seeds

In [ ]:
run_stage("baselines")  # SO(3) aug, learned SO(3), gravity + PCA heading, Mizell, OIT

In [ ]:
run_stage("ceiling")    # in-domain ceilings on each pair's label set (c29 c64)

In [ ]:
run_stage("rotation")   # controlled device- and gravity-frame rotations, 4 datasets (RQ3)

In [ ]:
# UniMTS (c46.3): clones the official repo, installs CLIP, downloads the checkpoint.
# To reduce cost: exp_unimts(epochs=5, max_train=20000)
run_stage("unimts")

## Tier 2

In [ ]:
run_stage("site")       # where the yaw invariance must live (c39 c41)

In [ ]:
run_stage("posture")    # sit / stand / lie (c44)

In [ ]:
run_stage("hparam")     # source-only selection + target sensitivity grid (c40)

In [ ]:
run_stage("inference")  # latency with CPU model, threads, batch; TorchScript/ONNX export (c27 c56)

## Tier 3 (optional)

In [ ]:
# run_stage("augspec"); run_stage("positions"); run_stage("labeleff"); run_stage("autoaug")

---
## Tables, tests, figures, headline numbers
Can be run at any point; tables for stages that have not run yet are marked as not available.

In [ ]:
path = make_all_tables()
print(open(path).read()[:8000])